# Fit and issue the lagged baseline

For first-time setup, start with the [monthly forecast guide](../../docs/MONTHLY_FORECAST.md). It separates a new registry from continuation of an existing run. This page retains the detailed record of October's workflow.

The workflow covers final fitting, model registration, source normalization, inference and freezing. The final fit and authenticated ERA5/SEAS5 acquisition completed on 29 September 2026. **October's forecast was frozen on 30 September at 18:18 UTC**, after the original IRI recipe supplied the complete CFSv2 ensemble. The [operational record](OPERATIONAL_STATUS.md) identifies the frozen package, source-equivalence evidence and issued forecast.

The separate `baseline_inference.ipynb` bundles this code for Kaggle. Attach the official training data, the two prepared seasonal datasets, and the **complete existing registry** with its `eventos` and `objetos` directories. It must resume that chain rather than create a replacement with invented earlier receipt dates.

The first cell installs missing NetCDF/GRIB/CDS readers before importing xarray. Model-library versions are checked against the fixed training environment; they are not automatically upgraded. In a session where xarray was already imported before a reader was installed, restart from the saved registry instead of treating a cached missing reader as a missing source.

## Training

The package is fitted once on January 1993–December 2022 with atmospheric lag 4, index lag 3, the existing 29/31-feature tree models and local ridge. The archived NOAA table supplies only training origins; current inference reads the individually recorded PSL responses. Input hashes, source snapshot, features, settings and code are recorded. The 2023–2024 test partitions are not loaded. A competition model or historical fold cannot substitute for this fit.

```bash
python research/prospective/fit_baseline.py --registry outputs/prospective --output outputs/prospective_model_v1 --official /path/to/official --seas5 /path/to/seas5 --cfsv2 /path/to/cfsv2
```

An existing model event blocks refitting. A package directory is never silently overwritten. The registered files match the original plan exactly; the plan and earlier events are not rewritten.

## Sources

Public NOAA/CFSv2 collection continues through the existing adapters. ERA5 and SEAS5 use an existing CDS credential, supplied through the normal `cdsapi` environment or local `.cdsapirc`, never through committed code. On Kaggle, the operational run uses an enabled `CDS_API_KEY` secret, passed through the environment variable `CDSAPI_KEY` with `CDSAPI_URL=https://cds.climate.copernicus.eu/api`; do not print it. The secret label and environment-variable name are different. The CDS dataset terms must already have been accepted by the account owner.

```bash
python research/prospective/collect_cds.py --registry outputs/prospective --target 2026-10
```

The CDS adapter requests monthly **GRIB**, so the ERA5 `expver=1` evidence remains available. It requires the full quarter-degree atmospheric grid, specified units and all variables, plus all 51 SEAS5 members for system 51, forecast month 2. The existing 0.01 mm/day negative-residual tolerance is retained and each correction is audited. Unknown metadata, missing members, negative values beyond tolerance or an incomplete grid block normalization.

ERA5 documentation explains why a NetCDF containing only ERA5 or only ERA5T cannot reliably distinguish the two: [ECMWF data documentation](https://confluence.ecmwf.int/spaces/CKB/pages/76414402/ERA5+data+documentation). A delay alone is not treated as proof of a final vintage.

To register a GRIB already downloaded, preserving the actual *current* import time:

```bash
python research/prospective/prepare_cds.py --registry outputs/prospective --target 2026-10 --source seas5 --file /path/to/source.grib
```

Original bytes, normalized bytes and adapter source are linked in the registry. Local import cannot establish an earlier download time. The authenticated run validated June 2026 ERA5 final monthly fields and the 51-member September 2026 SEAS5 forecast for October; future acquisitions must pass the same checks.

## Inference

```bash
python research/prospective/issue_forecast.py --registry outputs/prospective --target 2026-10 --output outputs/forecast_2026_10.nc
```

Inference materializes the exact model and source objects selected by the registry, reconstructs nonnegative rainfall, and freezes both forecast and training climatology. A source-version change during inference invalidates freezing. An existing monthly prediction or a passed deadline blocks emission. A file produced before a failed freeze is only an unregistered artifact; it is not an issued forecast.

The October example is already completed in the latest registry. Do not rerun it to replace the prediction. Its local inference used the exact frozen NumPy 2.0.2, pandas 2.3.3, xarray 2025.12.0 and LightGBM 4.6.0 versions; the actual Windows/Python 3.12.7 environment is retained in the evidence. The synthetic fitted-package replay passed. GRIB checks passed separately in the acquisition environment; inference reuses the normalized registered inputs.

The U-Net experiment has a separate development protocol. It neither changes this baseline plan nor registers a neural prediction in the baseline's name.


In [ ]:
import importlib.util, subprocess, sys
READERS = {'netCDF4': 'netCDF4>=1.6,<2', 'cftime': 'cftime>=1.6,<2',
           'h5netcdf': 'h5netcdf>=1.3,<2', 'cdsapi': 'cdsapi>=0.7,<1',
           'eccodes': 'eccodes>=2.38,<3'}
missing = [requirement for name, requirement in READERS.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
from pathlib import Path
import hashlib, json, sys, runpy, shutil
BASE = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'outputs'
RUN_ROOT = BASE/'baseline_inference'
CODE_ROOT = RUN_ROOT/'code'
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = {'competition/metadata/NOAA/indices_noaa.csv': 'time_origem,nino34,nino12,tna,tsa\n1976-12-01,0.64,0.25,-0.60,-0.47\n1977-01-01,0.84,0.29,-0.40,-0.53\n1977-02-01,0.55,-0.14,-0.40,-0.27\n1977-03-01,0.40,-0.38,-0.27,-0.34\n1977-04-01,-0.25,-0.68,-0.25,-0.34\n1977-05-01,-0.03,-0.45,-0.28,-0.39\n1977-06-01,0.36,0.08,-0.20,-0.41\n1977-07-01,0.45,0.03,-0.42,-0.40\n1977-08-01,0.25,-0.52,-0.62,-0.19\n1977-09-01,0.44,-0.50,-0.61,-0.12\n1977-10-01,0.89,-0.19,-0.66,-0.31\n1977-11-01,1.03,-0.29,-0.57,-0.27\n1977-12-01,1.09,-0.56,-0.44,-0.34\n1978-01-01,0.72,-0.07,-0.13,-0.52\n1978-02-01,0.51,-0.35,0.02,-0.56\n1978-03-01,0.06,-0.77,0.09,-0.80\n1978-04-01,-0.37,-0.37,0.10,-0.96\n1978-05-01,-0.50,-0.54,-0.13,-0.99\n1978-06-01,-0.45,-0.86,-0.39,-0.78\n1978-07-01,-0.48,-0.40,-0.60,-0.33\n1978-08-01,-0.62,-0.64,-0.66,-0.40\n1978-09-01,-0.60,-0.30,-0.59,-0.50\n1978-10-01,-0.34,-0.43,-0.35,-0.61\n1978-11-01,-0.15,0.04,-0.30,-0.61\n1978-12-01,0.07,0.23,-0.48,-0.43\n1979-01-01,0.05,0.23,-0.08,-0.49\n1979-02-01,0.09,-0.58,0.10,-0.51\n1979-03-01,0.13,-0.25,-0.01,-0.65\n1979-04-01,0.17,0.51,0.14,-0.30\n1979-05-01,-0.09,0.27,0.15,-0.27\n1979-06-01,0.04,0.76,0.18,-0.21\n1979-07-01,-0.20,0.45,0.01,-0.12\n1979-08-01,0.21,0.50,-0.25,0.07\n1979-09-01,0.48,1.02,-0.27,0.14\n1979-10-01,0.32,0.67,-0.32,-0.17\n1979-11-01,0.35,0.12,-0.20,-0.12\n1979-12-01,0.69,-0.34,-0.03,-0.28\n1980-01-01,0.55,-0.10,0.15,-0.48\n1980-02-01,0.33,-0.25,0.23,-0.61\n1980-03-01,0.09,0.02,0.02,-0.61\n1980-04-01,0.20,-0.00,0.40,-0.42\n1980-05-01,0.13,0.25,0.50,-0.38\n1980-06-01,0.37,0.21,0.23,-0.39\n1980-07-01,0.02,-0.44,-0.08,-0.31\n1980-08-01,-0.23,-0.44,-0.04,-0.16\n1980-09-01,-0.12,-0.11,-0.17,-0.04\n1980-10-01,-0.11,-0.52,-0.38,-0.25\n1980-11-01,0.10,-0.68,-0.28,-0.43\n1980-12-01,0.36,-1.13,-0.31,-0.86\n1981-01-01,-0.46,-1.29,0.03,-0.61\n1981-02-01,-0.45,-0.83,-0.01,-0.73\n1981-03-01,-0.02,-0.19,0.45,-0.99\n1981-04-01,-0.17,-0.31,0.18,-0.83\n1981-05-01,-0.11,-0.22,0.16,-0.75\n1981-06-01,-0.15,-0.06,-0.06,-0.53\n1981-07-01,-0.43,-0.56,-0.12,-0.48\n1981-08-01,-0.18,-0.55,-0.24,-0.07\n1981-09-01,-0.07,-0.24,-0.23,0.05\n1981-10-01,-0.03,-0.52,-0.39,0.12\n1981-11-01,-0.27,-0.55,-0.18,0.08\n1981-12-01,0.07,-0.17,-0.02,-0.16\n1982-01-01,-0.04,-0.12,-0.06,-0.30\n1982-02-01,-0.13,-0.35,-0.24,-0.47\n1982-03-01,-0.02,-0.72,-0.23,-0.56\n1982-04-01,0.24,-0.51,-0.32,-0.66\n1982-05-01,0.65,0.08,-0.24,-0.71\n1982-06-01,0.92,0.27,-0.40,-0.76\n1982-07-01,0.64,0.60,-0.54,-0.70\n1982-08-01,0.93,1.09,-0.64,-0.77\n1982-09-01,1.39,2.01,-0.66,-0.46\n1982-10-01,2.00,2.28,-0.77,-0.33\n1982-11-01,2.07,3.08,-0.77,-0.46\n1982-12-01,2.33,3.20,-0.79,-0.43\n1983-01-01,2.43,2.73,-0.61,-0.01\n1983-02-01,2.22,2.03,-0.19,0.11\n1983-03-01,1.69,2.34,0.28,-0.48\n1983-04-01,1.12,3.36,0.29,-0.56\n1983-05-01,1.12,4.10,-0.03,-0.80\n1983-06-01,0.62,4.56,0.10,-0.76\n1983-07-01,-0.11,3.93,-0.05,-0.56\n1983-08-01,-0.13,3.11,-0.33,-0.28\n1983-09-01,-0.50,2.07,-0.56,-0.11\n1983-10-01,-1.03,1.02,-0.58,-0.31\n1983-11-01,-1.13,0.56,-0.26,-0.12\n1983-12-01,-0.95,0.22,-0.06,0.12\n1984-01-01,-0.80,-0.29,-0.30,0.35\n1984-02-01,-0.53,-0.77,-0.38,0.38\n1984-03-01,-0.48,-0.41,-0.25,0.44\n1984-04-01,-0.60,-0.25,-0.34,0.19\n1984-05-01,-0.64,-1.11,-0.36,-0.08\n1984-06-01,-0.90,-0.92,-0.61,0.14\n1984-07-01,-0.40,-0.54,-0.63,0.29\n1984-08-01,-0.40,-0.52,-0.62,0.57\n1984-09-01,-0.35,-0.36,-0.62,0.69\n1984-10-01,-0.83,-0.61,-0.92,0.36\n1984-11-01,-1.18,-0.18,-0.77,0.13\n1984-12-01,-1.53,-0.50,-0.79,-0.01\n1985-01-01,-0.89,-0.77,-0.68,-0.02\n1985-02-01,-0.91,-1.13,-0.55,0.25\n1985-03-01,-0.96,-0.59,-0.77,-0.21\n1985-04-01,-0.98,-1.38,-0.81,-0.16\n1985-05-01,-0.90,-1.41,-0.91,-0.10\n1985-06-01,-0.76,-1.25,-0.54,-0.27\n1985-07-01,-0.61,-1.04,-0.32,0.02\n1985-08-01,-0.49,-1.02,-0.42,-0.10\n1985-09-01,-0.57,-0.83,-0.54,-0.13\n1985-10-01,-0.64,-0.82,-0.57,0.05\n1985-11-01,-0.43,-0.92,-0.39,0.06\n1985-12-01,-0.40,-0.75,-0.54,0.03\n1986-01-01,-0.79,-0.59,-0.86,-0.14\n1986-02-01,-0.64,-0.35,-0.80,0.20\n1986-03-01,-0.51,-0.84,-0.76,0.04\n1986-04-01,-0.41,-0.86,-0.80,-0.03\n1986-05-01,-0.47,-0.82,-0.74,-0.08\n1986-06-01,-0.08,-0.90,-0.84,0.10\n1986-07-01,0.16,-0.14,-0.65,-0.06\n1986-08-01,0.24,0.17,-0.74,0.01\n1986-09-01,0.59,-0.13,-0.34,0.16\n1986-10-01,0.90,0.14,-0.63,-0.08\n1986-11-01,1.04,0.28,-0.91,-0.35\n1986-12-01,0.98,0.53,-0.98,-0.24\n1987-01-01,1.19,0.77,-0.57,-0.32\n1987-02-01,1.17,1.02,-0.35,-0.19\n1987-03-01,1.24,1.15,0.06,-0.18\n1987-04-01,0.89,1.51,0.17,-0.27\n1987-05-01,0.91,1.35,0.17,-0.06\n1987-06-01,1.24,1.06,0.37,0.11\n1987-07-01,1.34,1.19,0.21,0.25\n1987-08-01,1.48,0.91,0.22,0.24\n1987-09-01,1.53,1.31,0.02,0.17\n1987-10-01,1.36,1.50,0.02,0.19\n1987-11-01,1.29,0.90,0.05,0.45\n1987-12-01,1.05,0.89,0.22,0.13\n1988-01-01,0.69,0.09,-0.09,0.30\n1988-02-01,0.35,-0.27,-0.20,0.39\n1988-03-01,0.29,-0.52,0.12,0.43\n1988-04-01,-0.49,-0.90,0.22,0.28\n1988-05-01,-1.05,-1.08,0.08,0.37\n1988-06-01,-1.46,-1.96,0.10,0.42\n1988-07-01,-1.54,-1.54,-0.00,0.20\n1988-08-01,-1.44,-1.59,-0.31,0.30\n1988-09-01,-1.33,-1.43,-0.32,0.16\n1988-10-01,-2.09,-1.26,-0.63,0.15\n1988-11-01,-2.18,-0.65,-0.54,-0.05\n1988-12-01,-1.98,-0.97,-0.52,-0.07\n1989-01-01,-1.95,-0.28,-0.67,0.08\n1989-02-01,-1.37,0.05,-0.66,-0.04\n1989-03-01,-1.33,-0.45,-0.82,0.00\n1989-04-01,-1.11,-0.49,-1.04,-0.02\n1989-05-01,-0.80,-1.16,-0.86,-0.01\n1989-06-01,-0.64,-0.73,-0.42,0.00\n1989-07-01,-0.47,-0.44,-0.03,0.00\n1989-08-01,-0.58,-0.46,0.05,0.19\n1989-09-01,-0.38,-0.47,-0.32,0.06\n1989-10-01,-0.40,-0.29,-0.30,0.06\n1989-11-01,-0.30,-0.38,-0.35,-0.24\n1989-12-01,-0.13,-0.43,-0.07,-0.51\n1990-01-01,0.02,-0.33,-0.33,-0.68\n1990-02-01,0.36,-0.27,-0.28,-0.14\n1990-03-01,0.20,-0.41,-0.20,-0.03\n1990-04-01,0.26,-0.33,-0.10,0.04\n1990-05-01,0.32,-0.23,-0.07,-0.40\n1990-06-01,0.00,-0.05,-0.27,-0.23\n1990-07-01,0.15,-0.87,-0.22,-0.37\n1990-08-01,0.17,-0.61,-0.04,-0.33\n1990-09-01,0.12,-0.28,0.02,-0.22\n1990-10-01,0.13,-0.73,-0.01,-0.13\n1990-11-01,0.08,-0.55,0.02,-0.01\n1990-12-01,0.32,-0.44,-0.12,-0.04\n1991-01-01,0.51,-0.09,-0.39,-0.21\n1991-02-01,0.32,-0.10,-0.29,-0.20\n1991-03-01,0.10,0.00,-0.37,-0.21\n1991-04-01,0.21,-0.68,-0.54,-0.00\n1991-05-01,0.44,-0.01,-0.62,0.14\n1991-06-01,0.65,0.24,-0.67,0.13\n1991-07-01,0.62,0.60,-0.67,-0.20\n1991-08-01,0.43,0.00,-0.62,-0.52\n1991-09-01,0.22,0.16,-0.52,-0.51\n1991-10-01,0.91,0.42,-0.53,-0.59\n1991-11-01,1.14,0.64,-0.46,-0.52\n1991-12-01,1.62,0.45,-0.47,-0.34\n1992-01-01,1.61,0.36,-0.24,-0.61\n1992-02-01,1.62,0.44,-0.14,-0.53\n1992-03-01,1.50,0.92,-0.34,-0.74\n1992-04-01,1.41,1.72,-0.44,-0.88\n1992-05-01,1.27,1.83,-0.39,-1.01\n1992-06-01,0.53,0.71,-0.36,-1.11\n1992-07-01,0.27,-0.05,-0.29,-0.86\n1992-08-01,-0.15,-0.24,-0.56,-0.86\n1992-09-01,-0.14,-0.50,-0.58,-0.70\n1992-10-01,-0.35,-0.26,-0.48,-0.74\n1992-11-01,-0.14,-0.25,-0.61,-0.97\n1992-12-01,0.04,-0.31,-0.52,-0.63\n1993-01-01,0.28,0.03,-0.50,-0.59\n1993-02-01,0.42,0.20,-0.43,-0.39\n1993-03-01,0.47,0.53,-0.38,-0.44\n1993-04-01,0.92,0.83,-0.22,-0.11\n1993-05-01,0.93,1.13,-0.25,-0.32\n1993-06-01,0.64,0.88,-0.18,-0.34\n1993-07-01,0.33,0.32,-0.44,-0.21\n1993-08-01,0.16,0.21,-0.45,-0.16\n1993-09-01,0.26,0.12,-0.50,0.30\n1993-10-01,0.36,0.32,-0.56,0.47\n1993-11-01,0.28,-0.23,-0.65,0.57\n1993-12-01,0.19,-0.29,-0.70,0.24\n1994-01-01,0.04,-0.07,-0.83,-0.01\n1994-02-01,-0.16,-0.33,-0.78,-0.14\n1994-03-01,-0.06,-1.13,-0.76,-0.10\n1994-04-01,0.03,-1.22,-0.62,-0.13\n1994-05-01,0.14,-0.91,-0.70,-0.23\n1994-06-01,0.27,-0.59,-0.67,-0.49\n1994-07-01,0.16,-0.82,-0.67,-0.52\n1994-08-01,0.52,-0.90,-0.63,-0.37\n1994-09-01,0.40,-0.05,-0.62,-0.30\n1994-10-01,0.90,0.65,-0.50,-0.27\n1994-11-01,1.14,0.59,-0.34,-0.11\n1994-12-01,1.21,0.68,-0.42,-0.07\n1995-01-01,1.10,0.91,-0.30,0.07\n1995-02-01,0.87,0.24,-0.22,0.34\n1995-03-01,0.49,-0.57,-0.21,0.39\n1995-04-01,0.27,-0.99,-0.04,0.29\n1995-05-01,0.03,-1.01,0.10,0.24\n1995-06-01,0.08,-0.53,0.23,0.33\n1995-07-01,0.03,-0.25,0.31,0.09\n1995-08-01,-0.38,-0.52,0.25,-0.26\n1995-09-01,-0.57,-0.28,0.03,-0.19\n1995-10-01,-0.72,-0.41,-0.06,-0.27\n1995-11-01,-0.78,-0.26,0.20,-0.06\n1995-12-01,-0.72,-0.69,0.17,-0.26\n1996-01-01,-0.65,-0.61,0.49,-0.15\n1996-02-01,-0.66,-0.32,0.22,0.16\n1996-03-01,-0.48,-0.07,0.07,0.12\n1996-04-01,-0.34,-1.28,0.15,0.34\n1996-05-01,-0.40,-0.81,0.16,0.43\n1996-06-01,-0.12,-1.07,-0.06,0.56\n1996-07-01,-0.14,-0.98,-0.15,0.41\n1996-08-01,-0.30,-0.46,-0.36,0.15\n1996-09-01,-0.34,-0.53,-0.22,-0.01\n1996-10-01,-0.28,-0.80,-0.34,-0.10\n1996-11-01,-0.30,-0.79,-0.28,-0.06\n1996-12-01,-0.43,-1.03,-0.15,-0.33\n1997-01-01,-0.43,-0.56,0.06,-0.55\n1997-02-01,-0.24,-0.03,0.16,-0.60\n1997-03-01,-0.06,0.82,-0.00,-0.82\n1997-04-01,0.34,0.98,0.12,-0.81\n1997-05-01,0.87,2.16,0.23,-0.89\n1997-06-01,1.15,3.17,0.29,-0.98\n1997-07-01,1.60,3.63,0.06,-0.53\n1997-08-01,1.94,3.98,-0.17,-0.29\n1997-09-01,2.10,4.05,-0.12,0.06\n1997-10-01,2.29,4.00,0.16,0.47\n1997-11-01,2.42,4.12,0.27,0.60\n1997-12-01,2.30,4.27,0.15,0.76\n1998-01-01,2.42,3.55,0.19,0.38\n1998-02-01,2.08,2.88,0.59,0.47\n1998-03-01,1.49,2.88,0.59,0.34\n1998-04-01,0.90,3.00,0.43,0.23\n1998-05-01,0.68,3.01,0.39,0.20\n1998-06-01,-0.39,2.00,0.42,0.35\n1998-07-01,-0.73,1.31,0.29,0.59\n1998-08-01,-0.83,0.94,0.33,0.40\n1998-09-01,-0.82,0.31,0.13,0.26\n1998-10-01,-1.19,0.12,0.07,-0.01\n1998-11-01,-1.23,-0.21,0.12,0.14\n1998-12-01,-1.51,-0.14,0.00,-0.15\n1999-01-01,-1.53,-0.23,-0.35,-0.20\n1999-02-01,-1.41,0.05,-0.38,-0.22\n1999-03-01,-0.92,0.30,-0.38,-0.02\n1999-04-01,-0.81,-0.78,-0.28,0.14\n1999-05-01,-0.87,-0.43,-0.09,0.18\n1999-06-01,-0.95,-0.55,-0.10,0.18\n1999-07-01,-0.84,-0.65,-0.10,0.38\n1999-08-01,-0.98,-0.51,0.01,0.35\n1999-09-01,-0.84,-0.97,0.01,0.09\n1999-10-01,-1.03,-0.76,-0.18,0.05\n1999-11-01,-1.41,-1.11,-0.18,0.15\n1999-12-01,-1.54,-1.06,-0.02,0.10\n2000-01-01,-1.79,-0.89,-0.42,-0.12\n2000-02-01,-1.53,-0.47,-0.28,0.12\n2000-03-01,-1.26,-0.56,-0.09,0.05\n2000-04-01,-0.80,0.46,-0.17,0.06\n2000-05-01,-0.80,-0.29,-0.27,-0.05\n2000-06-01,-0.75,-0.65,-0.40,-0.11\n2000-07-01,-0.57,-0.57,-0.27,-0.10\n2000-08-01,-0.36,-0.59,-0.19,-0.00\n2000-09-01,-0.39,-0.34,-0.36,-0.13\n2000-10-01,-0.55,-0.49,-0.42,-0.12\n2000-11-01,-0.75,-0.70,-0.43,-0.11\n2000-12-01,-0.92,-0.55,-0.49,-0.27\n2001-01-01,-0.88,-0.76,-0.38,-0.17\n2001-02-01,-0.63,-0.33,-0.32,-0.10\n2001-03-01,-0.48,0.35,-0.01,-0.06\n2001-04-01,-0.30,0.42,-0.11,0.07\n2001-05-01,-0.30,-0.64,-0.14,0.12\n2001-06-01,-0.11,-0.86,-0.01,0.04\n2001-07-01,0.01,-0.70,0.02,-0.11\n2001-08-01,-0.07,-0.77,0.00,-0.06\n2001-09-01,-0.28,-1.20,0.13,-0.04\n2001-10-01,-0.26,-1.14,0.06,-0.03\n2001-11-01,-0.28,-0.94,0.31,-0.02\n2001-12-01,-0.46,-1.12,0.37,-0.25\n2002-01-01,-0.14,-0.94,0.53,-0.19\n2002-02-01,0.00,-0.30,0.35,-0.33\n2002-03-01,0.11,0.62,0.21,-0.06\n2002-04-01,0.14,0.33,-0.18,-0.00\n2002-05-01,0.21,0.18,-0.40,0.15\n2002-06-01,0.68,-0.40,-0.38,0.02\n2002-07-01,0.57,-0.44,-0.33,0.22\n2002-08-01,0.70,-0.26,-0.38,-0.19\n2002-09-01,0.82,0.04,-0.31,-0.23\n2002-10-01,1.16,0.57,-0.07,-0.09\n2002-11-01,1.41,0.78,-0.09,-0.25\n2002-12-01,1.41,0.69,-0.09,0.08\n2003-01-01,0.98,0.10,0.02,0.22\n2003-02-01,0.64,-0.21,-0.17,0.30\n2003-03-01,0.48,-0.49,-0.11,0.28\n2003-04-01,-0.03,-0.90,-0.15,0.11\n2003-05-01,-0.52,-1.35,-0.14,0.02\n2003-06-01,-0.19,-0.91,-0.02,-0.21\n2003-07-01,0.14,-0.58,0.09,0.14\n2003-08-01,0.05,0.08,0.26,0.30\n2003-09-01,0.15,-0.43,0.28,0.34\n2003-10-01,0.46,-0.07,0.38,0.31\n2003-11-01,0.39,0.29,0.17,0.36\n2003-12-01,0.32,0.31,0.20,0.30\n2004-01-01,0.26,-0.03,0.25,0.07\n2004-02-01,0.17,-0.20,0.34,-0.05\n2004-03-01,-0.10,-0.34,0.20,-0.36\n2004-04-01,0.06,-0.45,0.30,-0.57\n2004-05-01,0.10,-0.96,-0.03,-0.47\n2004-06-01,0.14,-0.69,0.02,-0.39\n2004-07-01,0.41,-0.70,0.18,-0.18\n2004-08-01,0.66,-0.44,0.35,0.02\n2004-09-01,0.67,-0.10,0.21,0.18\n2004-10-01,0.73,0.11,0.20,-0.08\n2004-11-01,0.62,0.39,0.32,0.06\n2004-12-01,0.71,0.27,0.24,0.17\n2005-01-01,0.56,0.03,0.22,0.28\n2005-02-01,0.26,-0.61,0.20,0.22\n2005-03-01,0.28,-1.15,0.53,0.26\n2005-04-01,0.28,-0.36,0.65,-0.02\n2005-05-01,0.30,0.22,0.81,-0.40\n2005-06-01,0.22,-0.15,0.67,-0.57\n2005-07-01,-0.01,-0.06,0.57,-0.57\n2005-08-01,-0.04,-0.30,0.41,-0.19\n2005-09-01,-0.08,-0.61,0.35,-0.23\n2005-10-01,-0.15,-1.05,0.12,-0.22\n2005-11-01,-0.44,-1.41,0.09,-0.26\n2005-12-01,-0.75,-1.19,0.26,-0.29\n2006-01-01,-0.98,-0.69,0.16,-0.30\n2006-02-01,-0.71,0.33,0.01,-0.04\n2006-03-01,-0.73,-0.27,0.02,0.06\n2006-04-01,-0.30,-0.89,0.26,-0.00\n2006-05-01,-0.11,-0.39,0.23,-0.11\n2006-06-01,0.09,-0.19,0.28,0.10\n2006-07-01,0.03,0.18,0.09,0.05\n2006-08-01,0.37,0.51,0.20,0.13\n2006-09-01,0.62,0.78,0.31,0.13\n2006-10-01,0.76,1.20,0.31,0.01\n2006-11-01,0.98,0.91,0.40,-0.10\n2006-12-01,1.10,0.62,0.18,0.05\n2007-01-01,0.59,0.57,0.21,-0.09\n2007-02-01,0.12,0.06,0.38,-0.06\n2007-03-01,-0.15,-0.65,0.17,-0.31\n2007-04-01,-0.16,-1.05,0.24,-0.07\n2007-05-01,-0.39,-1.47,0.01,-0.06\n2007-06-01,-0.16,-1.40,-0.08,0.12\n2007-07-01,-0.37,-0.94,-0.18,-0.02\n2007-08-01,-0.57,-1.36,-0.33,-0.04\n2007-09-01,-1.04,-1.34,-0.27,0.12\n2007-10-01,-1.41,-1.65,-0.19,-0.11\n2007-11-01,-1.58,-1.95,-0.07,-0.08\n2007-12-01,-1.61,-1.63,-0.21,-0.06\n2008-01-01,-1.79,-0.80,-0.10,-0.14\n2008-02-01,-1.70,-0.16,-0.00,0.09\n2008-03-01,-1.17,0.33,0.12,0.25\n2008-04-01,-0.89,0.50,-0.06,0.33\n2008-05-01,-0.64,-0.32,0.07,0.47\n2008-06-01,-0.44,0.26,0.15,0.39\n2008-07-01,-0.04,0.47,0.17,0.40\n2008-08-01,-0.04,0.67,0.12,0.31\n2008-09-01,-0.28,0.32,0.20,0.13\n2008-10-01,-0.30,-0.17,0.01,0.05\n2008-11-01,-0.37,-0.21,-0.04,0.13\n2008-12-01,-0.90,-0.33,-0.03,0.15\n2009-01-01,-1.00,-0.17,-0.17,0.11\n2009-02-01,-0.71,-0.31,-0.55,0.08\n2009-03-01,-0.72,-0.85,-0.49,0.20\n2009-04-01,-0.25,0.00,-0.47,0.21\n2009-05-01,0.17,0.17,-0.43,0.44\n2009-06-01,0.49,0.54,-0.19,0.28\n2009-07-01,0.69,0.70,0.17,0.14\n2009-08-01,0.62,0.66,0.07,0.04\n2009-09-01,0.68,0.22,-0.03,0.04\n2009-10-01,0.96,0.22,0.20,0.23\n2009-11-01,1.49,0.14,0.20,0.11\n2009-12-01,1.81,0.49,0.25,0.43\n2010-01-01,1.43,0.36,0.55,0.46\n2010-02-01,1.18,0.04,0.90,0.41\n2010-03-01,1.07,-0.04,1.12,0.54\n2010-04-01,0.56,0.51,1.25,0.37\n2010-05-01,-0.15,0.39,1.15,0.27\n2010-06-01,-0.62,0.14,0.87,0.54\n2010-07-01,-0.89,-1.05,0.74,0.35\n2010-08-01,-1.33,-1.23,0.63,0.29\n2010-09-01,-1.56,-1.29,0.37,0.08\n2010-10-01,-1.65,-1.53,0.35,0.21\n2010-11-01,-1.57,-1.39,0.39,0.22\n2010-12-01,-1.63,-1.03,0.67,0.15\n2011-01-01,-1.70,-0.61,0.73,0.12\n2011-02-01,-1.26,0.07,0.35,0.32\n2011-03-01,-0.98,-0.43,0.25,0.38\n2011-04-01,-0.74,-0.05,0.17,0.18\n2011-05-01,-0.53,0.25,0.20,-0.04\n2011-06-01,-0.25,0.36,0.35,-0.11\n2011-07-01,-0.23,0.27,0.15,0.02\n2011-08-01,-0.66,-0.20,0.15,0.00\n2011-09-01,-0.76,-1.01,-0.01,0.09\n2011-10-01,-0.93,-0.71,0.02,0.06\n2011-11-01,-1.09,-0.80,-0.10,-0.23\n2011-12-01,-1.05,-0.68,-0.06,-0.74\n2012-01-01,-0.93,-0.39,-0.09,-0.66\n2012-02-01,-0.61,0.40,-0.19,-0.78\n2012-03-01,-0.48,-0.02,-0.41,-0.43\n2012-04-01,-0.29,0.48,-0.33,-0.64\n2012-05-01,-0.18,0.43,-0.11,-0.38\n2012-06-01,0.14,0.99,0.07,-0.27\n2012-07-01,0.44,0.82,0.02,-0.29\n2012-08-01,0.66,0.27,0.19,-0.28\n2012-09-01,0.44,0.04,0.34,-0.13\n2012-10-01,0.23,-0.39,0.35,-0.19\n2012-11-01,0.33,-0.46,0.44,-0.16\n2012-12-01,-0.13,-0.61,0.26,-0.09\n2013-01-01,-0.42,-0.70,0.19,0.05\n2013-02-01,-0.40,-0.32,0.22,-0.02\n2013-03-01,-0.14,-0.38,0.55,-0.09\n2013-04-01,-0.08,-1.32,0.37,0.04\n2013-05-01,-0.28,-1.44,0.44,0.13\n2013-06-01,-0.33,-1.44,0.07,-0.03\n2013-07-01,-0.28,-1.36,0.06,-0.03\n2013-08-01,-0.29,-0.76,0.22,0.05\n2013-09-01,-0.09,-0.77,0.23,-0.12\n2013-10-01,-0.24,-0.16,0.23,-0.18\n2013-11-01,-0.02,-0.50,0.02,-0.26\n2013-12-01,-0.09,-0.25,-0.02,-0.16\n2014-01-01,-0.42,0.37,-0.12,-0.23\n2014-02-01,-0.45,-0.67,-0.25,-0.05\n2014-03-01,-0.07,-0.51,-0.50,-0.13\n2014-04-01,0.28,-0.25,-0.36,-0.08\n2014-05-01,0.45,0.74,-0.27,0.14\n2014-06-01,0.48,1.17,-0.29,0.15\n2014-07-01,0.13,1.04,-0.13,-0.09\n2014-08-01,0.14,0.67,-0.10,0.05\n2014-09-01,0.37,0.15,0.13,0.11\n2014-10-01,0.48,0.34,0.13,-0.06\n2014-11-01,0.89,0.57,0.04,-0.50\n2014-12-01,0.77,0.14,0.05,-0.38\n2015-01-01,0.59,-0.16,-0.08,0.10\n2015-02-01,0.57,-0.62,-0.06,0.15\n2015-03-01,0.48,-0.01,-0.27,-0.11\n2015-04-01,0.90,0.64,-0.40,0.05\n2015-05-01,1.04,1.59,-0.13,0.11\n2015-06-01,1.28,2.19,-0.31,0.07\n2015-07-01,1.56,2.32,-0.12,-0.19\n2015-08-01,1.87,2.08,0.09,-0.31\n2015-09-01,2.01,2.48,0.35,-0.19\n2015-10-01,2.21,2.38,0.51,0.06\n2015-11-01,2.57,2.21,0.18,0.19\n2015-12-01,2.56,2.09,0.13,0.44\n2016-01-01,2.56,1.82,0.25,0.47\n2016-02-01,2.11,1.09,0.28,0.32\n2016-03-01,1.60,1.10,0.24,0.25\n2016-04-01,1.05,0.29,0.17,0.08\n2016-05-01,0.45,0.45,0.18,0.01\n2016-06-01,0.06,0.52,0.15,0.37\n2016-07-01,-0.25,-0.06,0.10,0.37\n2016-08-01,-0.48,0.19,0.15,0.33\n2016-09-01,-0.46,0.25,0.01,0.11\n2016-10-01,-0.75,-0.03,0.12,0.24\n2016-11-01,-0.63,-0.08,0.15,0.10\n2016-12-01,-0.51,0.02,0.23,0.41\n2017-01-01,-0.34,0.33,0.24,0.25\n2017-02-01,-0.01,0.93,0.06,-0.18\n2017-03-01,-0.09,1.56,-0.03,-0.06\n2017-04-01,0.22,0.49,0.31,0.09\n2017-05-01,0.30,0.25,0.38,0.12\n2017-06-01,0.22,0.20,0.35,0.26\n2017-07-01,0.22,-0.46,0.32,0.10\n2017-08-01,-0.18,-0.76,0.33,0.39\n2017-09-01,-0.56,-1.28,0.15,0.12\n2017-10-01,-0.52,-1.32,0.03,0.19\n2017-11-01,-0.84,-1.44,0.22,0.02\n2017-12-01,-0.85,-1.54,0.25,-0.09\n2018-01-01,-0.86,-1.25,0.03,0.29\n2018-02-01,-0.73,-0.43,-0.27,-0.27\n2018-03-01,-0.73,-1.30,-0.13,-0.18\n2018-04-01,-0.36,-0.81,-0.33,-0.03\n2018-05-01,-0.12,-0.46,-0.60,-0.07\n2018-06-01,0.12,-0.47,-0.72,0.15\n2018-07-01,0.27,-0.08,-0.48,0.12\n2018-08-01,0.05,-0.54,-0.32,0.21\n2018-09-01,0.30,0.12,-0.04,0.21\n2018-10-01,0.84,0.30,-0.13,0.31\n2018-11-01,1.00,0.69,-0.32,0.21\n2018-12-01,0.97,0.79,-0.33,0.40\n2019-01-01,0.51,0.67,-0.17,0.41\n2019-02-01,0.62,0.66,-0.04,0.41\n2019-03-01,0.81,-0.18,-0.15,0.67\n2019-04-01,0.67,0.36,-0.17,0.34\n2019-05-01,0.61,0.24,-0.08,0.31\n2019-06-01,0.66,-0.11,0.06,0.27\n2019-07-01,0.41,-0.36,0.09,0.24\n2019-08-01,0.19,-0.29,0.09,0.22\n2019-09-01,0.11,-0.68,0.20,0.33\n2019-10-01,0.55,-0.71,0.01,0.25\n2019-11-01,0.74,0.02,-0.05,0.87\n2019-12-01,0.51,-0.19,-0.04,0.80\n2020-01-01,0.64,-0.12,0.01,0.95\n2020-02-01,0.48,-0.03,0.32,0.57\n2020-03-01,0.36,-0.44,0.58,0.31\n2020-04-01,0.49,0.03,0.46,0.41\n2020-05-01,-0.12,-0.24,0.32,0.53\n2020-06-01,-0.21,-0.43,0.47,0.29\n2020-07-01,-0.04,-1.09,0.38,0.29\n2020-08-01,-0.42,-0.43,0.25,0.30\n2020-09-01,-0.66,-0.85,0.15,0.09\n2020-10-01,-1.19,-0.99,0.19,0.17\n2020-11-01,-1.01,-0.87,0.08,-0.06\n2020-12-01,-0.98,-0.76,0.17,-0.38\n2021-01-01,-1.04,-0.58,0.27,-0.12\n2021-02-01,-0.94,-0.74,0.23,-0.33\n2021-03-01,-0.72,-0.50,-0.01,-0.21\n2021-04-01,-0.55,-0.96,0.16,0.06\n2021-05-01,-0.41,-0.73,0.04,0.41\n2021-06-01,-0.06,0.08,-0.10,0.58\n2021-07-01,-0.20,-0.14,-0.02,0.86\n2021-08-01,-0.38,-0.17,-0.07,0.72\n2021-09-01,-0.50,-0.29,0.16,0.43\n2021-10-01,-0.78,-0.60,0.03,0.42\n2021-11-01,-0.88,-0.98,0.11,0.32\n2021-12-01,-1.07,-1.29,0.16,0.00\n2022-01-01,-0.77,-0.90,0.36,0.14\n2022-02-01,-0.68,-1.14,0.39,0.46\n2022-03-01,-0.84,-0.59,-0.07,0.22\n2022-04-01,-0.90,-1.04,-0.11,0.15\n2022-05-01,-1.02,-1.04,0.14,0.01\n2022-06-01,-0.77,-0.82,0.23,0.30\n2022-07-01,-0.56,-0.54,0.05,0.08\n2022-08-01,-0.96,-0.23,0.14,0.18\n2022-09-01,-1.06,-0.77,0.25,0.20\n2022-10-01,-0.99,-1.37,0.08,0.30\n2022-11-01,-0.76,-0.92,0.02,0.33\n2022-12-01,-0.86,-0.67,0.18,0.34\n2023-01-01,-0.78,-0.35,0.23,0.49\n2023-02-01,-0.62,-0.14,0.04,0.28\n2023-03-01,-0.13,1.00,0.49,0.14\n2023-04-01,0.24,2.50,0.62,0.19\n2023-05-01,0.47,1.69,0.83,0.44\n2023-06-01,0.95,2.39,1.17,0.26\n2023-07-01,1.20,2.45,1.14,0.28\n2023-08-01,1.56,2.55,1.02,0.30\n2023-09-01,1.65,2.04,1.06,0.25\n2023-10-01,1.59,1.80,0.89,0.19\n2023-11-01,2.01,1.94,0.78,0.73\n2023-12-01,1.81,1.37,0.88,1.05\n2024-01-01,1.71,1.14,1.17,0.99\n2024-02-01,1.47,1.08,1.41,1.14\n2024-03-01,1.10,-0.06,1.21,1.33\n2024-04-01,0.93,-0.11,1.17,1.10\n2024-05-01,0.41,-0.41,1.22,0.82\n2024-06-01,0.25,-0.48,1.09,0.54\n2024-07-01,0.20,-0.44,0.87,0.33\n2024-08-01,0.02,-0.62,0.79,0.19\n2024-09-01,-0.11,-0.58,0.66,0.30\n2024-10-01,-0.24,-0.47,0.61,0.28\n2024-11-01,-0.05,-0.14,0.67,0.22\n', 'research/common/__init__.py': '"""Audited input loading and fixed-reference model helpers."""\n', 'research/common/baseline.py': '"""Fit and predict the fixed lagged hybrid, without changing its architecture."""\nfrom pathlib import Path\nimport gc\nimport numpy as np\nimport xarray as xr\nfrom .inputs import require, sha256, write_json\n\n\ndef fit(data, cutoff, folder):\n    folder = Path(folder)\n    folder.mkdir(parents=True, exist_ok=True)\n    m = data.lib\n    train = data.rain.sel(time=slice(None, cutoff))\n    X, y, ids, dates, climate, atmosphere, sums, counts, seas, cfs = m.amostrar_arvores_multissistema(\n        train, data.atmosphere, data.seas, data.cfs, cutoff)\n    models = []\n    for name, features in [(\'arvores_seas5\', m.FEATURES_SEAS5), (\'arvores_multissistema\', m.FEATURES_MULTISSISTEMA)]:\n        dataset = m.lgb.Dataset(np.ascontiguousarray(X[:, :len(features)]), label=y, feature_name=features)\n        model = m.lgb.train(m.PARAMETROS, dataset, num_boost_round=m.ARVORES)\n        require(model.feature_name() == features, \'Baseline feature order changed.\')\n        model.save_model(str(folder / (name + \'.txt\')))\n        models.append(model)\n        del dataset\n    ridge = m.mos_fit(train, data.seas, data.cfs, cutoff)\n    np.savez_compressed(folder / \'ridge.npz\', **ridge)\n    np.savez_compressed(folder / \'clima_atmosfera_indices.npz\', **atmosphere)\n    for name, field in [(\'chuva\', climate), (\'seas5\', seas), (\'cfsv2\', cfs)]:\n        field.to_netcdf(folder / (\'clima_\' + name + \'.nc\'))\n    audit = dict(training_start=str(dates[0].date()), training_end=str(dates[-1].date()),\n        training_months=len(dates), examples=len(y), parameters=m.PARAMETROS,\n        tree_count=m.ARVORES, weights=[.375, .375, .25], residual_scales=[.9, .875],\n        sample_hash=__import__(\'hashlib\').sha256(ids.tobytes()).hexdigest(), inputs=data.audit)\n    del X, y, ids\n    gc.collect()\n    write_json(folder / \'fit.json\', audit)\n    return dict(models=models, ridge=ridge, climate=climate, atmosphere=atmosphere, seas=seas, cfs=cfs, audit=audit)\n\n\ndef predict(data, state, dates):\n    m = data.lib\n    components = []\n    for model, name, scale in zip(state[\'models\'], m.NOMES_MODELOS, [.9, .875]):\n        anomalies = m.prever_par(model, name, data.atmosphere, data.seas, data.cfs,\n            state[\'atmosphere\'], state[\'climate\'], state[\'seas\'], state[\'cfs\'], dates)\n        components.append(m.reconstruir_chuva(state[\'climate\'], anomalies, scale).values.astype(\'float64\'))\n    ridge = m.mos_predict(state[\'ridge\'], data.seas, data.cfs, dates)\n    result = .75 * (.5 * components[0] + .5 * components[1]) + .25 * ridge\n    return xr.DataArray(result, dims=(\'time\', \'lat\', \'lon\'),\n        coords=dict(time=dates, lat=data.rain.lat, lon=data.rain.lon), attrs=dict(units=\'mm/day\'))\n', 'research/common/inputs.py': '"""Load the existing audited inputs; no downloads, imputation or new sources."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[2]\nLIBRARY = ROOT / \'research/lagged_sources/library.py\'\nREQUIRED = {\'numpy\': \'2.0.2\', \'pandas\': \'2.3.3\', \'xarray\': \'2025.12.0\', \'lightgbm\': \'4.6.0\'}\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise ValueError(message)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef write_json(path, obj):\n    Path(path).write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False, default=str) + \'\\n\', encoding=\'utf-8\')\n\n\ndef library(path=LIBRARY):\n    # Separate module globals: callers cannot change another run\'s index table.\n    spec = importlib.util.spec_from_file_location(\'fixed_lag_library\', path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef find_unique(filename, supplied=None):\n    if supplied:\n        p = Path(supplied)\n        require((p / filename).is_file(), f\'Missing {filename} in {p}\')\n        return p.resolve()\n    candidates = sorted({p.parent.resolve() for p in Path(\'/kaggle/input\').rglob(filename)})\n    require(len(candidates) == 1, f\'Attach exactly one input containing {filename}, or set its directory explicitly. Found {len(candidates)}.\')\n    return candidates[0]\n\n\ndef preflight(official=None, seas5=None, cfsv2=None, final=False):\n    paths = dict(official=find_unique(\'treino_tp.nc\', official),\n                 seas5=find_unique(\'seas5_51_manifesto.json\', seas5),\n                 cfsv2=find_unique(\'cfsv2_manifesto.json\', cfsv2))\n    expected = json.loads((ROOT / \'research/lagged_sources/official_hashes.json\').read_text())\n    files = {}\n    for row in expected:\n        if not row[\'nome\'].startswith(\'treino_\'):\n            continue\n        p = paths[\'official\'] / row[\'nome\']\n        require(p.is_file(), f\'Missing official input: {p.name}\')\n        require(sha256(p) == row[\'sha256\'], f\'Official input hash mismatch: {p.name}\')\n        files[p.name] = row[\'sha256\']\n    for source, name in [(\'seas5\', \'seas5_51_manifesto.json\'), (\'cfsv2\', \'cfsv2_manifesto.json\')]:\n        manifest = paths[source] / name\n        files[name] = sha256(manifest)\n        m = json.loads(manifest.read_text(encoding=\'utf-8\'))\n        for phase in [\'desenvolvimento\'] + ([\'somente_ajuste_final\'] if final else []):\n            row = m[\'arquivos\'][phase]\n            p = paths[source] / row[\'arquivo\']\n            require(p.is_file() and sha256(p) == row[\'sha256\'], f\'Missing or altered {source}: {phase}\')\n            files[p.name] = row[\'sha256\']\n    versions = {n: importlib.metadata.version(n) for n in REQUIRED}\n    require(versions == REQUIRED, f\'The fixed baseline needs {REQUIRED}; found {versions}.\')\n    return paths, dict(files=files, versions=versions, final_training=final)\n\n\nclass Inputs:\n    def __init__(self, official=None, seas5=None, cfsv2=None, output=\'outputs\', final=False):\n        self.paths, self.audit = preflight(official, seas5, cfsv2, final)\n        self.lib = m = library()\n        m.PASTA = self.paths[\'official\']\n        m.PASTA_SEAS5_MANUAL = self.paths[\'seas5\']\n        m.PASTA_CFSV2_MANUAL = self.paths[\'cfsv2\']\n        m.PASTA_SEAS5, m.MANIFESTO_SEAS5 = m.localizar_seas5()\n        m.PASTA_CFSV2, m.MANIFESTO_CFSV2 = m.localizar_cfsv2()\n        m.SAIDA = Path(output)\n        m.SAIDA.mkdir(parents=True, exist_ok=True)\n        end = \'2022-12-01\' if final else \'2020-12-01\'\n        start = \'1993-01-01\' if final else \'1976-10-01\'\n        with xr.open_dataset(m.PASTA / \'treino_tp.nc\') as ds:\n            require(ds.tp.attrs.get(\'units\') == \'mm/day\', \'Unexpected rainfall units.\')\n            self.rain = ds.tp.sel(time=slice(start, end)).transpose(\'time\', \'lat\', \'lon\').load()\n        require(pd.DatetimeIndex(self.rain.time.values).equals(pd.date_range(start, end, freq=\'MS\')), \'Incomplete rainfall calendar.\')\n        require(np.array_equal(self.rain.lat, np.arange(-60, 15.25, .25)) and\n                np.array_equal(self.rain.lon, np.arange(-90, -24.75, .25)), \'Different target grid.\')\n        require(np.isfinite(self.rain.values).all() and (self.rain.values >= 0).all(), \'Invalid rainfall.\')\n        atm_start = pd.Timestamp(start) - pd.DateOffset(months=4)\n        atm_end = pd.Timestamp(end) - pd.DateOffset(months=4)\n        self.atmosphere, _ = m.carregar_atmosfera(self.rain, atm_start, atm_end)\n        index_path = ROOT / (\'competition/metadata/NOAA/indices_noaa.csv\' if final else \'research/lagged_sources/ocean_indices.csv\')\n        m.INDICES_OC = pd.read_csv(index_path, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n        m.conferir_indices(m.INDICES_OC)\n        self.audit[\'indices_sha256\'] = sha256(index_path)\n        self.audit[\'library_sha256\'] = sha256(LIBRARY)\n        phases = [\'desenvolvimento\'] + ([\'somente_ajuste_final\'] if final else [])\n        self.seas = xr.concat([m.carregar_seas5(p, self.rain) for p in phases], dim=\'time\').sel(time=slice(start, end))\n        self.cfs = xr.concat([m.carregar_cfsv2(p, self.rain) for p in phases], dim=\'time\').sel(time=slice(start, end))\n        self.audit.update(rainfall_read=[start, end], forecast_partitions=phases,\n                          test_partition_read=False, sst_used=False)\n\n\nclass Context:\n    """Training-only climatologies, with a bounded reference window for inner selection."""\n    def __init__(self, data, reference, training):\n        self.data = data\n        m = data.lib\n        self.reference = pd.DatetimeIndex(reference)\n        self.training = pd.DatetimeIndex(training)\n        require(24 <= len(self.training) <= 360 and len(self.reference) <= 360, \'Invalid training window.\')\n        require(self.training.isin(self.reference).all() and self.reference.is_monotonic_increasing and\n                self.reference.is_unique and self.training.is_unique, \'Invalid training/reference dates.\')\n        self.cutoff = self.reference[-1]\n        rain = data.rain.sel(time=self.reference).values\n        self.counts = np.array([(self.reference.month == month).sum() for month in range(1, 13)])\n        require((self.counts > 1).all(), \'At least two reference years per month are needed.\')\n        self.sums = np.stack([rain[self.reference.month == month].sum(0, dtype=np.float64) for month in range(1, 13)])\n        self.climate = xr.DataArray((self.sums / self.counts[:, None, None]).astype(\'float32\'),\n            dims=(\'month\', \'lat\', \'lon\'), coords=dict(month=np.arange(1, 13), lat=data.rain.lat, lon=data.rain.lon), attrs=dict(units=\'mm/day\'))\n        origins = m.origem_mensal(self.reference, 4)\n        self.atm = m.ajustar_clima_atmosfera(data.atmosphere, origins)\n        origins = m.origem_mensal(self.reference, 3)\n        vals = m.INDICES_OC.loc[origins, m.INDICES_NOMES].to_numpy(dtype=np.float64)\n        self.atm[\'_clima_indices\'] = np.stack([vals[origins.month == month].mean(0) for month in range(1, 13)])\n        self.seasonal = []\n        for forecasts in [data.seas, data.cfs]:\n            v = forecasts.sel(time=self.training)\n            require(pd.DatetimeIndex(v.time_origem.values).equals(m.origem_mensal(self.training, 1)), \'Wrong seasonal initialization.\')\n            means = np.stack([v.values[self.training.month == month].mean(0, dtype=np.float64) for month in range(1, 13)]).astype(\'float32\')\n            require(np.isfinite(means).all(), \'Missing seasonal reference month.\')\n            self.seasonal.append(xr.DataArray(means, dims=self.climate.dims, coords=self.climate.coords))\n\n    def features(self, target, training=False):\n        d, m = self.data, self.data.lib\n        t = pd.Timestamp(target)\n        require(t in self.training if training else t > self.cutoff, \'Month crosses the training boundary.\')\n        x = np.column_stack([m.matriz_mes(d.atmosphere, self.atm, self.climate, t),\n            m.valores_seas5(d.seas, t), m.valores_anomalia_seas5(d.seas, self.seasonal[0], t),\n            m.valores_cfsv2(d.cfs, t), m.valores_anomalia_cfsv2(d.cfs, self.seasonal[1], t)]).astype(\'float32\')\n        if training:\n            # Exclude the entire target map, including neighbors seen by convolutions.\n            y = d.rain.sel(time=t).values\n            month = t.month - 1\n            x[:, 22] = ((self.sums[month] - y.astype(\'float64\')) / (self.counts[month] - 1)).ravel().astype(\'float32\')\n        require(np.isfinite(x).all(), \'Nonfinite features.\')\n        return x.T.reshape(31, d.rain.sizes[\'lat\'], d.rain.sizes[\'lon\'])\n\n    def save(self, folder):\n        folder = Path(folder)\n        folder.mkdir(parents=True, exist_ok=True)\n        self.climate.to_netcdf(folder / \'rainfall_climatology.nc\')\n        np.savez_compressed(folder / \'references.npz\', sums=self.sums, counts=self.counts,\n                            seas5=self.seasonal[0].values, cfsv2=self.seasonal[1].values, **self.atm)\n        write_json(folder / \'context.json\', dict(reference=self.reference.strftime(\'%Y-%m\').tolist(),\n                                                training=self.training.strftime(\'%Y-%m\').tolist(), features=self.data.lib.FEATURES_MULTISSISTEMA))\n', 'research/common/synthetic_fixture.py': '"""Small synthetic fields for tests; never operational inputs."""\nfrom types import SimpleNamespace\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom .inputs import library\n\n\ndef fixture():\n    m = library()\n    dates = pd.date_range(\'1976-06-01\', \'2009-01-01\', freq=\'MS\')\n    rng = np.random.default_rng(71)\n    coords = dict(time=dates, lat=np.linspace(-60, 15, 9), lon=np.linspace(-90, -25, 13))\n    def field():\n        return xr.DataArray(rng.uniform(1, 5, (len(dates), 9, 13)).astype(\'float32\'), dims=(\'time\', \'lat\', \'lon\'), coords=coords)\n    rain, seas, cfs = field(), field(), field()\n    origins = m.origem_mensal(dates, 1)\n    seas = seas.assign_coords(time_origem=(\'time\', origins.values))\n    cfs = cfs.assign_coords(time_origem=(\'time\', origins.values), inicializacao_mais_recente=(\'time\', origins.values))\n    atmosphere = {name: field() for name in m.VARIAVEIS}\n    m.INDICES_OC = pd.DataFrame(rng.normal(size=(len(dates), 4)), index=dates, columns=m.INDICES_NOMES)\n    return SimpleNamespace(lib=m, rain=rain, seas=seas, cfs=cfs, atmosphere=atmosphere)\n', 'research/lagged_sources/evidence/metricas_blocos.csv': 'modelo,bloco,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area,rmse,mae,vies,rmse_area\nclimatologia,A,1885464,6483793.032390847,247939.59247075464,2122255.6579590403,6038937.505138205,1619418.8409421304,1.854408671157335,1.125588002719246,0.1315005709314814,1.9310818134839693\nclimatologia,B,1885464,6485643.75706844,-150120.7514444294,2069800.4850491479,6102796.701143213,1619418.8409421304,1.8546733121631138,1.0977671729872052,-0.07962005715538954,1.9412651476568235\nclimatologia,C,1885464,6436085.131536505,262635.45612728415,2068391.4041558888,6110980.556575299,1619418.8409421304,1.8475736837464716,1.0970198339272925,0.13929486647704975,1.9425663306667187\nclimatologia,H1,1885464,6401491.782505364,-174106.14684297366,2030454.079381687,6064846.86348311,1619418.8409421304,1.8426017268670747,1.0768988850392727,-0.09234127347060121,1.9352199205750888\nclimatologia,H2,1885464,7433339.993815827,-154987.7861661935,2219894.617458496,7032419.844942207,1619418.8409421304,1.9855593655436024,1.1773731121137798,-0.08220140303192927,2.08388045581921\nclimatologia,H3,1885464,6360026.256373299,-147269.18968771095,2081298.099301327,5988413.4375352375,1619418.8409421304,1.83662432487597,1.1038652020411566,-0.07810766457896356,1.9229867607508768\nclimatologia,H4,1885464,5602144.191952371,46220.0596004611,1933057.2457860655,5254510.271905689,1619418.8409421304,1.7237251533320521,1.0252421927897142,0.0245138913288512,1.8013019900613414\ncontrole_defasado,A,1885464,5718423.53498111,23207.31116099255,1976427.6625193635,5295033.650443283,1619418.8409421304,1.7415222874502265,1.0482447092701657,0.01230854111295286,1.8082345720332853\ncontrole_defasado,B,1885464,6124671.955066858,-89840.5353237101,2009301.7122752154,5761746.876286968,1619418.8409421304,1.802321626064037,1.0656802316433596,-0.04764903245233539,1.8862423597027211\ncontrole_defasado,C,1885464,5713691.500375254,306106.6676142241,1943420.5199247042,5403561.990782675,1619418.8409421304,1.7408015775162955,1.0307385979921675,0.16235084181624473,1.8266715938581932\ncontrole_defasado,H1,1885464,5717386.989668707,-63237.31423828187,1954637.501365696,5386647.97874958,1619418.8409421304,1.7413644424984764,1.0366877868607918,-0.033539390960676985,1.8238104662436954\ncontrole_defasado,H2,1885464,6103053.668604041,-86302.90089028547,2042798.715182023,5709538.535129083,1619418.8409421304,1.7991379822396205,1.0834461518130407,-0.0457727651603454,1.8776771029787227\ncontrole_defasado,H3,1885464,5727186.065267311,-42184.875250228695,1992655.7199189912,5364761.4835922215,1619418.8409421304,1.7428560729125753,1.056851639659517,-0.022373736783215536,1.8201015318382443\ncontrole_defasado,H4,1885464,5472373.142781409,17137.38105349867,1902131.8049236808,5129050.457443741,1619418.8409421304,1.7036435332466897,1.0088401607899598,0.009089211490380443,1.779667591749282\nsst_defasada,A,1885464,5715477.845253004,65086.18084917702,1977404.2384530364,5290436.898130808,1619418.8409421304,1.7410736808460576,1.048762659193194,0.03451998067805963,1.8074495145862108\nsst_defasada,B,1885464,6123432.431848174,-95188.47479395218,2011196.380176176,5757629.408905434,1619418.8409421304,1.8021392381286345,1.0666851131478385,-0.05048543742757866,1.8855682647115124\nsst_defasada,C,1885464,5756161.930511232,316289.7261935604,1949143.450793509,5444593.505302643,1619418.8409421304,1.7472593739308742,1.0337738884399326,0.1677516654752148,1.8335938199767292\nsst_defasada,H1,1885464,5659258.808561306,-78454.42295129443,1945213.9619282382,5327502.105798034,1619418.8409421304,1.7324896764703681,1.0316897919706969,-0.04161014103228406,1.8137700285402012\nsst_defasada,H2,1885464,6087308.589751015,-43385.96637334982,2046189.1119687043,5693281.347570842,1619418.8409421304,1.796815713347927,1.0852443281699913,-0.02301076359630829,1.8750019735273016\nsst_defasada,H3,1885464,5754459.463448089,9217.205489752872,2004233.325897917,5394042.853104605,1619418.8409421304,1.7470009663710935,1.06299209419958,0.004888560847490523,1.8250619146150826\nsst_defasada,H4,1885464,5451203.129562975,53674.234981407244,1907135.917188048,5107821.079312822,1619418.8409421304,1.7003450461446072,1.0114942089523045,0.028467387858589314,1.7759807093276283\n', 'research/lagged_sources/library.py': 'from pathlib import Path\nfrom datetime import datetime, timezone\nfrom time import perf_counter\nimport base64, gzip, io, json, hashlib, gc, platform, importlib.metadata\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport lightgbm as lgb\nfrom sklearn.decomposition import PCA\nfrom threadpoolctl import threadpool_limits\nfrom IPython.display import display, Markdown, Image, FileLink\nDATASET, SYSTEM, SCHEMA = (\'seasonal-monthly-single-levels\', \'51\', \'worcap_seas5_v1\')\nPASTA_SEAS5_MANUAL = PASTA_CFSV2_MANUAL = None\nPONTOS_POR_MES = 5000\nANOS_TREINO = 30\nARVORES = 300\nSEMENTE = 42\nCORTE_FINAL = \'2022-12-01\'\nFIM_DESENVOLVIMENTO = \'2020-12-01\'\nPARAMETROS = {\'objective\': \'regression\', \'metric\': \'rmse\', \'learning_rate\': 0.05, \'num_leaves\': 31, \'min_data_in_leaf\': 150, \'lambda_l2\': 10.0, \'feature_fraction\': 0.9, \'bagging_fraction\': 0.8, \'bagging_freq\': 1, \'num_threads\': 4, \'seed\': SEMENTE, \'deterministic\': True, \'force_col_wise\': True, \'verbosity\': -1}\nBLOCOS = [{\'nome\': \'H1\', \'corte\': \'2006-12-01\', \'inicio\': \'2007-01-01\'}, {\'nome\': \'H2\', \'corte\': \'2008-12-01\', \'inicio\': \'2009-01-01\'}, {\'nome\': \'H3\', \'corte\': \'2010-12-01\', \'inicio\': \'2011-01-01\'}, {\'nome\': \'H4\', \'corte\': \'2012-12-01\', \'inicio\': \'2013-01-01\'}, {\'nome\': \'A\', \'corte\': \'2014-12-01\', \'inicio\': \'2015-01-01\'}, {\'nome\': \'B\', \'corte\': \'2016-12-01\', \'inicio\': \'2017-01-01\'}, {\'nome\': \'C\', \'corte\': \'2018-12-01\', \'inicio\': \'2019-01-01\'}]\nVARIAVEIS = [\'cloud_cover\', \'geopotential_850\', \'rel_hum_850\', \'shum_850\', \'surface_pressure\', \'t2\', \'temperature_850\', \'u_850\', \'v_850\']\nINDICES_NOMES = [\'nino34\', \'nino12\', \'tna\', \'tsa\']\nFONTES_INDICES = {\'fonte\': \'NOAA Physical Sciences Laboratory\', \'captura_utc\': \'2026-09-21T23:00:40.933121+00:00\', \'unidade\': \'degC\', \'tipo\': \'monthly sea surface temperature anomaly indices\', \'interpretacao_temporal\': \'Retrospective series with observation month strictly T-1. These are not archived real-time vintages.\', \'inicio\': \'1976-12-01\', \'fim\': \'2024-11-01\', \'meses\': 576, \'colunas\': [\'time_origem\', \'nino34\', \'nino12\', \'tna\', \'tsa\'], \'fontes\': [{\'indice\': \'nino34\', \'url\': \'https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data\', \'arquivo_bruto\': \'nino34.data\', \'sha256_bruto\': \'46e501b91f624ef85b525d73fbb427f9d04d30411ba6e5c7cff579b35fb4ff01\', \'inicio_declarado\': 1870, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  NINA34\\n 5N-5S 170W-120W \\n HadISST \\n  Anomaly from 1981-2010\\n https://psl.noaa.gov/data/timeseries/month/\\n  units=degC\'}, {\'indice\': \'nino12\', \'url\': \'https://psl.noaa.gov/data/timeseries/month/data/nino12.long.anom.data\', \'arquivo_bruto\': \'nino12.data\', \'sha256_bruto\': \'b13ad5ad085cd13e70ec5d82662b509d522204e38a8eefd133f14be270e51928\', \'inicio_declarado\': 1870, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  NINA12\\n 0N-10S 270W-280E \\n HadISST \\n  Anomaly from 1981-2010\\n https://psl.noaa.gov/data/timeseries/month/\\n  units=degC\'}, {\'indice\': \'tna\', \'url\': \'https://psl.noaa.gov/data/correlation/tna.data\', \'arquivo_bruto\': \'tna.data\', \'sha256_bruto\': \'387c4125f377e57bcb683a849aefcd76da7d7f3cdcb2ac68ab086ceadf300e8b\', \'inicio_declarado\': 1948, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  TNAa\\n Tropical Northern Atlantic Index\\n SST Anom 5.5N-23.5N; 15W-57.5W\\n  Using HadISST1.1: Climo 1991-2020\\n  NOAA/PSL\\n  https://psl.noaa.gov/data/timeseries/month/\'}, {\'indice\': \'tsa\', \'url\': \'https://psl.noaa.gov/data/correlation/tsa.data\', \'arquivo_bruto\': \'tsa.data\', \'sha256_bruto\': \'700c0dffda0f19c1254ba482808b345eaa0538f5bf09fc3bc025f46a5fbe3a69\', \'inicio_declarado\': 1948, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  TSAa\\n  NOAA/PSL\\n  https://psl.noaa.gov/data/timeseries/month/\'}], \'sha256_csv\': \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\', \'transformacao_no_modelo\': \'Monthly recentering within each fold, using only the 360 training origins; no standardization or moving averages.\'}\nINDICES_CSV_GZIP_BASE64 = \'H4sIAAAAAAACCm1cy44tuY3cz7ecGuj9+JqBF4bRC7eBmf5/zElGBMkse1N90YpSSqL4CJKqv/7459//51//+8c//v7Pz59//PmvPuw/tX3++vNvn7/+72//Ve9eP7X9lPop/73G90ebn5/vP8vzc+wHsL+jABwDXBsywOwEcIZpv1xHIJqm6EDg//aDMfv3IGI8iB9fwYl/O2QSUrp9wQabAfslZOE7fX1/FMyBvVQCNhcyPz5Pw3oJOADg09PGlv2s+sblFLbPadMvW1dtQNTCA7v4PUMsWyeXUe1IK1eAM507nxik8kVgzH4bX8RxHBfLttUV++3asWoiJBdbXbctN0xHAMVSbPq9n3/a904hQFLpmz+/37AdX00hoeAg5ohl3EvIIgQSOzgJ+84+hGxBTlwe3MLeCTmEQBprxKkOrfYKgt/EinCAhEA0dor2m50nYzIkpBJS7cBGTOUInmt5jqP1tG5b63XRlOkAXMZxCdAMWNzxY52VAEoGJ4nLyL1NIgYRGxKW4vXCcdeW+zFlsw35DbuuK+VR6/0oTIXCaAmSyVeDpDDpmt9QFbtjhQt9joXjUpTz3ONGnX/WiXGpSW+P9cEFg6ZxgspjtJtbW2izVuCm64ZIqVLPxTrFJUHTVPwUhgCcondfPwRmwn4A3UVl6iMd8nHKASsrxa3caARMFyU/MGkFOS6btXWWWIDZtGecJgufhQ3AbepaglSDdxEQaOIiRKqBY6z49wYQEFcNDML00VRPQniauKu00XHzH0RzA/xTeXt7hdo/iEqJPL9ikHbdEONEK0US9qLH5d+dkK7LTdP8EdqMzoOQ5cIme/X7fTTHfO23PaKtMISTiBWW4EdWEj+nJnHD1cNO44xxw2pIB5/HTWyQ0SZE0oEVx2DrZkGAcOGUnoRzP9SFB6FzhV/lhT/mB4kIsxXCb3FLWsiGl6fFhs2sPBAJpybTiQVbvPBA3sKBf2rdHc8DkdLAh9bQ/SUEtWbN8OQG3pUAqs1tH+0Zerc1w05xzUreaWsrtF+3u5+FV9nayIUP/p5zk/WFw+EnIJX2aH2jIuztTusBVAL2p3MXAPgMDYCv+ek0IfuGQnWKpD33q32vfwQbpRLAGb73t/F2mDpUjXdsYj2b6BbCnbienaKoj7A74iaGI4vj08cHFR9fsRjhAVAQ6wHM5eZha4Idetafw/4xx5g/ceJOdapjT2a8h4LMgtMMXeM2qSAWUsnbzGQluxSkwhDjt9tyd/IgdLevuQHdfgFG6McpEbf1Yi6KCE0xe7oNjxbC3I/QDoYM1f3OGESMVxijIPjDEPRByHQ997XqxGCe9BnZrotwjT5L7vdBuOlKurHMSekz541QMPzEqkRILrADWIIhFudww3XS3TWd7YsIl8sJK2mxaO1ENCJwqDOpCbRgJsncOHc4qNIIcenW8E3TJD2JkGjuisgRUzV9R7K555kE7vHUMKIzhPOcfKWALyCFEAlnwwfOEIBFZw9E0lm41YMmMu3HpXMN0YLM+HdcbXaIQCE6IC4fmL7TCJfrmeFY4OJuuJ7CHbve0ABrO4+leBArxAOp4WRBBHAdV4gHC8GFOsUiKyIkHviLM3iCCmFWSGfU+MChzSJE0hk79rtHqM4K6cBgQ4GM/FathK6lriC8izECEccdnFzttgOpHGdsjJOgLzUsJ1BwfItH9qvnVTqFDJN+qYoE0N3fx/pAcy796AMQr69wF7jmk1E4AeSg3/XrfkEssEPbfct3l5V0CSdyOD6cCFdylCC5Z7uTv+Zlp4/jFEXnOb9ZYjDQyuFN/zy0C4TPk8MHw+PgE4zxfPv07vNxGL0quq9cGwRQH5tWH5OjYW5dx2/xqwnJyN7gt52+T2xf8aXdwuDu6zoJNH6BKCuYO8Or7TQMpCCYO311A0FS2JCI+7i8w9r9IWDKX077iUDRTpiARQDi9EvHLl7jnL2aQak0KyUU1il7NUZSedftpH2fl4jH3VfR8eZ64IS92SlVOmzzUnUSUYmA51jJmBZB5ECgAHeHO0MILsr+QFJaaW0Pn8XZn7XCOkY4CPMj0s7dOH0xm12IGPJ12XKPcFOJtx9zHzAxsGQwHjcMFMNVpbHSd/bLyg0x44SQ90CQNJZuMC74Ta79xERwQAsIdx3jdzTkCJn9nvJ+ogyEvPkEbgADvGe7Nzg8LVCPn+sQkVinq8tx73KDxSsbGLkEcww38fgV89t9L5pierrC+UzZnn+6weRLiRiXi9FH5DhwNTaJ5w/V7hZ3HDCEK7GxrjnoOpgQOX4yrQEg14ET3Yk+V04hHl+Ox0CYovDIncannAPZoJ1GjcSKckye07NQ6dYQCkOSuCNmH26NJFfxQ+v7NcXwDJOHc7IzRMycjPUEoQRfg51Mp5vLY8tbgyU2Z4lrvxZJgYzui0QG0rKst7o8UgIBpmVyF+7KbZoW7GBeAuSIx0c6PVb+hLyJLRJmZUT2/DZ35StpwfCcym3uyjHDiLDFEhm3uSs3T8cYD9ZFnyBPHM+Pne7FOQSQKLYnWjhdF6LyUgVjn89JbucsFf78BmOHzlB/riePbktMMeUd5npBPNOV8vMwckb8b0tJ4BkkEHRMu40k8EhFCYTym5Dm+VNPcpEi2Zl21xHoaOIuEHx3FTHbTdMUub/bo26yPYajOR4EjJQFwaG35rz49gixnignZ5MtyLuJvj+X7yQK1vWJ7RnSt0Fo+sTxaFiqijtqPvz20JDlZsuumUUBt0ciePkXLI9gCaXb3VzBpDM9O0VDb3dRoNSCK7OLYr07woWMsNrKKxLhfijZ/518SGLuJSU393LCdYeLo1g403L1qBOhbPCIiH2X8BHDBcI83g2TZNWDO37RD1CYtcNgDBcJ7iMYDOgD/MyI5HzyVVhp515eDIRRVRT17nibrZlS8JVnKrNlUTms94ig685gIOXj7KV80jjlerYbb7OyuJoz1CMX0KrC5DvDgWzkhulNmeK408UBgYU3avrCennK/mFeFs54RnI+ygegEpr/vKIoC9QbrQ4Rb46ebEXlHG6uaHVrZGUbD8rNFW6sKRrDG37G4yxMYsSDZMwQiaOv6bkiHKxF2jdT9OXBYPNg/a7fuS2rj9kPS6DdRNC7MQLbrPFGSDTT88LMCu2KEZubyPljRypV2ajh1CL2y3Qj5LePWBH4riSUEr7WjI/v9L7KhDMZ1sJJXCqwSsgy8P4XQt7hL3IfgBcuNlInnclLd264YjsVTLpbRhYhJhGapEWcD9uwChHJdMliIJY5jQhqiiUhLGUAQ30qx6erYqOlRxb4cpxU/bv2jpokzPA9HCdX/4ZYHfwtFVJukPU7niTxiQpF0wcu8txf9Ry0V9XZyN2ei/9+doCG1vAh2zPxw9LYTSVRHZDn4Z/ZVSxlDvueyMI/SfIJmqSc7j2eg/9aifb4UDOH+vbx2Grc1zDu/Akf/hwNCoiwMxxWJeQ8w7BuItk3VddRpSgKJywNdFNpfXuew3IWiEFSXf08H71Dnn5ofqmCEdkelUwejSfeUXVoXmCCopzI7+LCtCpY5f4jvSu11eWEOiZyTkXMpadCiCYZ1dk5i5eNCM/u2kYSIy1CyDqdbEuPLzWT853Yqm38ELGihvDzrjsLIYGclDc0RBfipCQzi/tMBxFwX1PcLQDczn1VQyJIOcrq3htSGSkfUc8nDj3EYraa1c+GNGgrpYRYYNmQemcGuhHyTt2P8HGOUNIEYTiCeRTbJxEjMiKy1mZhvqoPwPwPZRmRCEI84z7j0OFjvtEKIPvliNWOQ4MOiDuOVCBA0FkEuaGO7hAA/4aABnHPgcsx/sOxhT+fESDitmm5Uaxqv+f6RgVfSCq1I6zHLei0rYS4S09pFTF2QpJTl9bJmRAxws/5TlBpJmC+HOEarufYcC6115zvQvoFCEVa1aP70uJAUp0d5gfdTOW1zptcdkVCFVG8PhJOHSH+cD9VOhH15fZhL1ntJqK9GhzM1DGtYZcxldkZoQwnd9+QD4iWM0msMDIoAECpE89dtCSSKLDX4b0tYHW4p1Ffb9EYMdAdw/HlXucn9e1pm0HWZ8owmuNoApCDkN3ky6ddkoMgRZpodOMulYM3T44P0bdwBlEQM/qw10hF6ahFQUa0CcG2nGc8KPo9nodqvocg6IuE101PLwR073Fy9avgqgSMV+PGLR6N4t721EFnEW2f4QS1CleOG6SROT9NsjOttIYyuKXB8ZM60izPbxzBt3FzXhLhx0GIYuPigjC+FtKjqM3Pe6fWVXiBU1ocbqnTqzMPq68HO2/Lw1c1kmG85ZwofnWEdU/MvBYvpIH+YAWJmC+mJ7w9UN+YubXprtRnKsTKR7xuZKQvAeotDQulkjYBFMJSN6c3ZXKcUljbQwKqJ8Ylhd2l+eR4Gq8pl2miUL4ewzzGXRX9sjr5DM/I6K6PhND8IgcXZwXGhQhVCS4O22o3fDJOxPjISRxjWyyZagLl171WdcgEOb48zyoVYtJDG5ADT1RnvhHn1d8Ed1XDJCU2DsK/UhGuAxFtcpPlM0/FcyNRMB8RkF5SbSBacu8qXS6ynS8ikXGEgKDrasQCwufw5Jz2RkCPwF+tlyXuw3q57R9VKVc4ifW7RQ7anaKQFfmR641HiO40Q+RHFIBGaBdl8u5NrHTKHL7pQu8T/APDrg9PiRZmARECh7Ntv5Hs0trkG0zRUDQ4vrZoezeuNne42Mvx5nUYXrZ+wgUnzl2nJyzVjEjESDlHXiVvDARipmiyIg2cY6+dfIPNwXReBNg71AJByB0pGBDkpNC3MlztKTLaUaktdqn7iHZeTuKcA4yCvObGbdlBOuaJUis9+iFEhAF9L6uH48WWz7/RjhKWH+7uBO1A8FGX16YhvRO0A46qS5AIHFIB3dRiFrcoiMDOuwdLlnZ/5C9y2/vw3Pf8yF0kYm4JSROtrQX27ryMlYpWzX3ueUWzSsaF8uRW9xKMqfoFT43uKrr5x3yK9moi443oHiYmZl5KpGzq9vDmvuxU1Emmx2CJmCM1ecRkjNwDkR6HKCge0EgCZmqGr9uDmDE4vjxR7A1Q153bjSrg/chzb4+eotMdRcIlSRcNX4+VPUvY07hn0pc71fKRR7kRxF4F7JAj9yYzdXwD6BZ8frsWz6Hbfs2t2uGOxWEZuaNW/wuvx2GmpqiE40PCMgfHRwoKpiXzJ5pEMDzzW4l+1QbUNL5+XjVZ5fJ9+p3a9mgBN+IiAs6rQcRy7+pRBEB2aUY7ed+6XbWEWVqTeQhGXY1HEEYJxu96HNsIcJOkHK3U0RZZk0EqHjQgRmsEtMhxKHy2A9N4f3n74RFwPwRIAdipNv2GCjCj71TPGIrb75oINuZdHtfYNaupk93OOFKgRWs8r+R/Kw4oBNxXU6OHYrS5NbFrNAHv+opIag1RlH+z7KYstYYwSnonxvdLttVEr2+PRAz7uxchLborPSzQiyQgfpUyUukFmteSVbrOANmXopXMaP2P/A0ffwCxEk+/15OnUKBg2eZDTrw+wfVvmVJIaKwycPym3zf178NzUbW5adKTk09q8QOg5vcx1iY09LIEgHcDz0pEzqboqYDRghh5FFh7evBRcl1JAUrtKZYanohAxss/MlLgXsk03UjXRLNBUOxAhju7mlg2e9FGZLKF2K8p+npnh2rqZGcN2g2+b/XGqywlqRhRc6uRguIxB+IQUV/vK2akwoxf1NTIrmTI66nHFzJeQlEEQBrTiXg/w2FXzHTeXXNFfEdfLmKlqlletK/NV6c6EDM9ydwjPxPD+ErPyFQTcuubquH9o146JvUuASfpWOLWvoubSAhSM13+twbrRvpzRM6QJykNOc4TCjtPOC7avbUGVSee8Zl5RoSqzqFqMG9w2ZWeJUwCXkmo/AoIe5i5xOQdRnEE03t3LADwnuT0+2rEfQpcNfo4jafU6dW9rxNuSh+3MCPTy3tWPizHGWPXB1jdQ5Xt/DJDM6p7VqQ7YouQwfTq3tzAqJSh32Z9zpaHvJhHh3V5dW8+PPI0j7A2h/nble9xPeLmcPfKZlXzgIhEXd43BXbnqTHc/vW+/fYDkVrl8PK8ldp8I/5Kte4241aUDDg5eY8S3RSLqanQbZa9TXGEyu9H78GM2J9d1ADUVwGheEUftzdVuKe7e4swEIGm8nb30mM6v1TcVhtTajIHoIexq6ofJQ2N2rZZYouge4Ql+1fiaSq3CYO8c95JycepvFaNF+VwWGOJi2H/+/fzv728lUqf9waQ9e6J0AKiXtTctUEQnMFlcIZ7rtY8dtshgzNZ0PMbY1OcVCrSixTlAAVor2zS6Km5dROSE05VtDO5rvyefEWdVbSSkPl6qTpWvIQq+tDKHbCoKe7mBvH8biQ8oQCNgEiHe0spwo5KwE13Qp2vadyf+A/PJJtJ6RrX64wSz5Pz/P4447EFyGOo6v4dv7m91n3Wdq2JB+RipJ5G0DhdwklXjylTAobnmr0+uZXxr8GoVzzTLJ7xr8Go14pq33Lmd38lwxEqXbcbQam9ZU/jjePXC1w/YtbFDVdwakSB20vkMBvxdhyvbpqzpsP11dffRqjRGXYeCbToOl8jWk1MYpPj7eV0u/Tecs0tOs57JPunH2ALZj2uVA3BtYbfamAmEfN3ApzQ1Uh3bGlsK6/80g8dlxk2B5xXXJ5Yp47gvijfmU75OEPu/fghf4GP4SEEoVPXhNTRbHPLZewwj8ySPjajJXZdorbF6KYR4XMMf2hFQtOJeKeYmM6rXkFriWLjRuG9y1KM0RLFHjJyWedaotgl/vqIOeOpnaS/qOBcxjzi0Tfe7YJ1e7lta6/39fc22k0NcgYIj31oOXW9OEN9dwWg8lkV0LRMsbenVHASdi8SwcajMQsq0T43CHB6fbyMbtzSslItsWs0sCDixJ/TaESM1ytKxvVVRr619IKpEaHwuhKwXss8Hnp07WMnz6sXi56ZbKndHPVA+2W0LQjgWfH4CypMvgAQKY8bT4dOWkN0eizvTbIrgaub+DV6IpB48tpsS/R6H/8zL4iwLgHt9ZdTqqcCLEnQMrvuclnDi8et59f6DXRzNYUfLVrMx8Yzc7aSDf22HurPJ3S/H6bcLX5q3alDe152fwNgXhdf20nMYk48RNTxdX/Et57Jiz8TNC/QujcQzOez1j50blq5P89n5yGLTLtzODKvFN2xP2Vikw/PvG42QXNj93JYrQf2evLo6QQOdThtqCotWJ90pdyDOF9ljfgO0+LKlmizbFJl0bRxfOU/ZIS/wXKVem1Bm/nswFOz/v2TX12hAhdnl55+1+pmeXlPQRu/Exnmm5a3RrSR8hjR8rBoA/4ffzwna7xKAAA=\'\nINDICES_CSV_SHA256 = \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\'\nSCHEMA_CFSV2 = \'worcap_cfsv2_pentad_media_auditada_v1\'\nPOLITICA_CFSV2 = \'ensemble_completo_ou_excecao_201908_m17_auditada_v2\'\nFONTE_CFSV2 = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\nFEATURE_SEAS5 = \'seas5_tp_prevista_media_T_emitida_Tmenos1\'\nFEATURE_SEAS5_ANOM = \'seas5_tp_anomalia_mensal_treino_fold\'\nFEATURE_CFSV2 = \'cfsv2_tp_prevista_media_T_emitida_Tmenos1\'\nFEATURE_CFSV2_ANOM = \'cfsv2_tp_anomalia_mensal_treino_fold\'\nNOMES_MODELOS = [\'arvores_seas5\', \'arvores_multissistema\']\nFASES_CFSV2 = dict(desenvolvimento=(\'1982-02-01\', \'2020-12-01\', 467), somente_ajuste_final=(\'2021-01-01\', \'2022-12-01\', 24), teste=(\'2023-01-01\', \'2024-12-01\', 24))\nMOS_ALPHA = 0.1\nMOS_FRACAO = 0.25\n\ndef exigir(condicao, mensagem):\n    if not bool(condicao):\n        raise ValueError(mensagem)\n\ndef sha256(arquivo):\n    digest = hashlib.sha256()\n    with Path(arquivo).open(\'rb\') as stream:\n        for bloco in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(bloco)\n    return digest.hexdigest()\n\ndef salvar_json(nome, conteudo):\n    caminho = SAIDA / nome\n    caminho.write_text(json.dumps(conteudo, indent=2, ensure_ascii=False, default=str, allow_nan=False), encoding=\'utf-8\')\n    return caminho\n\ndef localizar_dados(pasta_manual=None):\n    obrigatorios = (\'treino_tp.nc\', \'treino_tp_alvo.nc\', \'teste_features.nc\', \'sample_submission.csv\')\n    if pasta_manual is not None:\n        candidatas = [Path(pasta_manual)]\n    elif Path(\'/kaggle/input\').is_dir():\n        candidatas = sorted({p.parent for p in Path(\'/kaggle/input\').rglob(obrigatorios[0])})\n    else:\n        candidatas = [Path.cwd() / \'data\' / \'raw\', Path.cwd().parent / \'data\' / \'raw\']\n    validas = sorted({p.resolve() for p in candidatas if all(((p / nome).is_file() for nome in obrigatorios))})\n    exigir(len(validas) == 1, f\'Exactly one folder containing the four official files is required. Attach the competition data or set PASTA_DADOS_MANUAL. Complete folders found: {validas}\')\n    return validas[0]\n\ndef conferir_campo(campo, referencia, datas):\n    exigir(set(campo.dims) == {\'time\', \'lat\', \'lon\'}, \'Unexpected atmospheric dimensions.\')\n    for eixo in (\'lat\', \'lon\'):\n        exigir(np.array_equal(campo[eixo].values, referencia[eixo].values), f\'Misaligned atmospheric grid: {eixo}.\')\n    exigir(pd.DatetimeIndex(campo.time.values).equals(pd.DatetimeIndex(datas)), \'Misaligned atmospheric dates.\')\n\ndef ajustar_clima_atmosfera(campos, origens):\n    origens = pd.DatetimeIndex(origens)\n    resultado = {}\n    for nome in VARIAVEIS:\n        campo = campos[nome]\n        indices = pd.DatetimeIndex(campo.time.values).get_indexer(origens)\n        exigir((indices >= 0).all(), \'Missing atmospheric fitting months.\')\n        mapas = []\n        for mes in range(1, 13):\n            ids = indices[origens.month == mes]\n            exigir(len(ids) > 0, \'Calendar month with no atmospheric history.\')\n            valores = campo.values[ids]\n            exigir(np.isfinite(valores).all(), f\'{nome}: nonfinite training values.\')\n            mapas.append(valores.mean(axis=0, dtype=np.float64).astype(np.float32))\n        resultado[nome] = np.stack(mapas)\n    return resultado\n\ndef reconstruir_chuva(clima_tp, anomalias, peso):\n    peso = float(peso)\n    exigir(np.isfinite(peso) and peso >= 0, \'The weight must be finite and nonnegative.\')\n    meses = pd.DatetimeIndex(anomalias.time.values).month.to_numpy() - 1\n    valores = np.maximum(0, clima_tp.values[meses] + peso * anomalias.values)\n    exigir(np.isfinite(valores).all(), \'Rainfall reconstruction produced invalid values.\')\n    return xr.DataArray(valores.astype(np.float32), dims=anomalias.dims, coords=anomalias.coords, name=\'tp_mm_day\', attrs={\'units\': \'mm/day\'})\n\ndef auditar_chuva_desenv():\n    with xr.open_dataset(PASTA / \'treino_tp.nc\') as ds, xr.open_dataset(PASTA / \'treino_tp_alvo.nc\') as da:\n        bruto, alvo = (ds.tp, da.tp_alvo)\n        esperado = pd.date_range(\'1940-01-01\', CORTE_FINAL, freq=\'MS\')\n        exigir(set(bruto.dims) == set(alvo.dims) == {\'time\', \'lat\', \'lon\'}, \'Unexpected precipitation dimensions.\')\n        bruto, alvo = (bruto.transpose(\'time\', \'lat\', \'lon\'), alvo.transpose(\'time\', \'lat\', \'lon\'))\n        exigir(bruto.shape == alvo.shape == (996, 301, 261), \'Unexpected grid or calendar.\')\n        exigir(pd.DatetimeIndex(bruto.time.values).equals(esperado), \'Official calendar mismatch.\')\n        exigir(bruto.attrs.get(\'units\') == alvo.attrs.get(\'units\') == \'mm/day\', \'Rainfall must use mm/day; no conversion is applied.\')\n        for eixo in [\'time\', \'lat\', \'lon\']:\n            exigir(np.array_equal(bruto[eixo].values, alvo[eixo].values), f\'Misaligned target at {eixo}.\')\n        for eixo in [\'lat\', \'lon\']:\n            exigir(np.all(np.diff(bruto[eixo].values) == 0.25), f\'Invalid grid: {eixo}.\')\n        chuva = bruto.sel(time=slice(None, FIM_DESENVOLVIMENTO)).load()\n        exigir(np.isfinite(chuva.values).all() and (chuva.values >= 0).all(), \'Invalid historical precipitation.\')\n        for i in range(0, chuva.sizes[\'time\'] - 1, 24):\n            j = min(i + 24, chuva.sizes[\'time\'] - 1)\n            exigir(np.array_equal(alvo.isel(time=slice(i, j)).values, chuva.values[i + 1:j + 1]), \'tp_alvo[M] does not match tp[M+1].\')\n    return chuva\n\ndef carregar_atmosfera(referencia, inicio, fim):\n    resultado, metadados = ({}, [])\n    esperado_completo = pd.date_range(\'1940-01-01\', CORTE_FINAL, freq=\'MS\')\n    esperado_recorte = pd.date_range(inicio, fim, freq=\'MS\')\n    for nome in VARIAVEIS:\n        with xr.open_dataset(PASTA / f\'treino_{nome}.nc\') as ds:\n            bruto = ds[nome].transpose(\'time\', \'lat\', \'lon\')\n            conferir_campo(bruto, referencia, esperado_completo)\n            campo = bruto.sel(time=slice(inicio, fim)).astype(\'float32\').load()\n            conferir_campo(campo, referencia, esperado_recorte)\n            exigir(np.isfinite(campo.values).all(), f\'{nome}: nonfinite atmospheric data.\')\n            resultado[nome] = campo\n            metadados.append(dict(variavel=nome, atributos=dict(bruto.attrs), inicio=str(inicio), fim=str(fim), minimo=float(campo.min()), maximo=float(campo.max())))\n        print(\'Loaded:\', nome, campo.shape, flush=True)\n    return (resultado, metadados)\n\ndef carregar_indices_incorporados():\n    dados = gzip.decompress(base64.b64decode(INDICES_CSV_GZIP_BASE64))\n    exigir(hashlib.sha256(dados).hexdigest() == INDICES_CSV_SHA256, \'Embedded NOAA snapshot hash mismatch.\')\n    tabela = pd.read_csv(io.BytesIO(dados), parse_dates=[\'time_origem\'])\n    exigir(tabela.columns.tolist() == [\'time_origem\'] + INDICES_NOMES, \'Ocean index columns differ from the reference.\')\n    tabela = tabela.set_index(\'time_origem\')\n    conferir_indices(tabela)\n    esperado = pd.date_range(FONTES_INDICES[\'inicio\'], FONTES_INDICES[\'fim\'], freq=\'MS\')\n    exigir(tabela.index.equals(esperado), \'The NOAA snapshot has missing or altered dates.\')\n    return (tabela, dados)\n\ndef conferir_indices(tabela):\n    exigir(tabela.columns.tolist() == INDICES_NOMES, \'The order of the four ocean indices changed.\')\n    datas = pd.DatetimeIndex(tabela.index)\n    exigir(len(datas) > 0 and datas.is_unique and datas.is_monotonic_increasing, \'NOAA dates are empty, duplicated or unordered.\')\n    exigir(datas.equals(pd.date_range(datas.min(), datas.max(), freq=\'MS\')), \'NOAA dates must be consecutive first-of-month timestamps.\')\n    exigir(np.isfinite(tabela.to_numpy(dtype=np.float64)).all(), \'Missing or infinite NOAA index. Do not interpolate or fill from future months.\')\n\ndef auditar_dataset(ds, esperado=None):\n    exigir(ds.attrs.get(\'schema\') == SCHEMA, \'Unknown SEAS5 artifact version.\')\n    exigir(ds.attrs.get(\'dataset\') == DATASET and str(ds.attrs.get(\'system\')) == SYSTEM, \'Invalid SEAS5 provenance.\')\n    exigir(ds.attrs.get(\'product_type\') == \'monthly_mean\' and int(ds.attrs.get(\'leadtime_month\')) == 2, \'Incorrect product or lead time.\')\n    exigir(ds.seas5_tp_media.attrs.get(\'units\') == \'mm/day\', \'SEAS5 must use mm/day after conversion.\')\n    exigir(ds.seas5_tp_media.dims == (\'time\', \'lat\', \'lon\'), \'Unexpected SEAS5 dimensions.\')\n    datas = pd.DatetimeIndex(ds.time.values)\n    exigir(datas.is_unique and datas.is_monotonic_increasing, \'Duplicate or unordered targets.\')\n    exigir(datas.equals(pd.date_range(datas.min(), datas.max(), freq=\'MS\')), \'Missing SEAS5 target months.\')\n    if esperado is not None:\n        exigir(datas.equals(pd.DatetimeIndex(esperado)), \'SEAS5 dates do not match the request.\')\n    exigir(pd.DatetimeIndex(ds.time_origem.values).equals((datas.to_period(\'M\') - 1).to_timestamp()), \'SEAS5 must be initialized exactly in T-1.\')\n    exigir(np.isin(ds.n_membros.values, [25, 51]).all(), \'Incomplete ensemble member set.\')\n    for eixo in [\'lat\', \'lon\']:\n        v = ds[eixo].values\n        exigir(len(v) > 1 and np.all(np.diff(v) == 1), f\'Incorrect native one-degree grid: {eixo}.\')\n    exigir(ds.lat.min() <= -60 and ds.lat.max() >= 15 and (ds.lon.min() <= -90) and (ds.lon.max() >= -25), \'The input domain does not cover the official grid. Do not extrapolate.\')\n    exigir(np.isfinite(ds.seas5_tp_media.values).all() and (ds.seas5_tp_media.values >= 0).all(), \'Invalid SEAS5 values.\')\n\ndef localizar_seas5():\n    if PASTA_SEAS5_MANUAL:\n        candidatos = [Path(PASTA_SEAS5_MANUAL) / \'seas5_51_manifesto.json\']\n    else:\n        candidatos = []\n        for raiz in [Path(\'/kaggle/input\'), Path(\'/kaggle/working/worcap_SEAS5_dados\'), Path.cwd() / \'outputs/worcap_SEAS5_dados\']:\n            if raiz.is_dir():\n                candidatos.extend(raiz.rglob(\'seas5_51_manifesto.json\'))\n    candidatos = sorted(set((p.resolve() for p in candidatos if p.is_file())))\n    exigir(len(candidatos) == 1, \'Attach the prepared SEAS5 dataset or set PASTA_SEAS5_MANUAL. Exactly one SEAS5 manifest is required.\')\n    caminho = candidatos[0]\n    m = json.loads(caminho.read_text(encoding=\'utf-8\'))\n    exigir(m.get(\'schema\') == SCHEMA and m.get(\'system\') == SYSTEM and (m.get(\'dataset\') == DATASET), \'Incompatible SEAS5 manifest.\')\n    exigir(m.get(\'leadtime_month\') == 2 and m.get(\'paramId\') == 172228 and (m.get(\'product_type\') == \'monthly_mean\') and (m.get(\'variable\') == \'total_precipitation\'), \'SEAS5 product does not match the experiment.\')\n    return (caminho.parent, m)\n\ndef carregar_seas5(uso, referencia):\n    item = MANIFESTO_SEAS5[\'arquivos\'][uso]\n    caminho = PASTA_SEAS5 / item[\'arquivo\']\n    exigir(caminho.is_file() and sha256(caminho) == item[\'sha256\'], f\'Missing or altered SEAS5: {uso}.\')\n    with xr.open_dataset(caminho) as ds:\n        bruto = ds.load()\n    auditar_dataset(bruto, pd.date_range(item[\'inicio\'], item[\'fim\'], freq=\'MS\'))\n    exigir(bruto.attrs.get(\'uso\') == uso, \'SEAS5 file partition mismatch.\')\n    grade = bruto.seas5_tp_media.interp(lat=referencia.lat, lon=referencia.lon, method=\'linear\').astype(np.float32)\n    exigir(np.isfinite(grade.values).all() and (grade.values >= 0).all(), \'Invalid SEAS5 interpolation.\')\n    for coord in [\'lat\', \'lon\']:\n        exigir(np.array_equal(grade[coord].values, referencia[coord].values), \'Misaligned SEAS5 grid.\')\n    grade.attrs.update(units=\'mm/day\', system=SYSTEM, leadtime_month=2)\n    return grade\n\ndef valores_seas5(seas, destino, pontos=None):\n    destino = pd.Timestamp(destino)\n    exigir(pd.DatetimeIndex(seas.time.values).is_unique, \'Duplicate SEAS5 targets.\')\n    i = pd.DatetimeIndex(seas.time.values).get_indexer([destino])[0]\n    exigir(i >= 0, f\'Missing SEAS5 prediction for {destino.date()}.\')\n    origem = pd.Timestamp(seas.time_origem.values[i])\n    exigir(origem == (destino.to_period(\'M\') - 1).to_timestamp() and origem < destino, \'SEAS5 initialization is incorrect or beyond the cutoff.\')\n    v = seas.values[i].reshape(-1)\n    if pontos is not None:\n        v = v[pontos]\n    exigir(np.isfinite(v).all(), \'Nonfinite SEAS5 feature.\')\n    return v\n\ndef amostrar_m5_documentado(chuva, atmosfera, seas, corte):\n    destinos = calendario_pareado(corte)\n    origens = (destinos.to_period(\'M\') - 1).to_timestamp()\n    ref_origens, ref_alvos = calendario_pares(atmosfera[VARIAVEIS[0]].time.values, corte, 30)\n    clima, somas, contagens = estatisticas_climatologia(chuva, corte, 30)\n    exigir(np.all(contagens == 30), \'Rainfall climatology must contain exactly 30 years.\')\n    ca = ajustar_clima_atmosfera(atmosfera, ref_origens)\n    ca[\'_clima_indices\'] = ajustar_clima_indices(INDICES_OC, ref_origens, corte)\n    exigir(set(destinos).issubset(set(ref_alvos)), \'Example outside the reference window.\')\n    ngrade = chuva.sizes[\'lat\'] * chuva.sizes[\'lon\']\n    n = min(PONTOS_POR_MES, ngrade)\n    X = np.empty((len(destinos) * n, 28), dtype=np.float32)\n    y = np.empty(len(X), dtype=np.float32)\n    ids = np.empty((len(destinos), n), dtype=np.int32)\n    for i, t in enumerate(destinos):\n        pontos = np.random.default_rng(np.random.SeedSequence([SEMENTE, t.year, t.month])).choice(ngrade, size=n, replace=False)\n        ids[i] = pontos\n        basicas = matriz_mes(atmosfera, ca, clima, t, pontos)\n        obs = chuva.sel(time=t).values.reshape(-1)[pontos]\n        basicas[:, 22] = ((somas[t.month - 1].reshape(-1)[pontos] - obs.astype(np.float64)) / 29).astype(np.float32)\n        sl = slice(i * n, (i + 1) * n)\n        X[sl, :27] = basicas\n        X[sl, 27] = valores_seas5(seas, t, pontos)\n        y[sl] = obs - basicas[:, 22]\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Nonfinite training data.\')\n    exigir((origens < destinos).all() and destinos.max() <= pd.Timestamp(corte), \'Temporal cutoff violated.\')\n    return (X, y, ids, destinos, clima, ca, somas, contagens)\n\ndef valores_anomalia_seas5(seas, clima_seas, destino, pontos=None):\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(seas[eixo].values, clima_seas[eixo].values), \'Misaligned SEAS5 climatology.\')\n    bruto = valores_seas5(seas, destino, pontos)\n    referencia = clima_seas.sel(month=pd.Timestamp(destino).month).values.reshape(-1)\n    if pontos is not None:\n        referencia = referencia[pontos]\n    anomalia = (bruto - referencia).astype(np.float32)\n    exigir(np.isfinite(anomalia).all(), \'Nonfinite SEAS5 anomaly.\')\n    return anomalia\n\ndef amostrar_arvores_seas5(chuva, atmosfera, seas, corte):\n    x5, y, ids, destinos, clima, ca, somas, contagens = amostrar_m5_documentado(chuva, atmosfera, seas, corte)\n    clima_seas = ajustar_clima_seas5(seas, corte)\n    X = np.empty((len(x5), 29), dtype=np.float32)\n    X[:, :28] = x5\n    n = ids.shape[1]\n    for i, t in enumerate(destinos):\n        X[i * n:(i + 1) * n, 28] = valores_anomalia_seas5(seas, clima_seas, t, ids[i])\n    exigir(np.array_equal(X[:, :28], x5), \'The 28 base SEAS5 features changed.\')\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Nonfinite training data.\')\n    del x5\n    return (X, y, ids, destinos, clima, ca, somas, contagens, clima_seas)\n\ndef validar_manifesto_cfsv2(m):\n    exigir(m.get(\'schema\') == SCHEMA_CFSV2 and m.get(\'modelo\') == \'NCEP-CFSv2\' and (m.get(\'fonte\') == FONTE_CFSV2), \'Incompatible CFSv2 manifest.\')\n    exigir(m.get(\'lead_iri\') == 1.5 and m.get(\'unidade\') == \'mm/day\' and (m.get(\'chuva_observada_utilizada\') is False), \'Incorrect CFSv2 product or temporal alignment.\')\n    exigir(m.get(\'meses\') == 515 and m.get(\'inicio_alvos\') == \'1982-02-01\' and (m.get(\'fim_alvos\') == \'2024-12-01\'), \'CFSv2 coverage mismatch.\')\n    exigir(m.get(\'politica_membros\') == POLITICA_CFSV2 and m.get(\'excecoes_permitidas\') == {\'2019-08-01\': [17]}, \'CFSv2 ensemble policy mismatch.\')\n    aplicadas = m.get(\'excecoes_aplicadas\', [])\n    exigir(isinstance(aplicadas, list) and len(aplicadas) <= 1, \'Unspecified CFSv2 exceptions.\')\n    if aplicadas:\n        r = aplicadas[0]\n        exigir(r.get(\'time_origem\') == \'2019-08-01\' and r.get(\'time_alvo\') == \'2019-09-01\' and (r.get(\'membros\') == 23) and (r.get(\'membros_esperados\') == 24) and (r.get(\'membros_indisponiveis_ids\') == [17]), \'Incorrect CFSv2 exception.\')\n        a = r.get(\'auditoria_excecao\', {})\n        exigir(a.get(\'membros_indisponiveis_ids\') == [17] and np.isfinite(a.get(\'maximo_delta_media_mm_day\', np.nan)) and (0 <= a[\'maximo_delta_media_mm_day\'] <= 1e-05), \'Exception missing the individual-member mean check.\')\n    for fase, (inicio, fim, n) in FASES_CFSV2.items():\n        item = m.get(\'arquivos\', {}).get(fase, {})\n        exigir(item.get(\'arquivo\') == f\'cfsv2_{fase}.nc\' and item.get(\'inicio_alvos\') == inicio and (item.get(\'fim_alvos\') == fim) and (item.get(\'meses\') == n), \'Incorrect CFSv2 partition.\')\n\ndef localizar_cfsv2():\n    if PASTA_CFSV2_MANUAL:\n        candidatos = [Path(PASTA_CFSV2_MANUAL) / \'cfsv2_manifesto.json\']\n    else:\n        candidatos = []\n        for raiz in [Path(\'/kaggle/input\'), Path(\'/kaggle/working/worcap_CFSv2_dados\'), Path.cwd() / \'outputs/worcap_CFSv2_dados\']:\n            if raiz.is_dir():\n                candidatos.extend(raiz.rglob(\'cfsv2_manifesto.json\'))\n    candidatos = sorted(set((p.resolve() for p in candidatos if p.is_file())))\n    exigir(len(candidatos) == 1, \'Attach the prepared CFSv2 output or set PASTA_CFSV2_MANUAL. Exactly one CFSv2 manifest is required.\')\n    caminho = candidatos[0]\n    m = json.loads(caminho.read_text(encoding=\'utf-8\'))\n    validar_manifesto_cfsv2(m)\n    return (caminho.parent, m)\n\ndef auditar_cfsv2(ds, uso, m):\n    inicio, fim, n = FASES_CFSV2[uso]\n    datas = pd.date_range(inicio, fim, freq=\'MS\')\n    origens = (datas.to_period(\'M\') - 1).to_timestamp()\n    exigir(ds.attrs.get(\'schema\') == SCHEMA_CFSV2 and ds.attrs.get(\'modelo\') == \'NCEP-CFSv2\' and (ds.attrs.get(\'fonte\') == FONTE_CFSV2) and (ds.attrs.get(\'lead_iri\') == 1.5), \'CFSv2 NetCDF provenance mismatch.\')\n    exigir(ds.attrs.get(\'politica_membros\') == POLITICA_CFSV2, \'CFSv2 NetCDF is missing the corrected ensemble policy.\')\n    exigir(ds.cfsv2_tp_media.dims == (\'time\', \'lat\', \'lon\') and ds.cfsv2_tp_media.attrs.get(\'units\') == \'mm/day\', \'Invalid CFSv2 dimensions or units.\')\n    exigir(pd.DatetimeIndex(ds.time.values).equals(datas), \'Missing, extra or unordered CFSv2 dates.\')\n    exigir(pd.DatetimeIndex(ds.time_origem.values).equals(origens), \'CFSv2 nominal issue month is not T-1.\')\n    exigir(np.array_equal(ds.lat.values, np.arange(-61, 17)) and np.array_equal(ds.lon.values, np.arange(-91, -23)), \'CFSv2 native grid mismatch.\')\n    for var in [\'n_membros\', \'inicializacao_mais_antiga\', \'inicializacao_mais_recente\', \'fase_fonte\', \'time_origem\']:\n        exigir(ds[var].dims == (\'time\',), f\'Invalid CFSv2 coordinate: {var}.\')\n    dmin = pd.DatetimeIndex(ds.inicializacao_mais_antiga.values)\n    dmax = pd.DatetimeIndex(ds.inicializacao_mais_recente.values)\n    exigir(not dmin.hasnans and (not dmax.hasnans) and (dmin <= dmax).all() and (dmax < datas).all() and (dmin >= origens - pd.Timedelta(days=40)).all() and (dmax <= origens + pd.Timedelta(days=7)).all(), \'CFSv2 initializations are outside the allowed window.\')\n    esperados = np.where(origens.month == 11, 28, 24)\n    if m[\'excecoes_aplicadas\'] and pd.Timestamp(\'2019-09-01\') in datas:\n        esperados[datas == pd.Timestamp(\'2019-09-01\')] = 23\n    exigir(np.array_equal(ds.n_membros.values, esperados), \'CFSv2 member counts differ from the manifest.\')\n    exigir(np.array_equal(ds.fase_fonte.values, (origens >= pd.Timestamp(\'2011-04-01\')).astype(np.int8)), \'Incorrect CFSv2 hindcast/real-time transition.\')\n    v = ds.cfsv2_tp_media.values\n    exigir(np.isfinite(v).all() and (v >= 0).all(), \'Invalid CFSv2 predicted rainfall.\')\n\ndef carregar_cfsv2(uso, referencia):\n    item = MANIFESTO_CFSV2[\'arquivos\'][uso]\n    path = PASTA_CFSV2 / item[\'arquivo\']\n    exigir(path.is_file() and sha256(path) == item[\'sha256\'], f\'Missing or altered CFSv2: {uso}.\')\n    with xr.open_dataset(path) as d:\n        bruto = d.load()\n    auditar_cfsv2(bruto, uso, MANIFESTO_CFSV2)\n    grade = bruto.cfsv2_tp_media.interp(lat=referencia.lat, lon=referencia.lon, method=\'linear\').astype(np.float32)\n    exigir(np.isfinite(grade.values).all() and (grade.values >= 0).all(), \'Invalid CFSv2 interpolation; do not extrapolate.\')\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(grade[eixo].values, referencia[eixo].values), \'Misaligned CFSv2 grid.\')\n    for coord in [\'time_origem\', \'n_membros\', \'inicializacao_mais_antiga\', \'inicializacao_mais_recente\', \'fase_fonte\']:\n        grade = grade.assign_coords({coord: (\'time\', bruto[coord].values)})\n    grade.attrs.update(units=\'mm/day\', lead_iri=1.5)\n    salvar_json(f\'auditoria_CFSv2_{uso}.json\', dict(arquivo=str(path), sha256=item[\'sha256\'], meses=bruto.sizes[\'time\'], minimo=float(grade.min()), maximo=float(grade.max()), membros_usados=np.unique(bruto.n_membros.values).tolist(), interpolacao=\'bilinear without extrapolation\'))\n    return grade\n\ndef ajustar_clima_previsao(previsoes, corte, nome):\n    treino = calendario_pareado(corte)\n    datas = pd.DatetimeIndex(previsoes.time.values)\n    exigir(datas.is_unique and datas.is_monotonic_increasing and treino.isin(datas).all(), f\'Incomplete or unordered dates: {nome}.\')\n    sel = previsoes.sel(time=treino)\n    exigir(pd.DatetimeIndex(sel.time_origem.values).equals((treino.to_period(\'M\') - 1).to_timestamp()), f\'Incorrect origin in climatology {nome}.\')\n    v = sel.values\n    exigir(np.isfinite(v).all() and (v >= 0).all(), f\'Invalid values in climatology {nome}.\')\n    n = np.array([(treino.month == m).sum() for m in range(1, 13)], dtype=np.int16)\n    exigir((n > 0).all() and n.max() <= 30, \'Forecast climatology exceeds 30 years or has a missing calendar month.\')\n    medias = np.stack([v[treino.month == m].mean(axis=0, dtype=np.float64) for m in range(1, 13)]).astype(np.float32)\n    return xr.DataArray(medias, dims=(\'month\', \'lat\', \'lon\'), coords={\'month\': np.arange(1, 13), \'lat\': previsoes.lat, \'lon\': previsoes.lon, \'n_meses\': (\'month\', n)}, name=f\'climatologia_{nome}\', attrs=dict(units=\'mm/day\', inicio=str(treino[0].date()), corte=str(pd.Timestamp(corte).date()), meses_treino=len(treino), anos_min=int(n.min()), anos_max=int(n.max()), referencia=\'training-month predictors only; one ensemble mean per year and month\'))\n\ndef ajustar_clima_seas5(seas, corte):\n    return ajustar_clima_previsao(seas, corte, \'SEAS5\')\n\ndef valores_cfsv2(cfs, destino, pontos=None):\n    destino = pd.Timestamp(destino)\n    exigir(pd.Timestamp(cfs.inicializacao_mais_recente.sel(time=destino).values) < destino, \'A real CFSv2 initialization extends into the target month.\')\n    return valores_seas5(cfs, destino, pontos)\n\ndef valores_anomalia_cfsv2(cfs, clima_cfs, destino, pontos=None):\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(cfs[eixo], clima_cfs[eixo]), \'Misaligned CFSv2 climatology.\')\n    bruto = valores_cfsv2(cfs, destino, pontos)\n    ref = clima_cfs.sel(month=pd.Timestamp(destino).month).values.ravel()\n    if pontos is not None:\n        ref = ref[pontos]\n    anom = (bruto - ref).astype(np.float32)\n    exigir(np.isfinite(anom).all(), \'Invalid CFSv2 anomaly.\')\n    return anom\n\n# Both tree components receive exactly the same sampled pixels and residual targets.\ndef amostrar_arvores_multissistema(chuva, atmosfera, seas, cfs, corte):\n    x6, y, ids, destinos, clima, ca, somas, contagens, cs = amostrar_arvores_seas5(chuva, atmosfera, seas, corte)\n    cc = ajustar_clima_previsao(cfs, corte, \'CFSv2\')\n    X = np.empty((len(x6), 31), dtype=np.float32)\n    X[:, :29] = x6\n    n = ids.shape[1]\n    for i, t in enumerate(destinos):\n        sl = slice(i * n, (i + 1) * n)\n        X[sl, 29] = valores_cfsv2(cfs, t, ids[i])\n        X[sl, 30] = valores_anomalia_cfsv2(cfs, cc, t, ids[i])\n    exigir(np.array_equal(X[:, :29], x6), \'The 29 control features changed.\')\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Invalid training matrix.\')\n    return (X, y, ids, destinos, clima, ca, somas, contagens, cs, cc)\n\ndef prever_par(modelo, nome, atmosfera, seas, cfs, ca, clima, cs, cc, destinos):\n    destinos = pd.DatetimeIndex(destinos)\n    exigir(nome in NOMES_MODELOS, \'Unknown model.\')\n    exigir((destinos > pd.Timestamp(cs.attrs[\'corte\'])).all() and cs.attrs[\'corte\'] == cc.attrs[\'corte\'], \'Inference must follow the training cutoff.\')\n    out = np.empty((len(destinos), clima.sizes[\'lat\'], clima.sizes[\'lon\']), dtype=np.float32)\n    for i, t in enumerate(destinos):\n        x = np.column_stack([matriz_mes(atmosfera, ca, clima, t), valores_seas5(seas, t), valores_anomalia_seas5(seas, cs, t)]).astype(np.float32)\n        if nome == \'arvores_multissistema\':\n            x = np.column_stack([x, valores_cfsv2(cfs, t), valores_anomalia_cfsv2(cfs, cc, t)]).astype(np.float32)\n        exigir(x.shape[1] == (29 if nome == \'arvores_seas5\' else 31) and np.isfinite(x).all(), \'Invalid inference.\')\n        out[i] = modelo.predict(x).reshape(out.shape[1:])\n    exigir(np.isfinite(out).all(), \'Invalid predictions.\')\n    return xr.DataArray(out, dims=(\'time\', \'lat\', \'lon\'), coords={\'time\': destinos, \'lat\': clima.lat, \'lon\': clima.lon}, attrs={\'units\': \'mm/day\'})\n\n# Fit a local two-predictor ridge with training-only monthly means and scales.\ndef mos_fit(chuva, seas, cfs, corte):\n    dates = calendario_pareado(corte)\n    r = chuva.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    s = seas.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    c = cfs.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    for pre in [seas, cfs]:\n        assert pd.DatetimeIndex(pre.sel(time=dates).time_origem.values).equals((dates.to_period(\'M\') - 1).to_timestamp())\n    assert (pd.DatetimeIndex(cfs.sel(time=dates).inicializacao_mais_recente.values) < dates).all()\n    baseline = estatisticas_climatologia(chuva, corte, 30)[0]\n    shape = baseline.shape[1:]\n    ng = r.shape[1]\n    coef = np.zeros((2, ng), np.float64)\n    mu = np.zeros((2, 12, ng), np.float64)\n    for lo in range(0, ng, 4096):\n        hi = min(lo + 4096, ng)\n        y = r[:, lo:hi].astype(np.float64)\n        x = s[:, lo:hi].astype(np.float64)\n        z = c[:, lo:hi].astype(np.float64)\n        assert np.isfinite(y).all() and np.isfinite(x).all() and np.isfinite(z).all()\n        for m in range(1, 13):\n            ix = dates.month == m\n            mu[0, m - 1, lo:hi] = x[ix].mean(0)\n            mu[1, m - 1, lo:hi] = z[ix].mean(0)\n            x[ix] -= mu[0, m - 1, lo:hi]\n            z[ix] -= mu[1, m - 1, lo:hi]\n            y[ix] -= y[ix].mean(0)\n        sx = np.sqrt(np.mean(x * x, axis=0))\n        sz = np.sqrt(np.mean(z * z, axis=0))\n        sx = np.where(sx > 1e-12, sx, 1.0)\n        sz = np.where(sz > 1e-12, sz, 1.0)\n        x /= sx\n        z /= sz\n        a = np.mean(x * x, axis=0) + MOS_ALPHA\n        b = np.mean(x * z, axis=0)\n        d = np.mean(z * z, axis=0) + MOS_ALPHA\n        u = np.mean(x * y, axis=0)\n        v = np.mean(z * y, axis=0)\n        det = a * d - b * b\n        assert (det > 0).all()\n        coef[0, lo:hi] = (d * u - b * v) / det / sx\n        coef[1, lo:hi] = (a * v - b * u) / det / sz\n    assert np.isfinite(coef).all()\n    return dict(coef=coef.reshape(2, *shape), mu=mu.reshape(2, 12, *shape), baseline=baseline.values, lat=baseline.lat.values, lon=baseline.lon.values, corte=str(pd.Timestamp(corte).date()), inicio=str(dates[0].date()), meses=len(dates))\n\ndef mos_predict(fit, seas, cfs, dates):\n    dates = pd.DatetimeIndex(dates)\n    assert (dates > pd.Timestamp(fit[\'corte\'])).all()\n    for pre in [seas, cfs]:\n        assert np.array_equal(pre.lat, fit[\'lat\']) and np.array_equal(pre.lon, fit[\'lon\'])\n    out = []\n    for t in dates:\n        s = valores_seas5(seas, t).reshape(fit[\'baseline\'].shape[1:]).astype(np.float64)\n        c = valores_cfsv2(cfs, t).reshape(s.shape).astype(np.float64)\n        anom = fit[\'coef\'][0] * (s - fit[\'mu\'][0, t.month - 1]) + fit[\'coef\'][1] * (c - fit[\'mu\'][1, t.month - 1])\n        out.append(np.maximum(0, fit[\'baseline\'][t.month - 1].astype(np.float64) + anom))\n    return np.stack(out)\n\ndef matriz_mes(campos, clima_atmos, clima_tp, destino, pontos=None):\n    """Build features at the specified source month. The full training rainfall climatology is used here; the sampler replaces its training column with the leave-one-out value."""\n    destino = pd.Timestamp(destino)\n    origem = origem_mensal([destino], LAG_ATMOSFERA)[0]\n    nlat = clima_tp.sizes[\'lat\']\n    nlon = clima_tp.sizes[\'lon\']\n    pontos = np.arange(nlat * nlon) if pontos is None else np.asarray(pontos)\n    exigir(pontos.ndim == 1 and np.issubdtype(pontos.dtype, np.integer), \'Invalid indices.\')\n    exigir(((pontos >= 0) & (pontos < nlat * nlon)).all(), \'Point outside the grid.\')\n    X = np.empty((len(pontos), len(FEATURES)), dtype=np.float32)\n    for j, nome in enumerate(VARIAVEIS):\n        campo = campos[nome]\n        i = pd.DatetimeIndex(campo.time.values).get_indexer([origem])[0]\n        exigir(i >= 0, f\'Origin {origem.date()} missing in {nome}.\')\n        valores = campo.values[i].reshape(-1)[pontos]\n        clima_origem = clima_atmos[nome][origem.month - 1].reshape(-1)[pontos]\n        X[:, j] = valores\n        X[:, j + len(VARIAVEIS)] = valores - clima_origem\n    X[:, 18] = clima_tp.lat.values[pontos // nlon]\n    X[:, 19] = clima_tp.lon.values[pontos % nlon]\n    X[:, 20] = np.sin(2 * np.pi * destino.month / 12)\n    X[:, 21] = np.cos(2 * np.pi * destino.month / 12)\n    X[:, 22] = clima_tp.values[destino.month - 1].reshape(-1)[pontos]\n    exigir(origem < destino, \'Temporal violation: input origin is not earlier than its target.\')\n    valores_oceano = valores_indices_mes(INDICES_OC, clima_atmos[\'_clima_indices\'], destino)\n    X[:, 23:] = valores_oceano[None, :]\n    exigir(np.isfinite(X).all(), \'Features contain missing or infinite values.\')\n    return X\nSST_COMPONENTES = 8\nSST_NOME = \'ersstv5_4graus_198201_202411.nc\'\nSST_HASH = \'908034f6418833302ea432f25850ad2982d1114b747b7b298e26a350eaf2891d\'\n\ndef sst_auditar_arquivo(path):\n    with xr.open_dataset(path) as ds:\n        exigir(\'sst\' in ds and ds.sst.dims == (\'time\', \'lat\', \'lon\'), \'Incorrect SST variable or dimensions.\')\n        exigir(ds.sst.attrs.get(\'units\') == \'degC\', \'SST must use degrees Celsius.\')\n        exigir(\'ERSSTv5\' in str(ds.attrs.get(\'title\', \'\')) and str(ds.attrs.get(\'product_version\', \'\')) == \'Version 5\', \'Product differs from ERSSTv5.\')\n        t = pd.DatetimeIndex(ds.time.values)\n        exigir(t.equals(pd.date_range(\'1982-01-01\', \'2024-11-01\', freq=\'MS\')), \'Incomplete or different SST calendar.\')\n        exigir(np.array_equal(ds.lat.values, np.arange(60, -61, -4)) and np.array_equal(ds.lon.values, np.arange(0, 360, 4)), \'SST must use the 60S-60N subset on a four-degree grid.\')\n        sst = ds.sst.astype(\'float32\').load()\n        atributos = {k: str(v) for k, v in ds.attrs.items()}\n    v = sst.values\n    validos = v[np.isfinite(v)]\n    exigir(validos.size > 0 and (not np.isinf(v).any()) and (validos.min() >= -1.801) and (validos.max() <= 45), "SST values outside the product\'s physical range.")\n    return (sst, dict(arquivo=Path(path).name, sha256=sha256(path), bytes=Path(path).stat().st_size, inicio_origens=str(t[0].date()), fim_origens=str(t[-1].date()), meses=len(t), grade=[31, 90], minimo=float(validos.min()), maximo=float(validos.max()), atributos=atributos, referencia=\'https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html\', origem=\'NOAA PSL / ERSSTv5; retrospective snapshot, without verified historical operational vintages\'))\n\n# Fit the ocean mask, monthly anomalies and PCA using training SST only.\ndef sst_ajustar_pca(sst, alvos_treino):\n    alvos_treino = pd.DatetimeIndex(alvos_treino)\n    exigir(alvos_treino.is_unique and alvos_treino.equals(pd.date_range(alvos_treino.min(), alvos_treino.max(), freq=\'MS\')), \'Invalid PCA fitting calendar.\')\n    origens = origem_mensal(alvos_treino, LAG_SST)\n    treino = sst.sel(time=origens).transpose(\'time\', \'lat\', \'lon\')\n    bruto = treino.values.reshape(len(origens), -1).astype(np.float64)\n    mascara = np.isfinite(bruto).all(axis=0)\n    exigir(mascara.any(), \'No complete training ocean grid cells.\')\n    mensal = np.full((12, bruto.shape[1]), np.nan, dtype=np.float64)\n    contagens = np.array([(origens.month == m).sum() for m in range(1, 13)])\n    exigir((contagens >= 2).all(), \'Insufficient monthly SST climatology.\')\n    for m in range(1, 13):\n        mensal[m - 1, mascara] = bruto[origens.month == m][:, mascara].mean(axis=0)\n    anom = bruto[:, mascara] - mensal[origens.month - 1][:, mascara]\n    ativos = np.flatnonzero(mascara)\n    mascara[ativos[np.std(anom, axis=0) <= 1e-08]] = False\n    latitudes = np.repeat(sst.lat.values, sst.sizes[\'lon\'])\n    pesos = np.sqrt(np.cos(np.deg2rad(latitudes[mascara]))).astype(np.float64)\n    exigir(int(mascara.sum()) > SST_COMPONENTES and len(origens) > SST_COMPONENTES, \'Insufficient dimensions for PCA.\')\n    matriz = (bruto[:, mascara] - mensal[origens.month - 1][:, mascara]) * pesos\n    exigir(np.isfinite(matriz).all(), \'Invalid PCA matrix.\')\n    with threadpool_limits(limits=1):\n        pca = PCA(n_components=SST_COMPONENTES, svd_solver=\'full\').fit(matriz)\n    return dict(mascara=mascara, clima_mensal=mensal[:, mascara], pesos_area=pesos, centro=pca.mean_, componentes=pca.components_, variancia_explicada=pca.explained_variance_ratio_, contagens_mensais=contagens, origens_treino=origens.values, lat=sst.lat.values.copy(), lon=sst.lon.values.copy())\n\ndef sst_transformar(sst, estado, alvos):\n    alvos = pd.DatetimeIndex(alvos)\n    origens = origem_mensal(alvos, LAG_SST)\n    exigir(np.array_equal(sst.lat, estado[\'lat\']) and np.array_equal(sst.lon, estado[\'lon\']), \'SST grid changed.\')\n    v = sst.sel(time=origens).values.reshape(len(origens), -1).astype(np.float64)[:, estado[\'mascara\']]\n    exigir(np.isfinite(v).all(), \'Missing SST in the training-selected ocean mask; do not impute from the future.\')\n    x = (v - estado[\'clima_mensal\'][origens.month - 1]) * estado[\'pesos_area\'] - estado[\'centro\']\n    with threadpool_limits(limits=1):\n        scores = x @ estado[\'componentes\'].T\n    exigir(scores.shape == (len(alvos), SST_COMPONENTES) and np.isfinite(scores).all(), \'Invalid PCA scores.\')\n    return scores.astype(np.float32)\n\ndef pesquisa_prever_sst(modelo, nome, atmosfera, seas, cfs, ca, clima, cs, cc, datas, scores):\n    datas = pd.DatetimeIndex(datas)\n    exigir((datas > pd.Timestamp(cs.attrs[\'corte\'])).all(), \'Inference overlaps training.\')\n    exigir(scores.shape == (len(datas), 8), \'Misaligned SST scores.\')\n    out = np.empty((len(datas), clima.sizes[\'lat\'], clima.sizes[\'lon\']), dtype=np.float32)\n    expected = (FEATURES_SEAS5 if nome == \'arvores_seas5\' else FEATURES_MULTISSISTEMA) + SST_FEATURES\n    exigir(modelo.feature_name() == expected, \'Incorrect SST feature order.\')\n    for i, t in enumerate(datas):\n        x = np.column_stack([matriz_mes(atmosfera, ca, clima, t), valores_seas5(seas, t), valores_anomalia_seas5(seas, cs, t)]).astype(np.float32)\n        if nome == \'arvores_multissistema\':\n            x = np.column_stack([x, valores_cfsv2(cfs, t), valores_anomalia_cfsv2(cfs, cc, t)]).astype(np.float32)\n        x = np.column_stack([x, np.broadcast_to(scores[i], (len(x), 8))]).astype(np.float32)\n        exigir(x.shape[1] == len(expected) and np.isfinite(x).all(), \'Invalid SST features.\')\n        out[i] = modelo.predict(x).reshape(out.shape[1:])\n    return xr.DataArray(out, dims=(\'time\', \'lat\', \'lon\'), coords=dict(time=datas, lat=clima.lat, lon=clima.lon), attrs=dict(units=\'mm/day\'))\n\ndef pesquisa_diagnosticos(obs, previsoes, bloco):\n    """Complete fields, without pixel subsampling or hiding errors."""\n    linhas = []\n    lat = obs.lat.values\n    regioes = {\'dominio_inteiro\': np.ones(lat.size, dtype=bool), \'lat_menor_que_menos35\': lat < -35, \'lat_menos35_a_menos15\': (lat >= -35) & (lat < -15), \'lat_maior_igual_menos15\': lat >= -15}\n    for modelo, pred in previsoes.items():\n        obs, pred = xr.align(obs, pred, join=\'exact\')\n        for i, t in enumerate(pd.DatetimeIndex(obs.time.values)):\n            erro = pred.values[i].astype(np.float64) - obs.values[i].astype(np.float64)\n            exigir(np.isfinite(erro).all(), \'Nonfinite diagnostic error.\')\n            for regiao, mask in regioes.items():\n                if not mask.any():\n                    continue\n                e = erro[mask]\n                w = np.broadcast_to(np.cos(np.deg2rad(lat[mask]))[:, None], e.shape)\n                linhas.append(dict(modelo=modelo, bloco=bloco, mes_alvo=str(t.date()), ano=t.year, mes=t.month, regiao=regiao, n=e.size, sse=float(np.square(e).sum()), soma_erro=float(e.sum()), soma_erro_absoluto=float(np.abs(e).sum()), sse_area=float((w * np.square(e)).sum()), peso_area=float(w.sum())))\n    return pd.DataFrame(linhas)\n\ndef pesquisa_resumir(mensais, grupos):\n    cols = [\'n\', \'sse\', \'soma_erro\', \'soma_erro_absoluto\', \'sse_area\', \'peso_area\']\n    r = mensais.groupby(grupos, as_index=False)[cols].sum()\n    r[\'rmse\'] = np.sqrt(r.sse / r.n)\n    r[\'mae\'] = r.soma_erro_absoluto / r.n\n    r[\'vies\'] = r.soma_erro / r.n\n    r[\'rmse_area\'] = np.sqrt(r.sse_area / r.peso_area)\n    return r\n\'Candidate lag policy; not evidence of historical publication dates or vintages.\'\nLAG_ATMOSFERA = 4\nLAG_INDICES = 3\nLAG_SST = 2\nLAG_CHUVA_TREINO = 4\nINICIO_COMUM = pd.Timestamp(\'1982-03-01\')\nFEATURES = [v + \'_lag4\' for v in VARIAVEIS] + [v + \'_lag4_anomalia\' for v in VARIAVEIS] + [\'latitude\', \'longitude\', \'mes_alvo_sin\', \'mes_alvo_cos\', \'clima_tp_alvo\'] + [n + \'_lag3_anomalia_fold\' for n in INDICES_NOMES]\nFEATURES_SEAS5 = FEATURES + [FEATURE_SEAS5, FEATURE_SEAS5_ANOM]\nFEATURES_MULTISSISTEMA = FEATURES_SEAS5 + [FEATURE_CFSV2, FEATURE_CFSV2_ANOM]\nSST_FEATURES = [f\'sst_lag2_pc_{i:02d}_treino_fold\' for i in range(1, 9)]\n\ndef origem_mensal(alvos, lag):\n    datas = pd.DatetimeIndex(alvos)\n    exigir(lag >= 1 and datas.equals(datas.to_period(\'M\').to_timestamp()), \'Invalid dates or lag.\')\n    return (datas.to_period(\'M\') - lag).to_timestamp()\n\ndef janela_referencia(corte, anos=30):\n    corte = pd.Timestamp(corte)\n    exigir(corte.day == 1, \'The cutoff must identify a month.\')\n    return pd.date_range(end=corte, periods=12 * anos, freq=\'MS\')\n\ndef calendario_pareado(corte):\n    datas = janela_referencia(corte)\n    datas = datas[datas >= INICIO_COMUM]\n    exigir(24 <= len(datas) <= 360, \'Invalid common window.\')\n    return datas\n\ndef calendario_pares(datas_disponiveis, corte, anos=30):\n    alvos = janela_referencia(corte, anos)\n    origens = origem_mensal(alvos, LAG_ATMOSFERA)\n    disponiveis = pd.DatetimeIndex(datas_disponiveis)\n    exigir(disponiveis.is_unique and origens.isin(disponiveis).all(), \'Missing atmospheric origins.\')\n    return (origens, alvos)\n\n# Monthly rainfall sums and counts support leave-one-out training references.\ndef estatisticas_climatologia(tp, corte, anos=30):\n    datas = janela_referencia(corte, anos)\n    treino = tp.sel(time=datas).transpose(\'time\', \'lat\', \'lon\')\n    exigir(np.isfinite(treino.values).all(), \'Incomplete training rainfall.\')\n    somas = np.stack([treino.values[datas.month == m].sum(axis=0, dtype=np.float64) for m in range(1, 13)])\n    contagens = np.array([(datas.month == m).sum() for m in range(1, 13)], dtype=np.int64)\n    exigir((contagens == anos).all(), \'Climatology requires every calendar month.\')\n    clima = xr.DataArray((somas / contagens[:, None, None]).astype(np.float32), dims=(\'month\', \'lat\', \'lon\'), coords=dict(month=np.arange(1, 13), lat=tp.lat, lon=tp.lon), attrs=dict(units=\'mm/day\', ajuste_inicio=str(datas[0].date()), ajuste_fim=str(datas[-1].date())))\n    return (clima, somas, contagens)\n\ndef ajustar_clima_indices(tabela, origens_atmosfera, corte):\n    alvos = janela_referencia(corte)\n    exigir(pd.DatetimeIndex(origens_atmosfera).equals(origem_mensal(alvos, LAG_ATMOSFERA)), \'Atmospheric reference outside the protocol.\')\n    origens = origem_mensal(alvos, LAG_INDICES)\n    valores = tabela.loc[origens, INDICES_NOMES].to_numpy(dtype=np.float64)\n    exigir(np.isfinite(valores).all(), \'Incomplete training indices.\')\n    return np.stack([valores[origens.month == m].mean(axis=0, dtype=np.float64) for m in range(1, 13)])\n\ndef valores_indices_mes(tabela, clima_indices, destino):\n    origem = origem_mensal([destino], LAG_INDICES)[0]\n    valores = tabela.loc[origem, INDICES_NOMES].to_numpy(dtype=np.float64)\n    exigir(np.isfinite(valores).all(), \'Unavailable index; do not fill.\')\n    return (valores - clima_indices[origem.month - 1]).astype(np.float32)\n\ndef exigir_disponibilidade_real(registros, emitido_em, fontes_necessarias):\n    """Validate supplied availability metadata for a future production gate. Receipt records must identify the actual bytes. This check alone cannot authenticate a historical publication timestamp."""\n    emissao = pd.Timestamp(emitido_em)\n    exigir(emissao.tzinfo is not None, \'Issue time must include a timezone.\')\n    exigir(len(registros) == len(fontes_necessarias), \'Incorrect source count.\')\n    exigir({r.get(\'fonte\') for r in registros} == set(fontes_necessarias), \'Missing or duplicate source.\')\n    for r in registros:\n        for chave in [\'disponivel_em\', \'recebido_em\', \'url\', \'sha256\', \'versao\']:\n            exigir(bool(r.get(chave)), \'Unknown availability: \' + r.get(\'fonte\', \'?\') + \'/\' + chave)\n        recebido, publicado = (pd.Timestamp(r[\'recebido_em\']), pd.Timestamp(r[\'disponivel_em\']))\n        exigir(recebido.tzinfo is not None and publicado.tzinfo is not None, \'Dates must include a timezone.\')\n        exigir(publicado <= recebido <= emissao, \'A source arrived after issue time or its chronology is invalid.\')\n        exigir(len(r[\'sha256\']) == 64 and set(r[\'sha256\']) <= set(\'0123456789abcdef\'), \'Invalid SHA-256.\')\n    return True\n\ndef calendario_emissoes(datas, corte):\n    datas = pd.DatetimeIndex(datas)\n    limite = origem_mensal([datas[0]], LAG_CHUVA_TREINO)[0]\n    exigir(pd.Timestamp(corte) <= limite, \'Training rainfall is too recent for the first issue date.\')\n    return pd.DataFrame(dict(time_alvo=datas, limite_emissao=datas - pd.Timedelta(nanoseconds=1), atmosfera=origem_mensal(datas, LAG_ATMOSFERA), indices=origem_mensal(datas, LAG_INDICES), sst=origem_mensal(datas, LAG_SST), inicializacao_sazonal=origem_mensal(datas, 1), ultimo_alvo_treino=pd.Timestamp(corte), disponibilidade_historica_comprovada=False))\n\ndef auditar_invariancia_defasada(chuva, atmosfera, seas, cfs, sst, corte, datas, clima, ca, cs, cc, pca):\n    """Mutate small copies of real data; values beyond each cutoff must not affect T."""\n    primeiro = pd.Timestamp(datas[0])\n    pontos = np.array([0, 1, chuva.sizes[\'lon\']], dtype=np.int32)\n    ch = chuva.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True)\n    at = {k: v.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True) for k, v in atmosfera.items()}\n    alvos_ref = janela_referencia(corte)\n    orig_at = origem_mensal(alvos_ref, LAG_ATMOSFERA)\n    c0, _, _ = estatisticas_climatologia(ch, corte)\n    a0 = ajustar_clima_atmosfera(at, orig_at)\n    a0[\'_clima_indices\'] = ajustar_clima_indices(INDICES_OC, orig_at, corte)\n    x0 = matriz_mes(at, a0, c0, primeiro)\n    ch.values[pd.DatetimeIndex(ch.time.values) > pd.Timestamp(corte)] = 123456\n    c1, _, _ = estatisticas_climatologia(ch, corte)\n    exigir(np.array_equal(c0.values, c1.values), \'Post-cutoff rainfall changed the climatology.\')\n    for v in at.values():\n        v.values[pd.DatetimeIndex(v.time.values) > origem_mensal([primeiro], LAG_ATMOSFERA)[0]] = 7654321\n    a1 = ajustar_clima_atmosfera(at, orig_at)\n    a1[\'_clima_indices\'] = a0[\'_clima_indices\']\n    exigir(all((np.array_equal(a0[k], a1[k]) for k in a0)), \'Future data changed atmospheric climatology.\')\n    exigir(np.array_equal(x0, matriz_mes(at, a1, c1, primeiro)), \'Recent or future atmospheric data changed T.\')\n    ni = INDICES_OC.copy(deep=True)\n    ni.loc[ni.index > origem_mensal([primeiro], LAG_INDICES)[0], :] = 99999\n    ci = ajustar_clima_indices(ni, orig_at, corte)\n    exigir(np.array_equal(ci, a0[\'_clima_indices\']), \'Future indices changed the fit.\')\n    exigir(np.array_equal(valores_indices_mes(ni, ci, primeiro), x0[0, 23:]), \'A future index changed the feature.\')\n    for pre, nome in [(seas, \'SEAS5\'), (cfs, \'CFSv2\')]:\n        pequeno = pre.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True)\n        antes = ajustar_clima_previsao(pequeno, corte, nome)\n        pequeno.values[pd.DatetimeIndex(pequeno.time.values) > pd.Timestamp(corte)] = 12345\n        exigir(np.array_equal(antes, ajustar_clima_previsao(pequeno, corte, nome)), \'Future data changed seasonal climatology.\')\n    copia = sst.copy(deep=True)\n    copia.values[pd.DatetimeIndex(copia.time.values) > origem_mensal([primeiro], LAG_SST)[0]] = 12345\n    novo = sst_ajustar_pca(copia, calendario_pareado(corte))\n    for k in [\'mascara\', \'clima_mensal\', \'centro\', \'componentes\', \'origens_treino\']:\n        exigir(np.array_equal(novo[k], pca[k]), \'Future SST changed PCA: \' + k)\n    exigir(np.array_equal(sst_transformar(copia, novo, [primeiro]), sst_transformar(sst, pca, [primeiro])), \'SST more recent than T-2 changed the prediction for T.\')\n    return dict(chuva_pos_corte_invariante=True, atmosfera_apos_Tmenos4_invariante=True, indices_apos_Tmenos3_invariantes=True, sst_apos_Tmenos2_invariante=True, climas_sazonais_somente_treino=True, publicacao_historica_comprovada=False)\n', 'research/lagged_sources/ocean_indices.csv': 'time_origem,nino34,nino12,tna,tsa\n1976-07-01,0.150000000,2.020000000,-0.640000000,-0.930000000\n1976-08-01,0.310000000,1.810000000,-0.270000000,-0.680000000\n1976-09-01,0.570000000,1.230000000,-0.240000000,-0.470000000\n1976-10-01,0.860000000,0.760000000,-0.250000000,-0.580000000\n1976-11-01,0.880000000,0.440000000,-0.270000000,-0.630000000\n1976-12-01,0.640000000,0.250000000,-0.600000000,-0.470000000\n1977-01-01,0.840000000,0.290000000,-0.400000000,-0.530000000\n1977-02-01,0.550000000,-0.140000000,-0.400000000,-0.270000000\n1977-03-01,0.400000000,-0.380000000,-0.270000000,-0.340000000\n1977-04-01,-0.250000000,-0.680000000,-0.250000000,-0.340000000\n1977-05-01,-0.030000000,-0.450000000,-0.280000000,-0.390000000\n1977-06-01,0.360000000,0.080000000,-0.200000000,-0.410000000\n1977-07-01,0.450000000,0.030000000,-0.420000000,-0.400000000\n1977-08-01,0.250000000,-0.520000000,-0.620000000,-0.190000000\n1977-09-01,0.440000000,-0.500000000,-0.610000000,-0.120000000\n1977-10-01,0.890000000,-0.190000000,-0.660000000,-0.310000000\n1977-11-01,1.030000000,-0.290000000,-0.570000000,-0.270000000\n1977-12-01,1.090000000,-0.560000000,-0.440000000,-0.340000000\n1978-01-01,0.720000000,-0.070000000,-0.130000000,-0.520000000\n1978-02-01,0.510000000,-0.350000000,0.020000000,-0.560000000\n1978-03-01,0.060000000,-0.770000000,0.090000000,-0.800000000\n1978-04-01,-0.370000000,-0.370000000,0.100000000,-0.960000000\n1978-05-01,-0.500000000,-0.540000000,-0.130000000,-0.990000000\n1978-06-01,-0.450000000,-0.860000000,-0.390000000,-0.780000000\n1978-07-01,-0.480000000,-0.400000000,-0.600000000,-0.330000000\n1978-08-01,-0.620000000,-0.640000000,-0.660000000,-0.400000000\n1978-09-01,-0.600000000,-0.300000000,-0.590000000,-0.500000000\n1978-10-01,-0.340000000,-0.430000000,-0.350000000,-0.610000000\n1978-11-01,-0.150000000,0.040000000,-0.300000000,-0.610000000\n1978-12-01,0.070000000,0.230000000,-0.480000000,-0.430000000\n1979-01-01,0.050000000,0.230000000,-0.080000000,-0.490000000\n1979-02-01,0.090000000,-0.580000000,0.100000000,-0.510000000\n1979-03-01,0.130000000,-0.250000000,-0.010000000,-0.650000000\n1979-04-01,0.170000000,0.510000000,0.140000000,-0.300000000\n1979-05-01,-0.090000000,0.270000000,0.150000000,-0.270000000\n1979-06-01,0.040000000,0.760000000,0.180000000,-0.210000000\n1979-07-01,-0.200000000,0.450000000,0.010000000,-0.120000000\n1979-08-01,0.210000000,0.500000000,-0.250000000,0.070000000\n1979-09-01,0.480000000,1.020000000,-0.270000000,0.140000000\n1979-10-01,0.320000000,0.670000000,-0.320000000,-0.170000000\n1979-11-01,0.350000000,0.120000000,-0.200000000,-0.120000000\n1979-12-01,0.690000000,-0.340000000,-0.030000000,-0.280000000\n1980-01-01,0.550000000,-0.100000000,0.150000000,-0.480000000\n1980-02-01,0.330000000,-0.250000000,0.230000000,-0.610000000\n1980-03-01,0.090000000,0.020000000,0.020000000,-0.610000000\n1980-04-01,0.200000000,-0.000000000,0.400000000,-0.420000000\n1980-05-01,0.130000000,0.250000000,0.500000000,-0.380000000\n1980-06-01,0.370000000,0.210000000,0.230000000,-0.390000000\n1980-07-01,0.020000000,-0.440000000,-0.080000000,-0.310000000\n1980-08-01,-0.230000000,-0.440000000,-0.040000000,-0.160000000\n1980-09-01,-0.120000000,-0.110000000,-0.170000000,-0.040000000\n1980-10-01,-0.110000000,-0.520000000,-0.380000000,-0.250000000\n1980-11-01,0.100000000,-0.680000000,-0.280000000,-0.430000000\n1980-12-01,0.360000000,-1.130000000,-0.310000000,-0.860000000\n1981-01-01,-0.460000000,-1.290000000,0.030000000,-0.610000000\n1981-02-01,-0.450000000,-0.830000000,-0.010000000,-0.730000000\n1981-03-01,-0.020000000,-0.190000000,0.450000000,-0.990000000\n1981-04-01,-0.170000000,-0.310000000,0.180000000,-0.830000000\n1981-05-01,-0.110000000,-0.220000000,0.160000000,-0.750000000\n1981-06-01,-0.150000000,-0.060000000,-0.060000000,-0.530000000\n1981-07-01,-0.430000000,-0.560000000,-0.120000000,-0.480000000\n1981-08-01,-0.180000000,-0.550000000,-0.240000000,-0.070000000\n1981-09-01,-0.070000000,-0.240000000,-0.230000000,0.050000000\n1981-10-01,-0.030000000,-0.520000000,-0.390000000,0.120000000\n1981-11-01,-0.270000000,-0.550000000,-0.180000000,0.080000000\n1981-12-01,0.070000000,-0.170000000,-0.020000000,-0.160000000\n1982-01-01,-0.040000000,-0.120000000,-0.060000000,-0.300000000\n1982-02-01,-0.130000000,-0.350000000,-0.240000000,-0.470000000\n1982-03-01,-0.020000000,-0.720000000,-0.230000000,-0.560000000\n1982-04-01,0.240000000,-0.510000000,-0.320000000,-0.660000000\n1982-05-01,0.650000000,0.080000000,-0.240000000,-0.710000000\n1982-06-01,0.920000000,0.270000000,-0.400000000,-0.760000000\n1982-07-01,0.640000000,0.600000000,-0.540000000,-0.700000000\n1982-08-01,0.930000000,1.090000000,-0.640000000,-0.770000000\n1982-09-01,1.390000000,2.010000000,-0.660000000,-0.460000000\n1982-10-01,2.000000000,2.280000000,-0.770000000,-0.330000000\n1982-11-01,2.070000000,3.080000000,-0.770000000,-0.460000000\n1982-12-01,2.330000000,3.200000000,-0.790000000,-0.430000000\n1983-01-01,2.430000000,2.730000000,-0.610000000,-0.010000000\n1983-02-01,2.220000000,2.030000000,-0.190000000,0.110000000\n1983-03-01,1.690000000,2.340000000,0.280000000,-0.480000000\n1983-04-01,1.120000000,3.360000000,0.290000000,-0.560000000\n1983-05-01,1.120000000,4.100000000,-0.030000000,-0.800000000\n1983-06-01,0.620000000,4.560000000,0.100000000,-0.760000000\n1983-07-01,-0.110000000,3.930000000,-0.050000000,-0.560000000\n1983-08-01,-0.130000000,3.110000000,-0.330000000,-0.280000000\n1983-09-01,-0.500000000,2.070000000,-0.560000000,-0.110000000\n1983-10-01,-1.030000000,1.020000000,-0.580000000,-0.310000000\n1983-11-01,-1.130000000,0.560000000,-0.260000000,-0.120000000\n1983-12-01,-0.950000000,0.220000000,-0.060000000,0.120000000\n1984-01-01,-0.800000000,-0.290000000,-0.300000000,0.350000000\n1984-02-01,-0.530000000,-0.770000000,-0.380000000,0.380000000\n1984-03-01,-0.480000000,-0.410000000,-0.250000000,0.440000000\n1984-04-01,-0.600000000,-0.250000000,-0.340000000,0.190000000\n1984-05-01,-0.640000000,-1.110000000,-0.360000000,-0.080000000\n1984-06-01,-0.900000000,-0.920000000,-0.610000000,0.140000000\n1984-07-01,-0.400000000,-0.540000000,-0.630000000,0.290000000\n1984-08-01,-0.400000000,-0.520000000,-0.620000000,0.570000000\n1984-09-01,-0.350000000,-0.360000000,-0.620000000,0.690000000\n1984-10-01,-0.830000000,-0.610000000,-0.920000000,0.360000000\n1984-11-01,-1.180000000,-0.180000000,-0.770000000,0.130000000\n1984-12-01,-1.530000000,-0.500000000,-0.790000000,-0.010000000\n1985-01-01,-0.890000000,-0.770000000,-0.680000000,-0.020000000\n1985-02-01,-0.910000000,-1.130000000,-0.550000000,0.250000000\n1985-03-01,-0.960000000,-0.590000000,-0.770000000,-0.210000000\n1985-04-01,-0.980000000,-1.380000000,-0.810000000,-0.160000000\n1985-05-01,-0.900000000,-1.410000000,-0.910000000,-0.100000000\n1985-06-01,-0.760000000,-1.250000000,-0.540000000,-0.270000000\n1985-07-01,-0.610000000,-1.040000000,-0.320000000,0.020000000\n1985-08-01,-0.490000000,-1.020000000,-0.420000000,-0.100000000\n1985-09-01,-0.570000000,-0.830000000,-0.540000000,-0.130000000\n1985-10-01,-0.640000000,-0.820000000,-0.570000000,0.050000000\n1985-11-01,-0.430000000,-0.920000000,-0.390000000,0.060000000\n1985-12-01,-0.400000000,-0.750000000,-0.540000000,0.030000000\n1986-01-01,-0.790000000,-0.590000000,-0.860000000,-0.140000000\n1986-02-01,-0.640000000,-0.350000000,-0.800000000,0.200000000\n1986-03-01,-0.510000000,-0.840000000,-0.760000000,0.040000000\n1986-04-01,-0.410000000,-0.860000000,-0.800000000,-0.030000000\n1986-05-01,-0.470000000,-0.820000000,-0.740000000,-0.080000000\n1986-06-01,-0.080000000,-0.900000000,-0.840000000,0.100000000\n1986-07-01,0.160000000,-0.140000000,-0.650000000,-0.060000000\n1986-08-01,0.240000000,0.170000000,-0.740000000,0.010000000\n1986-09-01,0.590000000,-0.130000000,-0.340000000,0.160000000\n1986-10-01,0.900000000,0.140000000,-0.630000000,-0.080000000\n1986-11-01,1.040000000,0.280000000,-0.910000000,-0.350000000\n1986-12-01,0.980000000,0.530000000,-0.980000000,-0.240000000\n1987-01-01,1.190000000,0.770000000,-0.570000000,-0.320000000\n1987-02-01,1.170000000,1.020000000,-0.350000000,-0.190000000\n1987-03-01,1.240000000,1.150000000,0.060000000,-0.180000000\n1987-04-01,0.890000000,1.510000000,0.170000000,-0.270000000\n1987-05-01,0.910000000,1.350000000,0.170000000,-0.060000000\n1987-06-01,1.240000000,1.060000000,0.370000000,0.110000000\n1987-07-01,1.340000000,1.190000000,0.210000000,0.250000000\n1987-08-01,1.480000000,0.910000000,0.220000000,0.240000000\n1987-09-01,1.530000000,1.310000000,0.020000000,0.170000000\n1987-10-01,1.360000000,1.500000000,0.020000000,0.190000000\n1987-11-01,1.290000000,0.900000000,0.050000000,0.450000000\n1987-12-01,1.050000000,0.890000000,0.220000000,0.130000000\n1988-01-01,0.690000000,0.090000000,-0.090000000,0.300000000\n1988-02-01,0.350000000,-0.270000000,-0.200000000,0.390000000\n1988-03-01,0.290000000,-0.520000000,0.120000000,0.430000000\n1988-04-01,-0.490000000,-0.900000000,0.220000000,0.280000000\n1988-05-01,-1.050000000,-1.080000000,0.080000000,0.370000000\n1988-06-01,-1.460000000,-1.960000000,0.100000000,0.420000000\n1988-07-01,-1.540000000,-1.540000000,-0.000000000,0.200000000\n1988-08-01,-1.440000000,-1.590000000,-0.310000000,0.300000000\n1988-09-01,-1.330000000,-1.430000000,-0.320000000,0.160000000\n1988-10-01,-2.090000000,-1.260000000,-0.630000000,0.150000000\n1988-11-01,-2.180000000,-0.650000000,-0.540000000,-0.050000000\n1988-12-01,-1.980000000,-0.970000000,-0.520000000,-0.070000000\n1989-01-01,-1.950000000,-0.280000000,-0.670000000,0.080000000\n1989-02-01,-1.370000000,0.050000000,-0.660000000,-0.040000000\n1989-03-01,-1.330000000,-0.450000000,-0.820000000,0.000000000\n1989-04-01,-1.110000000,-0.490000000,-1.040000000,-0.020000000\n1989-05-01,-0.800000000,-1.160000000,-0.860000000,-0.010000000\n1989-06-01,-0.640000000,-0.730000000,-0.420000000,0.000000000\n1989-07-01,-0.470000000,-0.440000000,-0.030000000,0.000000000\n1989-08-01,-0.580000000,-0.460000000,0.050000000,0.190000000\n1989-09-01,-0.380000000,-0.470000000,-0.320000000,0.060000000\n1989-10-01,-0.400000000,-0.290000000,-0.300000000,0.060000000\n1989-11-01,-0.300000000,-0.380000000,-0.350000000,-0.240000000\n1989-12-01,-0.130000000,-0.430000000,-0.070000000,-0.510000000\n1990-01-01,0.020000000,-0.330000000,-0.330000000,-0.680000000\n1990-02-01,0.360000000,-0.270000000,-0.280000000,-0.140000000\n1990-03-01,0.200000000,-0.410000000,-0.200000000,-0.030000000\n1990-04-01,0.260000000,-0.330000000,-0.100000000,0.040000000\n1990-05-01,0.320000000,-0.230000000,-0.070000000,-0.400000000\n1990-06-01,0.000000000,-0.050000000,-0.270000000,-0.230000000\n1990-07-01,0.150000000,-0.870000000,-0.220000000,-0.370000000\n1990-08-01,0.170000000,-0.610000000,-0.040000000,-0.330000000\n1990-09-01,0.120000000,-0.280000000,0.020000000,-0.220000000\n1990-10-01,0.130000000,-0.730000000,-0.010000000,-0.130000000\n1990-11-01,0.080000000,-0.550000000,0.020000000,-0.010000000\n1990-12-01,0.320000000,-0.440000000,-0.120000000,-0.040000000\n1991-01-01,0.510000000,-0.090000000,-0.390000000,-0.210000000\n1991-02-01,0.320000000,-0.100000000,-0.290000000,-0.200000000\n1991-03-01,0.100000000,0.000000000,-0.370000000,-0.210000000\n1991-04-01,0.210000000,-0.680000000,-0.540000000,-0.000000000\n1991-05-01,0.440000000,-0.010000000,-0.620000000,0.140000000\n1991-06-01,0.650000000,0.240000000,-0.670000000,0.130000000\n1991-07-01,0.620000000,0.600000000,-0.670000000,-0.200000000\n1991-08-01,0.430000000,0.000000000,-0.620000000,-0.520000000\n1991-09-01,0.220000000,0.160000000,-0.520000000,-0.510000000\n1991-10-01,0.910000000,0.420000000,-0.530000000,-0.590000000\n1991-11-01,1.140000000,0.640000000,-0.460000000,-0.520000000\n1991-12-01,1.620000000,0.450000000,-0.470000000,-0.340000000\n1992-01-01,1.610000000,0.360000000,-0.240000000,-0.610000000\n1992-02-01,1.620000000,0.440000000,-0.140000000,-0.530000000\n1992-03-01,1.500000000,0.920000000,-0.340000000,-0.740000000\n1992-04-01,1.410000000,1.720000000,-0.440000000,-0.880000000\n1992-05-01,1.270000000,1.830000000,-0.390000000,-1.010000000\n1992-06-01,0.530000000,0.710000000,-0.360000000,-1.110000000\n1992-07-01,0.270000000,-0.050000000,-0.290000000,-0.860000000\n1992-08-01,-0.150000000,-0.240000000,-0.560000000,-0.860000000\n1992-09-01,-0.140000000,-0.500000000,-0.580000000,-0.700000000\n1992-10-01,-0.350000000,-0.260000000,-0.480000000,-0.740000000\n1992-11-01,-0.140000000,-0.250000000,-0.610000000,-0.970000000\n1992-12-01,0.040000000,-0.310000000,-0.520000000,-0.630000000\n1993-01-01,0.280000000,0.030000000,-0.500000000,-0.590000000\n1993-02-01,0.420000000,0.200000000,-0.430000000,-0.390000000\n1993-03-01,0.470000000,0.530000000,-0.380000000,-0.440000000\n1993-04-01,0.920000000,0.830000000,-0.220000000,-0.110000000\n1993-05-01,0.930000000,1.130000000,-0.250000000,-0.320000000\n1993-06-01,0.640000000,0.880000000,-0.180000000,-0.340000000\n1993-07-01,0.330000000,0.320000000,-0.440000000,-0.210000000\n1993-08-01,0.160000000,0.210000000,-0.450000000,-0.160000000\n1993-09-01,0.260000000,0.120000000,-0.500000000,0.300000000\n1993-10-01,0.360000000,0.320000000,-0.560000000,0.470000000\n1993-11-01,0.280000000,-0.230000000,-0.650000000,0.570000000\n1993-12-01,0.190000000,-0.290000000,-0.700000000,0.240000000\n1994-01-01,0.040000000,-0.070000000,-0.830000000,-0.010000000\n1994-02-01,-0.160000000,-0.330000000,-0.780000000,-0.140000000\n1994-03-01,-0.060000000,-1.130000000,-0.760000000,-0.100000000\n1994-04-01,0.030000000,-1.220000000,-0.620000000,-0.130000000\n1994-05-01,0.140000000,-0.910000000,-0.700000000,-0.230000000\n1994-06-01,0.270000000,-0.590000000,-0.670000000,-0.490000000\n1994-07-01,0.160000000,-0.820000000,-0.670000000,-0.520000000\n1994-08-01,0.520000000,-0.900000000,-0.630000000,-0.370000000\n1994-09-01,0.400000000,-0.050000000,-0.620000000,-0.300000000\n1994-10-01,0.900000000,0.650000000,-0.500000000,-0.270000000\n1994-11-01,1.140000000,0.590000000,-0.340000000,-0.110000000\n1994-12-01,1.210000000,0.680000000,-0.420000000,-0.070000000\n1995-01-01,1.100000000,0.910000000,-0.300000000,0.070000000\n1995-02-01,0.870000000,0.240000000,-0.220000000,0.340000000\n1995-03-01,0.490000000,-0.570000000,-0.210000000,0.390000000\n1995-04-01,0.270000000,-0.990000000,-0.040000000,0.290000000\n1995-05-01,0.030000000,-1.010000000,0.100000000,0.240000000\n1995-06-01,0.080000000,-0.530000000,0.230000000,0.330000000\n1995-07-01,0.030000000,-0.250000000,0.310000000,0.090000000\n1995-08-01,-0.380000000,-0.520000000,0.250000000,-0.260000000\n1995-09-01,-0.570000000,-0.280000000,0.030000000,-0.190000000\n1995-10-01,-0.720000000,-0.410000000,-0.060000000,-0.270000000\n1995-11-01,-0.780000000,-0.260000000,0.200000000,-0.060000000\n1995-12-01,-0.720000000,-0.690000000,0.170000000,-0.260000000\n1996-01-01,-0.650000000,-0.610000000,0.490000000,-0.150000000\n1996-02-01,-0.660000000,-0.320000000,0.220000000,0.160000000\n1996-03-01,-0.480000000,-0.070000000,0.070000000,0.120000000\n1996-04-01,-0.340000000,-1.280000000,0.150000000,0.340000000\n1996-05-01,-0.400000000,-0.810000000,0.160000000,0.430000000\n1996-06-01,-0.120000000,-1.070000000,-0.060000000,0.560000000\n1996-07-01,-0.140000000,-0.980000000,-0.150000000,0.410000000\n1996-08-01,-0.300000000,-0.460000000,-0.360000000,0.150000000\n1996-09-01,-0.340000000,-0.530000000,-0.220000000,-0.010000000\n1996-10-01,-0.280000000,-0.800000000,-0.340000000,-0.100000000\n1996-11-01,-0.300000000,-0.790000000,-0.280000000,-0.060000000\n1996-12-01,-0.430000000,-1.030000000,-0.150000000,-0.330000000\n1997-01-01,-0.430000000,-0.560000000,0.060000000,-0.550000000\n1997-02-01,-0.240000000,-0.030000000,0.160000000,-0.600000000\n1997-03-01,-0.060000000,0.820000000,-0.000000000,-0.820000000\n1997-04-01,0.340000000,0.980000000,0.120000000,-0.810000000\n1997-05-01,0.870000000,2.160000000,0.230000000,-0.890000000\n1997-06-01,1.150000000,3.170000000,0.290000000,-0.980000000\n1997-07-01,1.600000000,3.630000000,0.060000000,-0.530000000\n1997-08-01,1.940000000,3.980000000,-0.170000000,-0.290000000\n1997-09-01,2.100000000,4.050000000,-0.120000000,0.060000000\n1997-10-01,2.290000000,4.000000000,0.160000000,0.470000000\n1997-11-01,2.420000000,4.120000000,0.270000000,0.600000000\n1997-12-01,2.300000000,4.270000000,0.150000000,0.760000000\n1998-01-01,2.420000000,3.550000000,0.190000000,0.380000000\n1998-02-01,2.080000000,2.880000000,0.590000000,0.470000000\n1998-03-01,1.490000000,2.880000000,0.590000000,0.340000000\n1998-04-01,0.900000000,3.000000000,0.430000000,0.230000000\n1998-05-01,0.680000000,3.010000000,0.390000000,0.200000000\n1998-06-01,-0.390000000,2.000000000,0.420000000,0.350000000\n1998-07-01,-0.730000000,1.310000000,0.290000000,0.590000000\n1998-08-01,-0.830000000,0.940000000,0.330000000,0.400000000\n1998-09-01,-0.820000000,0.310000000,0.130000000,0.260000000\n1998-10-01,-1.190000000,0.120000000,0.070000000,-0.010000000\n1998-11-01,-1.230000000,-0.210000000,0.120000000,0.140000000\n1998-12-01,-1.510000000,-0.140000000,0.000000000,-0.150000000\n1999-01-01,-1.530000000,-0.230000000,-0.350000000,-0.200000000\n1999-02-01,-1.410000000,0.050000000,-0.380000000,-0.220000000\n1999-03-01,-0.920000000,0.300000000,-0.380000000,-0.020000000\n1999-04-01,-0.810000000,-0.780000000,-0.280000000,0.140000000\n1999-05-01,-0.870000000,-0.430000000,-0.090000000,0.180000000\n1999-06-01,-0.950000000,-0.550000000,-0.100000000,0.180000000\n1999-07-01,-0.840000000,-0.650000000,-0.100000000,0.380000000\n1999-08-01,-0.980000000,-0.510000000,0.010000000,0.350000000\n1999-09-01,-0.840000000,-0.970000000,0.010000000,0.090000000\n1999-10-01,-1.030000000,-0.760000000,-0.180000000,0.050000000\n1999-11-01,-1.410000000,-1.110000000,-0.180000000,0.150000000\n1999-12-01,-1.540000000,-1.060000000,-0.020000000,0.100000000\n2000-01-01,-1.790000000,-0.890000000,-0.420000000,-0.120000000\n2000-02-01,-1.530000000,-0.470000000,-0.280000000,0.120000000\n2000-03-01,-1.260000000,-0.560000000,-0.090000000,0.050000000\n2000-04-01,-0.800000000,0.460000000,-0.170000000,0.060000000\n2000-05-01,-0.800000000,-0.290000000,-0.270000000,-0.050000000\n2000-06-01,-0.750000000,-0.650000000,-0.400000000,-0.110000000\n2000-07-01,-0.570000000,-0.570000000,-0.270000000,-0.100000000\n2000-08-01,-0.360000000,-0.590000000,-0.190000000,-0.000000000\n2000-09-01,-0.390000000,-0.340000000,-0.360000000,-0.130000000\n2000-10-01,-0.550000000,-0.490000000,-0.420000000,-0.120000000\n2000-11-01,-0.750000000,-0.700000000,-0.430000000,-0.110000000\n2000-12-01,-0.920000000,-0.550000000,-0.490000000,-0.270000000\n2001-01-01,-0.880000000,-0.760000000,-0.380000000,-0.170000000\n2001-02-01,-0.630000000,-0.330000000,-0.320000000,-0.100000000\n2001-03-01,-0.480000000,0.350000000,-0.010000000,-0.060000000\n2001-04-01,-0.300000000,0.420000000,-0.110000000,0.070000000\n2001-05-01,-0.300000000,-0.640000000,-0.140000000,0.120000000\n2001-06-01,-0.110000000,-0.860000000,-0.010000000,0.040000000\n2001-07-01,0.010000000,-0.700000000,0.020000000,-0.110000000\n2001-08-01,-0.070000000,-0.770000000,0.000000000,-0.060000000\n2001-09-01,-0.280000000,-1.200000000,0.130000000,-0.040000000\n2001-10-01,-0.260000000,-1.140000000,0.060000000,-0.030000000\n2001-11-01,-0.280000000,-0.940000000,0.310000000,-0.020000000\n2001-12-01,-0.460000000,-1.120000000,0.370000000,-0.250000000\n2002-01-01,-0.140000000,-0.940000000,0.530000000,-0.190000000\n2002-02-01,0.000000000,-0.300000000,0.350000000,-0.330000000\n2002-03-01,0.110000000,0.620000000,0.210000000,-0.060000000\n2002-04-01,0.140000000,0.330000000,-0.180000000,-0.000000000\n2002-05-01,0.210000000,0.180000000,-0.400000000,0.150000000\n2002-06-01,0.680000000,-0.400000000,-0.380000000,0.020000000\n2002-07-01,0.570000000,-0.440000000,-0.330000000,0.220000000\n2002-08-01,0.700000000,-0.260000000,-0.380000000,-0.190000000\n2002-09-01,0.820000000,0.040000000,-0.310000000,-0.230000000\n2002-10-01,1.160000000,0.570000000,-0.070000000,-0.090000000\n2002-11-01,1.410000000,0.780000000,-0.090000000,-0.250000000\n2002-12-01,1.410000000,0.690000000,-0.090000000,0.080000000\n2003-01-01,0.980000000,0.100000000,0.020000000,0.220000000\n2003-02-01,0.640000000,-0.210000000,-0.170000000,0.300000000\n2003-03-01,0.480000000,-0.490000000,-0.110000000,0.280000000\n2003-04-01,-0.030000000,-0.900000000,-0.150000000,0.110000000\n2003-05-01,-0.520000000,-1.350000000,-0.140000000,0.020000000\n2003-06-01,-0.190000000,-0.910000000,-0.020000000,-0.210000000\n2003-07-01,0.140000000,-0.580000000,0.090000000,0.140000000\n2003-08-01,0.050000000,0.080000000,0.260000000,0.300000000\n2003-09-01,0.150000000,-0.430000000,0.280000000,0.340000000\n2003-10-01,0.460000000,-0.070000000,0.380000000,0.310000000\n2003-11-01,0.390000000,0.290000000,0.170000000,0.360000000\n2003-12-01,0.320000000,0.310000000,0.200000000,0.300000000\n2004-01-01,0.260000000,-0.030000000,0.250000000,0.070000000\n2004-02-01,0.170000000,-0.200000000,0.340000000,-0.050000000\n2004-03-01,-0.100000000,-0.340000000,0.200000000,-0.360000000\n2004-04-01,0.060000000,-0.450000000,0.300000000,-0.570000000\n2004-05-01,0.100000000,-0.960000000,-0.030000000,-0.470000000\n2004-06-01,0.140000000,-0.690000000,0.020000000,-0.390000000\n2004-07-01,0.410000000,-0.700000000,0.180000000,-0.180000000\n2004-08-01,0.660000000,-0.440000000,0.350000000,0.020000000\n2004-09-01,0.670000000,-0.100000000,0.210000000,0.180000000\n2004-10-01,0.730000000,0.110000000,0.200000000,-0.080000000\n2004-11-01,0.620000000,0.390000000,0.320000000,0.060000000\n2004-12-01,0.710000000,0.270000000,0.240000000,0.170000000\n2005-01-01,0.560000000,0.030000000,0.220000000,0.280000000\n2005-02-01,0.260000000,-0.610000000,0.200000000,0.220000000\n2005-03-01,0.280000000,-1.150000000,0.530000000,0.260000000\n2005-04-01,0.280000000,-0.360000000,0.650000000,-0.020000000\n2005-05-01,0.300000000,0.220000000,0.810000000,-0.400000000\n2005-06-01,0.220000000,-0.150000000,0.670000000,-0.570000000\n2005-07-01,-0.010000000,-0.060000000,0.570000000,-0.570000000\n2005-08-01,-0.040000000,-0.300000000,0.410000000,-0.190000000\n2005-09-01,-0.080000000,-0.610000000,0.350000000,-0.230000000\n2005-10-01,-0.150000000,-1.050000000,0.120000000,-0.220000000\n2005-11-01,-0.440000000,-1.410000000,0.090000000,-0.260000000\n2005-12-01,-0.750000000,-1.190000000,0.260000000,-0.290000000\n2006-01-01,-0.980000000,-0.690000000,0.160000000,-0.300000000\n2006-02-01,-0.710000000,0.330000000,0.010000000,-0.040000000\n2006-03-01,-0.730000000,-0.270000000,0.020000000,0.060000000\n2006-04-01,-0.300000000,-0.890000000,0.260000000,-0.000000000\n2006-05-01,-0.110000000,-0.390000000,0.230000000,-0.110000000\n2006-06-01,0.090000000,-0.190000000,0.280000000,0.100000000\n2006-07-01,0.030000000,0.180000000,0.090000000,0.050000000\n2006-08-01,0.370000000,0.510000000,0.200000000,0.130000000\n2006-09-01,0.620000000,0.780000000,0.310000000,0.130000000\n2006-10-01,0.760000000,1.200000000,0.310000000,0.010000000\n2006-11-01,0.980000000,0.910000000,0.400000000,-0.100000000\n2006-12-01,1.100000000,0.620000000,0.180000000,0.050000000\n2007-01-01,0.590000000,0.570000000,0.210000000,-0.090000000\n2007-02-01,0.120000000,0.060000000,0.380000000,-0.060000000\n2007-03-01,-0.150000000,-0.650000000,0.170000000,-0.310000000\n2007-04-01,-0.160000000,-1.050000000,0.240000000,-0.070000000\n2007-05-01,-0.390000000,-1.470000000,0.010000000,-0.060000000\n2007-06-01,-0.160000000,-1.400000000,-0.080000000,0.120000000\n2007-07-01,-0.370000000,-0.940000000,-0.180000000,-0.020000000\n2007-08-01,-0.570000000,-1.360000000,-0.330000000,-0.040000000\n2007-09-01,-1.040000000,-1.340000000,-0.270000000,0.120000000\n2007-10-01,-1.410000000,-1.650000000,-0.190000000,-0.110000000\n2007-11-01,-1.580000000,-1.950000000,-0.070000000,-0.080000000\n2007-12-01,-1.610000000,-1.630000000,-0.210000000,-0.060000000\n2008-01-01,-1.790000000,-0.800000000,-0.100000000,-0.140000000\n2008-02-01,-1.700000000,-0.160000000,-0.000000000,0.090000000\n2008-03-01,-1.170000000,0.330000000,0.120000000,0.250000000\n2008-04-01,-0.890000000,0.500000000,-0.060000000,0.330000000\n2008-05-01,-0.640000000,-0.320000000,0.070000000,0.470000000\n2008-06-01,-0.440000000,0.260000000,0.150000000,0.390000000\n2008-07-01,-0.040000000,0.470000000,0.170000000,0.400000000\n2008-08-01,-0.040000000,0.670000000,0.120000000,0.310000000\n2008-09-01,-0.280000000,0.320000000,0.200000000,0.130000000\n2008-10-01,-0.300000000,-0.170000000,0.010000000,0.050000000\n2008-11-01,-0.370000000,-0.210000000,-0.040000000,0.130000000\n2008-12-01,-0.900000000,-0.330000000,-0.030000000,0.150000000\n2009-01-01,-1.000000000,-0.170000000,-0.170000000,0.110000000\n2009-02-01,-0.710000000,-0.310000000,-0.550000000,0.080000000\n2009-03-01,-0.720000000,-0.850000000,-0.490000000,0.200000000\n2009-04-01,-0.250000000,0.000000000,-0.470000000,0.210000000\n2009-05-01,0.170000000,0.170000000,-0.430000000,0.440000000\n2009-06-01,0.490000000,0.540000000,-0.190000000,0.280000000\n2009-07-01,0.690000000,0.700000000,0.170000000,0.140000000\n2009-08-01,0.620000000,0.660000000,0.070000000,0.040000000\n2009-09-01,0.680000000,0.220000000,-0.030000000,0.040000000\n2009-10-01,0.960000000,0.220000000,0.200000000,0.230000000\n2009-11-01,1.490000000,0.140000000,0.200000000,0.110000000\n2009-12-01,1.810000000,0.490000000,0.250000000,0.430000000\n2010-01-01,1.430000000,0.360000000,0.550000000,0.460000000\n2010-02-01,1.180000000,0.040000000,0.900000000,0.410000000\n2010-03-01,1.070000000,-0.040000000,1.120000000,0.540000000\n2010-04-01,0.560000000,0.510000000,1.250000000,0.370000000\n2010-05-01,-0.150000000,0.390000000,1.150000000,0.270000000\n2010-06-01,-0.620000000,0.140000000,0.870000000,0.540000000\n2010-07-01,-0.890000000,-1.050000000,0.740000000,0.350000000\n2010-08-01,-1.330000000,-1.230000000,0.630000000,0.290000000\n2010-09-01,-1.560000000,-1.290000000,0.370000000,0.080000000\n2010-10-01,-1.650000000,-1.530000000,0.350000000,0.210000000\n2010-11-01,-1.570000000,-1.390000000,0.390000000,0.220000000\n2010-12-01,-1.630000000,-1.030000000,0.670000000,0.150000000\n2011-01-01,-1.700000000,-0.610000000,0.730000000,0.120000000\n2011-02-01,-1.260000000,0.070000000,0.350000000,0.320000000\n2011-03-01,-0.980000000,-0.430000000,0.250000000,0.380000000\n2011-04-01,-0.740000000,-0.050000000,0.170000000,0.180000000\n2011-05-01,-0.530000000,0.250000000,0.200000000,-0.040000000\n2011-06-01,-0.250000000,0.360000000,0.350000000,-0.110000000\n2011-07-01,-0.230000000,0.270000000,0.150000000,0.020000000\n2011-08-01,-0.660000000,-0.200000000,0.150000000,0.000000000\n2011-09-01,-0.760000000,-1.010000000,-0.010000000,0.090000000\n2011-10-01,-0.930000000,-0.710000000,0.020000000,0.060000000\n2011-11-01,-1.090000000,-0.800000000,-0.100000000,-0.230000000\n2011-12-01,-1.050000000,-0.680000000,-0.060000000,-0.740000000\n2012-01-01,-0.930000000,-0.390000000,-0.090000000,-0.660000000\n2012-02-01,-0.610000000,0.400000000,-0.190000000,-0.780000000\n2012-03-01,-0.480000000,-0.020000000,-0.410000000,-0.430000000\n2012-04-01,-0.290000000,0.480000000,-0.330000000,-0.640000000\n2012-05-01,-0.180000000,0.430000000,-0.110000000,-0.380000000\n2012-06-01,0.140000000,0.990000000,0.070000000,-0.270000000\n2012-07-01,0.440000000,0.820000000,0.020000000,-0.290000000\n2012-08-01,0.660000000,0.270000000,0.190000000,-0.280000000\n2012-09-01,0.440000000,0.040000000,0.340000000,-0.130000000\n2012-10-01,0.230000000,-0.390000000,0.350000000,-0.190000000\n2012-11-01,0.330000000,-0.460000000,0.440000000,-0.160000000\n2012-12-01,-0.130000000,-0.610000000,0.260000000,-0.090000000\n2013-01-01,-0.420000000,-0.700000000,0.190000000,0.050000000\n2013-02-01,-0.400000000,-0.320000000,0.220000000,-0.020000000\n2013-03-01,-0.140000000,-0.380000000,0.550000000,-0.090000000\n2013-04-01,-0.080000000,-1.320000000,0.370000000,0.040000000\n2013-05-01,-0.280000000,-1.440000000,0.440000000,0.130000000\n2013-06-01,-0.330000000,-1.440000000,0.070000000,-0.030000000\n2013-07-01,-0.280000000,-1.360000000,0.060000000,-0.030000000\n2013-08-01,-0.290000000,-0.760000000,0.220000000,0.050000000\n2013-09-01,-0.090000000,-0.770000000,0.230000000,-0.120000000\n2013-10-01,-0.240000000,-0.160000000,0.230000000,-0.180000000\n2013-11-01,-0.020000000,-0.500000000,0.020000000,-0.260000000\n2013-12-01,-0.090000000,-0.250000000,-0.020000000,-0.160000000\n2014-01-01,-0.420000000,0.370000000,-0.120000000,-0.230000000\n2014-02-01,-0.450000000,-0.670000000,-0.250000000,-0.050000000\n2014-03-01,-0.070000000,-0.510000000,-0.500000000,-0.130000000\n2014-04-01,0.280000000,-0.250000000,-0.360000000,-0.080000000\n2014-05-01,0.450000000,0.740000000,-0.270000000,0.140000000\n2014-06-01,0.480000000,1.170000000,-0.290000000,0.150000000\n2014-07-01,0.130000000,1.040000000,-0.130000000,-0.090000000\n2014-08-01,0.140000000,0.670000000,-0.100000000,0.050000000\n2014-09-01,0.370000000,0.150000000,0.130000000,0.110000000\n2014-10-01,0.480000000,0.340000000,0.130000000,-0.060000000\n2014-11-01,0.890000000,0.570000000,0.040000000,-0.500000000\n2014-12-01,0.770000000,0.140000000,0.050000000,-0.380000000\n2015-01-01,0.590000000,-0.160000000,-0.080000000,0.100000000\n2015-02-01,0.570000000,-0.620000000,-0.060000000,0.150000000\n2015-03-01,0.480000000,-0.010000000,-0.270000000,-0.110000000\n2015-04-01,0.900000000,0.640000000,-0.400000000,0.050000000\n2015-05-01,1.040000000,1.590000000,-0.130000000,0.110000000\n2015-06-01,1.280000000,2.190000000,-0.310000000,0.070000000\n2015-07-01,1.560000000,2.320000000,-0.120000000,-0.190000000\n2015-08-01,1.870000000,2.080000000,0.090000000,-0.310000000\n2015-09-01,2.010000000,2.480000000,0.350000000,-0.190000000\n2015-10-01,2.210000000,2.380000000,0.510000000,0.060000000\n2015-11-01,2.570000000,2.210000000,0.180000000,0.190000000\n2015-12-01,2.560000000,2.090000000,0.130000000,0.440000000\n2016-01-01,2.560000000,1.820000000,0.250000000,0.470000000\n2016-02-01,2.110000000,1.090000000,0.280000000,0.320000000\n2016-03-01,1.600000000,1.100000000,0.240000000,0.250000000\n2016-04-01,1.050000000,0.290000000,0.170000000,0.080000000\n2016-05-01,0.450000000,0.450000000,0.180000000,0.010000000\n2016-06-01,0.060000000,0.520000000,0.150000000,0.370000000\n2016-07-01,-0.250000000,-0.060000000,0.100000000,0.370000000\n2016-08-01,-0.480000000,0.190000000,0.150000000,0.330000000\n2016-09-01,-0.460000000,0.250000000,0.010000000,0.110000000\n2016-10-01,-0.750000000,-0.030000000,0.120000000,0.240000000\n2016-11-01,-0.630000000,-0.080000000,0.150000000,0.100000000\n2016-12-01,-0.510000000,0.020000000,0.230000000,0.410000000\n2017-01-01,-0.340000000,0.330000000,0.240000000,0.250000000\n2017-02-01,-0.010000000,0.930000000,0.060000000,-0.180000000\n2017-03-01,-0.090000000,1.560000000,-0.030000000,-0.060000000\n2017-04-01,0.220000000,0.490000000,0.310000000,0.090000000\n2017-05-01,0.300000000,0.250000000,0.380000000,0.120000000\n2017-06-01,0.220000000,0.200000000,0.350000000,0.260000000\n2017-07-01,0.220000000,-0.460000000,0.320000000,0.100000000\n2017-08-01,-0.180000000,-0.760000000,0.330000000,0.390000000\n2017-09-01,-0.560000000,-1.280000000,0.150000000,0.120000000\n2017-10-01,-0.520000000,-1.320000000,0.030000000,0.190000000\n2017-11-01,-0.840000000,-1.440000000,0.220000000,0.020000000\n2017-12-01,-0.850000000,-1.540000000,0.250000000,-0.090000000\n2018-01-01,-0.860000000,-1.250000000,0.030000000,0.290000000\n2018-02-01,-0.730000000,-0.430000000,-0.270000000,-0.270000000\n2018-03-01,-0.730000000,-1.300000000,-0.130000000,-0.180000000\n2018-04-01,-0.360000000,-0.810000000,-0.330000000,-0.030000000\n2018-05-01,-0.120000000,-0.460000000,-0.600000000,-0.070000000\n2018-06-01,0.120000000,-0.470000000,-0.720000000,0.150000000\n2018-07-01,0.270000000,-0.080000000,-0.480000000,0.120000000\n2018-08-01,0.050000000,-0.540000000,-0.320000000,0.210000000\n2018-09-01,0.300000000,0.120000000,-0.040000000,0.210000000\n2018-10-01,0.840000000,0.300000000,-0.130000000,0.310000000\n2018-11-01,1.000000000,0.690000000,-0.320000000,0.210000000\n2018-12-01,0.970000000,0.790000000,-0.330000000,0.400000000\n2019-01-01,0.510000000,0.670000000,-0.170000000,0.410000000\n2019-02-01,0.620000000,0.660000000,-0.040000000,0.410000000\n2019-03-01,0.810000000,-0.180000000,-0.150000000,0.670000000\n2019-04-01,0.670000000,0.360000000,-0.170000000,0.340000000\n2019-05-01,0.610000000,0.240000000,-0.080000000,0.310000000\n2019-06-01,0.660000000,-0.110000000,0.060000000,0.270000000\n2019-07-01,0.410000000,-0.360000000,0.090000000,0.240000000\n2019-08-01,0.190000000,-0.290000000,0.090000000,0.220000000\n2019-09-01,0.110000000,-0.680000000,0.200000000,0.330000000\n2019-10-01,0.550000000,-0.710000000,0.010000000,0.250000000\n2019-11-01,0.740000000,0.020000000,-0.050000000,0.870000000\n2019-12-01,0.510000000,-0.190000000,-0.040000000,0.800000000\n2020-01-01,0.640000000,-0.120000000,0.010000000,0.950000000\n2020-02-01,0.480000000,-0.030000000,0.320000000,0.570000000\n2020-03-01,0.360000000,-0.440000000,0.580000000,0.310000000\n2020-04-01,0.490000000,0.030000000,0.460000000,0.410000000\n2020-05-01,-0.120000000,-0.240000000,0.320000000,0.530000000\n2020-06-01,-0.210000000,-0.430000000,0.470000000,0.290000000\n2020-07-01,-0.040000000,-1.090000000,0.380000000,0.290000000\n2020-08-01,-0.420000000,-0.430000000,0.250000000,0.300000000\n2020-09-01,-0.660000000,-0.850000000,0.150000000,0.090000000\n', 'research/lagged_sources/official_hashes.json': '[\n  {\n    "nome": "treino_tp.nc",\n    "sha256": "012bebbe0e38aefdf4f9e2432c3ca22f71d94ea1a1b5421da9d9e5048632cab0",\n    "bytes": 186685299\n  },\n  {\n    "nome": "treino_tp_alvo.nc",\n    "sha256": "915f67f2627e37399830d1dd92b209ce1160082b0645662675eab17df957abe4",\n    "bytes": 186509303\n  },\n  {\n    "nome": "teste_features.nc",\n    "sha256": "70bcb5c9bb9c011c8901660834b42dc57ff7e54a4f34161d489ec5e9f05c1f23",\n    "bytes": 39057097\n  },\n  {\n    "nome": "sample_submission.csv",\n    "sha256": "34fe449ddb309b0579a87786f76049863833d82c26f183d18e943dc76f1e4c8c",\n    "bytes": 48145117\n  },\n  {\n    "nome": "treino_cloud_cover.nc",\n    "sha256": "607ee1210615a0ca8a5c954737617e668a5f016eb881554cb1468fda74b4fde1",\n    "bytes": 190610590\n  },\n  {\n    "nome": "treino_geopotential_850.nc",\n    "sha256": "ca3baca0177e2e90b42901803f348a4b4affe88bee202b76af3f293fa7035c17",\n    "bytes": 159707312\n  },\n  {\n    "nome": "treino_rel_hum_850.nc",\n    "sha256": "90f62b44b1b461259e7539249914e26cf09993f3bcd572ba4871931e45264b19",\n    "bytes": 180398147\n  },\n  {\n    "nome": "treino_shum_850.nc",\n    "sha256": "35cf8071b68d1e924ad87e0f8bd801d0a66d1d78ae75fc3cb623bde85c3f47bb",\n    "bytes": 175651693\n  },\n  {\n    "nome": "treino_surface_pressure.nc",\n    "sha256": "f00a5e44a03ea923ac0f693b975ef71cd347f8ad3d07692c602e3ac9a25c10fb",\n    "bytes": 160228025\n  },\n  {\n    "nome": "treino_t2.nc",\n    "sha256": "7d8c53b438289cddf87a11414ad10c8167f52ff33c6eff459a3288975079ef23",\n    "bytes": 175952066\n  },\n  {\n    "nome": "treino_temperature_850.nc",\n    "sha256": "1db7b4fdeba87b6a25f7fb6f1e40abec50964234dd9cc918cd34f902b8e41bac",\n    "bytes": 172474226\n  },\n  {\n    "nome": "treino_u_850.nc",\n    "sha256": "73c898f27bea25cf2ab0914bbe5c29acc9794a93f8b6b3affa39854bebccf761",\n    "bytes": 188451602\n  },\n  {\n    "nome": "treino_v_850.nc",\n    "sha256": "7bab583972e5718bb0e0fe0fe6a056bd56c968af1f1497205182f7621e26c241",\n    "bytes": 197354831\n  }\n]', 'research/prospective/cfsv2_adapter.py': '"""CFSv2 audit: actual dates, members, lead time, grid and units."""\nfrom pathlib import Path\nimport hashlib\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport cftime\nLEAD = 1.5\nLAT = np.arange(-61, 17, dtype=np.float32)\nLON_360 = np.arange(269, 337, dtype=np.float32)\nEXCECOES_MEMBROS = {}\nPOLITICA_MEMBROS = \'todos_membros_esperados_sem_lacunas_v1\'\nSCHEMA = \'nimbus_cfsv2_prospectivo_v1\'\nFONTE = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\n\ndef exigir(ok, mensagem):\n    if not bool(ok):\n        raise ValueError(mensagem)\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for parte in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(parte)\n    return h.hexdigest()\n\ndef meses_de_S(s):\n    exigir(str(s.attrs.get(\'units\', \'\')).strip() == \'months since 1960-01-01\', \'Unexpected S units.\')\n    exigir(str(s.attrs.get(\'calendar\')) in (\'360\', \'360_day\'), \'Unexpected S calendar.\')\n    valores = np.asarray(s.values, dtype=np.float64)\n    exigir(valores.ndim == 1 and np.isfinite(valores).all(), \'Invalid S.\')\n    exigir(np.equal(valores, np.rint(valores)).all(), \'S contains a fractional month.\')\n    return pd.DatetimeIndex([(pd.Period(\'1960-01\', freq=\'M\') + int(v)).to_timestamp() for v in valores])\n\ndef tabela_inicializacoes(path, origens):\n    with xr.open_dataset(path, decode_times=False, engine=\'netcdf4\') as d:\n        ds = d.load()\n    exigir(set(ds.data_vars) == {\'sampleS\'} and set(ds.sampleS.dims) == {\'S\', \'M\'}, \'Unexpected sampleS table format.\')\n    exigir(meses_de_S(ds.S).equals(pd.DatetimeIndex(origens)), \'sampleS coverage mismatch.\')\n    exigir(np.array_equal(ds.M.values, np.arange(1, 29)), \'Unexpected table member IDs.\')\n    exigir(ds.sampleS.attrs.get(\'units\') == \'days since 1960-01-01\', \'Unexpected sampleS units.\')\n    exigir(str(ds.sampleS.attrs.get(\'calendar\')) in (\'365\', \'365_day\', \'noleap\'), \'Unexpected sampleS calendar.\')\n    resultado = {}\n    for k, origem in enumerate(origens):\n        numeros = ds.sampleS.transpose(\'S\', \'M\').values[k].astype(np.float64)\n        exigir(not np.isinf(numeros).any(), \'Infinite initialization date.\')\n        mascara = np.isfinite(numeros)\n        n = 28 if origem.month == 11 else 24\n        exigir(np.array_equal(mascara, np.arange(28) < n), f\'Member table mismatch in {origem.date()}: expected {n}.\')\n        datas_cf = cftime.num2date(numeros[mascara], \'days since 1960-01-01\', calendar=\'365_day\')\n        datas = pd.DatetimeIndex([pd.Timestamp(t.year, t.month, t.day) for t in datas_cf])\n        alvo = origem + pd.offsets.MonthBegin(1)\n        exigir((datas < alvo).all(), f\'Initialization overlaps the target month: {origem.date()}.\')\n        exigir(datas.min() >= origem - pd.Timedelta(days=40) and datas.max() <= origem + pd.Timedelta(days=7), \'Unexpected pentad initialization window.\')\n        exigir(len(set(datas)) == n // 4, \'Four members per initialization date are required.\')\n        exigir(np.all(np.unique(datas.values, return_counts=True)[1] == 4), \'Pentad grouping mismatch.\')\n        resultado[origem] = dict(mascara=mascara, datas=datas, numeros=numeros, fase_fonte=\'hindcast\' if origem < pd.Timestamp(\'2011-04-01\') else \'operacional\')\n    return resultado\n\ndef carregar_membros(path, origens):\n    with xr.open_dataset(path, decode_times=False, engine=\'netcdf4\') as d:\n        ds = d.load()\n    exigir(set(ds.data_vars) == {\'prec\'}, \'The file must contain only predicted rainfall prec.\')\n    exigir(set(ds.prec.dims) == {\'S\', \'L\', \'M\', \'Y\', \'X\'}, \'Unexpected dimensions.\')\n    exigir(meses_de_S(ds.S).equals(pd.DatetimeIndex(origens)), \'NetCDF months differ from the request.\')\n    exigir(ds.sizes[\'L\'] == 1 and float(ds.L.values[0]) == LEAD, \'Incorrect lead; L=1.5 is required.\')\n    exigir(ds.L.attrs.get(\'units\') == \'months\' and float(ds.L.attrs.get(\'pointwidth\', -1)) == 1, \'A monthly forecast is required, not a multimonth average.\')\n    exigir(ds.prec.attrs.get(\'units\') == \'mm/day\', \'Unexpected units; no conversion will be assumed.\')\n    exigir(ds.prec.attrs.get(\'standard_name\') in (\'precipitation_rate\', \'lwe_precipitation_rate\'), \'Variable is not identified as a precipitation rate.\')\n    exigir(np.array_equal(ds.M.values, np.arange(1, 29)), \'Incomplete or duplicate member IDs.\')\n    exigir(np.array_equal(ds.Y.values, LAT) and np.array_equal(ds.X.values, LON_360), \'Native grid differs from the requested one-degree subset.\')\n    exigir(ds.X.attrs.get(\'units\') == \'degree_east\' and ds.Y.attrs.get(\'units\') == \'degree_north\', \'Geographic unit mismatch.\')\n    valores = ds.prec.transpose(\'S\', \'L\', \'M\', \'Y\', \'X\').values[:, 0].astype(np.float64)\n    marcador = ds.prec.attrs.get(\'file_missing_value\')\n    if marcador is not None and np.isfinite(float(marcador)):\n        valores[valores == float(marcador)] = np.nan\n    return valores\n\ndef membros_utilizaveis(membros, origem, info):\n    esperados = info[\'mascara\']\n    exigir(np.isnan(membros[~esperados]).all(), f\'A member outside the table contains data in {origem}.\')\n    completos = np.isfinite(membros).all(axis=(-1, -2))\n    faltantes = tuple((int(i + 1) for i in np.flatnonzero(esperados & ~completos)))\n    if faltantes:\n        exigir(faltantes == EXCECOES_MEMBROS.get(str(origem.date())), f\'Unexpected missing or partial members in {origem.date()}: {faltantes}.\')\n        exigir(np.isnan(membros[np.array(faltantes) - 1]).all(), \'The exception requires a completely absent member; partial gaps and infinities are rejected.\')\n    usados = esperados & completos\n    exigir((membros[usados] >= 0).all(), f\'Negative precipitation in {origem.date()}. Investigate before proceeding.\')\n    datas = info[\'datas\'][usados[esperados]]\n    return (usados, datas, faltantes)\n\ndef processar_bloco(path, origens, tabela):\n    origens = pd.DatetimeIndex(origens)\n    membros = carregar_membros(path, origens)\n    hash_bruto = sha256(path)\n    medias, linhas, datas_min, datas_max, ns, fases = ([], [], [], [], [], [])\n    for k, origem in enumerate(origens):\n        info = tabela[origem]\n        validos, datas, faltantes = membros_utilizaveis(membros[k], origem, info)\n        v = membros[k, validos]\n        exigir(np.isfinite(v).all(), f\'Missing or partial expected member in {origem}. Do not use a partial ensemble mean.\')\n        exigir((v >= 0).all(), f\'Negative precipitation in {origem}: minimum {v.min():.12g} mm/day. Investigate before proceeding.\')\n        media = v.mean(axis=0, dtype=np.float64).astype(np.float32)\n        alvo = origem + pd.offsets.MonthBegin(1)\n        medias.append(media)\n        ns.append(int(validos.sum()))\n        datas_min.append(datas.min())\n        datas_max.append(datas.max())\n        fases.append(0 if info[\'fase_fonte\'] == \'hindcast\' else 1)\n        linhas.append(dict(time_alvo=str(alvo.date()), time_origem=str(origem.date()), lead=LEAD, membros=ns[-1], membro_ids=\',\'.join(map(str, np.flatnonzero(validos) + 1)), membros_esperados=int(info[\'mascara\'].sum()), membros_indisponiveis_ids=list(faltantes), politica_membros=POLITICA_MEMBROS, inicializacao_mais_antiga=str(datas.min().date()), inicializacao_mais_recente=str(datas.max().date()), inicializacoes_anteriores_ao_alvo=True, fase_fonte=info[\'fase_fonte\'], minimo_membros_mm_day=float(v.min()), maximo_membros_mm_day=float(v.max()), minimo_media_mm_day=float(media.min()), maximo_media_mm_day=float(media.max()), media_espacial_mm_day=float(media.mean(dtype=np.float64)), membros_ausentes_estruturais=28 - int(info[\'mascara\'].sum()), pixels_ausentes_membros_validos=0, negativos_corrigidos=0, arquivo_bruto=path.name, sha256_bruto=hash_bruto))\n    ds = xr.Dataset({\'cfsv2_tp_media\': ((\'time\', \'lat\', \'lon\'), np.stack(medias)), \'n_membros\': (\'time\', np.array(ns, dtype=np.int16)), \'time_origem\': (\'time\', origens.values), \'inicializacao_mais_antiga\': (\'time\', pd.DatetimeIndex(datas_min).values), \'inicializacao_mais_recente\': (\'time\', pd.DatetimeIndex(datas_max).values), \'fase_fonte\': (\'time\', np.array(fases, dtype=np.int8))}, coords={\'time\': (origens.to_period(\'M\') + 1).to_timestamp(), \'lat\': LAT, \'lon\': LON_360 - 360}, attrs=dict(schema=SCHEMA, fonte=FONTE, modelo=\'NCEP-CFSv2\', amostragem=\'PENTAD_SAMPLES_FULL; 24 members, 28 in November\', produto=\'monthly forecast; local arithmetic mean of complete available members\', politica_membros=POLITICA_MEMBROS, lead_iri=LEAD, referencia_temporal=\'nominal S in the preceding month; actual initialization dates audited\', observacoes_de_chuva_utilizadas=\'nenhuma\', climatologia_aplicada=\'nenhuma\', grade=\'native one-degree grid; interpolation is performed in the modeling notebook\'))\n    ds.cfsv2_tp_media.attrs = dict(units=\'mm/day\', long_name=\'Predicted precipitation: CFSv2 ensemble mean\')\n    ds.lat.attrs[\'units\'], ds.lon.attrs[\'units\'] = (\'degrees_north\', \'degrees_east\')\n    ds.fase_fonte.attrs = dict(flag_values=np.array([0, 1], dtype=np.int8), flag_meanings=\'hindcast operacional\')\n    return (ds, linhas)\n', 'research/prospective/cfsv2_compatibility.py': '"""Compare CFSv2 services without fitting, filling gaps or authorizing issuance.\n\nSuccessor files are local serializations of decoded OPeNDAP subsets, not raw\nHTTP responses. Legacy NetCDF responses and HTTP metadata are kept separately.\nEven numerical agreement on sampled months cannot certify a source migration.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport requests\nimport xarray as xr\n\nif __package__:\n    from . import cfsv2_adapter as legacy\nelse:\n    import cfsv2_adapter as legacy\n\n\nSUCCESSOR = \'https://forecast.ccsr.columbia.edu/data/NMME/NOAA-NCEP/CFSv2/pentad_samples/forecast/pr\'\nNOTICE = \'https://iri.columbia.edu/resources/data-library/sunset/\'\nSOURCE_REGRID_NOTE = \'https://github.com/iridl/sigrid/blob/19d023912628a38a72435a335b68ba6f209bb178/server/test-regridding/compare_ingrid.py\'\nATOL = 1e-5  # Diagnostic numerical comparison, not an operational acceptance rule.\nRTOL = 1e-5\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise ValueError(message)\n\n\ndef utc():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef fingerprint(path):\n    b = Path(path).read_bytes()\n    return dict(sha256=hashlib.sha256(b).hexdigest(), bytes=len(b))\n\n\ndef write_json(path, value):\n    with Path(path).open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(value, f, indent=2, allow_nan=False)\n        f.write(\'\\n\')\n\n\ndef expected_mask(origin):\n    return np.arange(28) < (28 if pd.Timestamp(origin).month == 11 else 24)\n\n\ndef successor_fields(ds, origin):\n    """Validate coordinates and units; this does not establish initialization dates."""\n    t = pd.Timestamp(origin)\n    require(t == t.to_period(\'M\').start_time, \'Origin must be the first day of its month.\')\n    require(set(ds.data_vars) == {\'pr\'}, \'Unexpected successor data variables.\')\n    require(set(ds.pr.dims) == {\'S\', \'M\', \'L\', \'Y\', \'X\'}, \'Unexpected successor dimensions.\')\n    require(ds.pr.attrs.get(\'units\') == \'mm/day\', \'Successor precipitation must be in mm/day.\')\n    require(ds.pr.attrs.get(\'standard_name\') == \'lwe_precipitation_rate\', \'Wrong precipitation variable.\')\n    require(pd.DatetimeIndex(ds.S.values).equals(pd.DatetimeIndex([t])), \'Wrong nominal origin.\')\n    for name, values in [(\'M\', np.arange(1, 29)), (\'L\', [1]), (\'Y\', legacy.LAT), (\'X\', legacy.LON_360)]:\n        require(np.array_equal(ds[name].values, values), f\'Wrong successor {name} coordinates.\')\n    require(ds.target.dims == (\'S\', \'L\') and ds.target_bnds.dims == (\'S\', \'L\', \'nbound\'),\n            \'Unexpected target coordinate dimensions.\')\n    want = (t + pd.DateOffset(months=1)).to_datetime64()\n    end = (t + pd.DateOffset(months=2)).to_datetime64()\n    require(ds.target.shape == (1, 1) and ds.target.values[0, 0] == want, \'Wrong target month.\')\n    require(ds.target_bnds.shape == (1, 1, 2) and np.array_equal(ds.target_bnds.values[0, 0], [want, end]),\n            \'Wrong target bounds; lead labels alone are insufficient.\')\n    a = ds.pr.transpose(\'S\', \'M\', \'L\', \'Y\', \'X\').values[0, :, 0].astype(\'float64\')\n    require(not np.isinf(a).any() and not (a[np.isfinite(a)] < 0).any(), \'Invalid precipitation values.\')\n    require(np.isnan(a[~expected_mask(t)]).all(), \'Unexpected structurally inactive members.\')\n    return a\n\n\ndef compare_fields(old, new, origin):\n    """Compare the identical complete member subset; never fill an ensemble."""\n    shape = (28, len(legacy.LAT), len(legacy.LON_360))\n    require(old.shape == new.shape == shape, \'Wrong member/grid shape.\')\n    mask = expected_mask(origin)\n    old_counts = np.isfinite(old).sum(axis=(1, 2))\n    new_counts = np.isfinite(new).sum(axis=(1, 2))\n    size = len(legacy.LAT) * len(legacy.LON_360)\n    common = mask & (old_counts == size) & (new_counts == size)\n    result = dict(\n        origin=str(pd.Timestamp(origin).date()),\n        target=str((pd.Timestamp(origin) + pd.DateOffset(months=1)).date()),\n        expected_members=(np.flatnonzero(mask) + 1).tolist(),\n        legacy_finite_pixels=old_counts.tolist(), successor_finite_pixels=new_counts.tolist(),\n        legacy_complete=bool((old_counts[mask] == size).all()),\n        successor_complete=bool((new_counts[mask] == size).all()),\n        common_complete_members=(np.flatnonzero(common) + 1).tolist(),\n        numerical_agreement_on_overlap=None, numerical_tolerance=dict(atol=ATOL, rtol=RTOL),\n        operationally_authorized=False,\n        interpretation=\'Differences between forecast source fields, not errors against observed rainfall.\',\n    )\n    if not common.any():\n        result[\'status\'] = \'no_complete_overlap\'\n        return result\n    x, y = old[common], new[common]\n    diff = y - x\n    mean_diff = diff.mean(axis=0)\n    result.update(\n        status=\'compared\',\n        numerical_agreement_on_overlap=bool(np.allclose(x, y, atol=ATOL, rtol=RTOL)),\n        ensemble_mean_difference_rmse=float(np.sqrt(np.mean(mean_diff ** 2))),\n        member_field_difference_rmse=float(np.sqrt(np.mean(diff ** 2))),\n        maximum_absolute_member_difference=float(np.abs(diff).max()),\n        mean_signed_difference=float(diff.mean()),\n        memberwise_difference_rmse=np.sqrt(np.mean(diff ** 2, axis=(1, 2))).tolist(),\n    )\n    return result\n\n\ndef legacy_url(origin):\n    t = pd.Timestamp(origin)\n    month = (\'Jan\', \'Feb\', \'Mar\', \'Apr\', \'May\', \'Jun\', \'Jul\', \'Aug\', \'Sep\', \'Oct\', \'Nov\', \'Dec\')[t.month - 1]\n    s = f\'(0000%201%20{month}%20{t.year})\'\n    return legacy.FONTE + f\'S/{s}/{s}/RANGE/L/1.5/VALUES/Y/-61/16/RANGE/X/269/336/RANGE/data.nc\'\n\n\ndef download_raw(url, path):\n    started = utc()\n    with requests.get(url, stream=True, allow_redirects=False, timeout=(20, 90)) as r:\n        require(r.status_code == 200, f\'HTTP {r.status_code} from {url}\')\n        chunks, total = [], 0\n        for chunk in r.iter_content(1024 * 1024):\n            total += len(chunk)\n            require(total <= 32 * 1024 ** 2, \'Diagnostic response exceeded 32 MiB.\')\n            chunks.append(chunk)\n        with path.open(\'xb\') as f:\n            f.write(b\'\'.join(chunks))\n        receipt = dict(url=url, started_at_utc=started, received_at_utc=utc(),\n                       kind=\'raw_http_response\', **fingerprint(path),\n                       headers={k: r.headers[k] for k in [\'Date\', \'Last-Modified\', \'ETag\', \'Content-Type\'] if k in r.headers},\n                       first_publication_time_unknown=True)\n    write_json(path.with_suffix(path.suffix + \'.receipt.json\'), receipt)\n    return receipt\n\n\ndef acquire_pair(folder, origin):\n    """Small regional requests only. No model or observation data are opened."""\n    p = folder / str(pd.Timestamp(origin).date())\n    p.mkdir()  # A new acquisition may not overwrite a previous receipt.\n    print(f\'{origin}: acquiring original and successor subsets\', flush=True)\n    old_path, new_path = p / \'legacy_raw.nc\', p / \'successor_decoded.nc\'\n    old_receipt = download_raw(legacy_url(origin), old_path)\n    started = utc()\n    with xr.open_dataset(SUCCESSOR, engine=\'netcdf4\') as source:\n        subset = source.sel(S=[pd.Timestamp(origin)], L=[1], Y=slice(-61, 16), X=slice(269, 336)).load()\n    ended = utc()\n    # Record the exact decoded subset before validation, including invalid data.\n    subset.to_netcdf(new_path, engine=\'netcdf4\')\n    new_receipt = dict(url=SUCCESSOR, started_at_utc=started, received_at_utc=ended,\n                       kind=\'local_serialization_of_decoded_opendap_subset_not_raw_http\',\n                       selection=dict(S=str(pd.Timestamp(origin).date()), L=1, Y=[-61, 16], X=[269, 336]),\n                       **fingerprint(new_path), first_publication_time_unknown=True,\n                       initialization_dates_by_member_proven=False)\n    write_json(p / \'successor_decoded.receipt.json\', new_receipt)\n    old = legacy.carregar_membros(old_path, pd.DatetimeIndex([pd.Timestamp(origin)]))[0]\n    new = successor_fields(subset, origin)\n    result = compare_fields(old, new, origin)\n    result[\'files\'] = dict(legacy=old_receipt, successor=new_receipt)\n    write_json(p / \'comparison.json\', result)\n    print(f\'{origin}: {len(result["common_complete_members"])} common members; \'\n          f\'mean-field difference RMSE={result.get("ensemble_mean_difference_rmse")}\', flush=True)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\', type=Path, required=True, help=\'A new diagnostic folder.\')\n    parser.add_argument(\'--origins\', nargs=\'+\', required=True, help=\'Nominal origins in YYYY-MM format.\')\n    parser.add_argument(\'--download\', action=\'store_true\', help=\'Explicitly permit public subset downloads.\')\n    args = parser.parse_args()\n    require(args.download, \'Use --download for live acquisition. This command never issues a forecast.\')\n    origins = [pd.Timestamp(x + \'-01\') for x in args.origins]\n    require(len(set(origins)) == len(origins), \'Duplicate origins.\')\n    args.output.mkdir(parents=True, exist_ok=False)\n    report = dict(schema=\'worcap_cfsv2_service_comparison_v1\', started_at_utc=utc(),\n                  provider_notice=NOTICE, provider_regridding_note=SOURCE_REGRID_NOTE,\n                  source_policy_changed=False, model_changed=False, forecast_issued=False, months=[])\n    download_raw(SUCCESSOR + \'.dds\', args.output / \'successor.dds\')\n    download_raw(SUCCESSOR + \'.das\', args.output / \'successor.das\')\n    for origin in origins:\n        report[\'months\'].append(acquire_pair(args.output, origin))\n    report[\'finished_at_utc\'] = utc()\n    report[\'decision\'] = (\'not_numerically_interchangeable\' if any(\n        m[\'numerical_agreement_on_overlap\'] is False for m in report[\'months\'])\n        else \'insufficient_evidence_for_operational_migration\')\n    write_json(args.output / \'report.json\', report)\n    print(report[\'decision\'], flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'research/prospective/collect_cds.py': '"""Download three narrowly scoped GRIB inputs with an existing CDS credential."""\nfrom pathlib import Path\nimport argparse\nimport tempfile\nfrom registry import Registro, deslocar, agora\nfrom prepare_cds import prepare\n\n\ndef request_for(source, target):\n    month = deslocar(target, -1 if source == \'seas5\' else -4)\n    year, month = month.split(\'-\')\n    if source == \'seas5\':\n        return \'seasonal-monthly-single-levels\', dict(originating_centre=\'ecmwf\', system=\'51\',\n            variable=[\'total_precipitation\'], product_type=[\'monthly_mean\'], year=[year], month=[month],\n            leadtime_month=[\'2\'], area=[16, -91, -61, -24], data_format=\'grib\')\n    variables = dict(era5_sl=[\'total_cloud_cover\', \'2m_temperature\', \'surface_pressure\'],\n        era5_pl=[\'geopotential\', \'relative_humidity\', \'specific_humidity\', \'temperature\', \'u_component_of_wind\', \'v_component_of_wind\'])\n    level = \'single\' if source == \'era5_sl\' else \'pressure\'\n    request = dict(product_type=[\'monthly_averaged_reanalysis\'], variable=variables[source], year=[year], month=[month],\n        time=[\'00:00\'], area=[15, -90, -60, -25], grid=[.25, .25], data_format=\'grib\', download_format=\'unarchived\')\n    if source == \'era5_pl\':\n        request[\'pressure_level\'] = [\'850\']\n    return f\'reanalysis-era5-{level}-levels-monthly-means\', request\n\n\ndef execute(registry, target):\n    import cdsapi\n    r = Registro(registry)\n    r.prontidao(target)\n    # cdsapi reads the existing environment/.cdsapirc. Never print or persist a key.\n    client = cdsapi.Client(quiet=True, progress=False, debug=False)\n    records = []\n    for source in [\'era5_sl\', \'era5_pl\', \'seas5\']:\n        if source in r.prontidao(target)[\'fontes\']:\n            print(source, \'already audited in this registry; reusing its recorded bytes\', flush=True)\n            continue\n        dataset, request = request_for(source, target)\n        started = agora().isoformat()\n        with tempfile.TemporaryDirectory() as temp:\n            path = Path(temp) / (source + \'.grib\')\n            print(\'Requesting\', source, \'for target\', target, flush=True)\n            client.retrieve(dataset, request).download(str(path))\n            records.append(prepare(r, source, target, path, transport=dict(metodo=\'CDS API\',\n                dataset=dataset, request=request, started_utc=started, completed_utc=agora().isoformat())))\n    r.exportar_resumo(Path(registry) / \'resumo_registro.json\')\n    return records\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--registry\', required=True)\n    p.add_argument(\'--target\', required=True)\n    execute(**vars(p.parse_args()))\n', 'research/prospective/collect_sources.py': '"""Acquire public sources as separate versions without publishing or issuing a forecast."""\nfrom pathlib import Path\nfrom urllib.parse import urlparse\nimport argparse\nimport json\nimport requests\nfrom registry import Registro, agora, deslocar, exigir\nfrom data_contracts import indices_psl, inspecionar_netcdf\nFONTES = {\'nino34\': \'https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data\', \'nino12\': \'https://psl.noaa.gov/data/timeseries/month/data/nino12.long.anom.data\', \'tna\': \'https://psl.noaa.gov/data/correlation/tna.data\', \'tsa\': \'https://psl.noaa.gov/data/correlation/tsa.data\'}\nHOSTS = {\'psl.noaa.gov\', \'www.ncei.noaa.gov\', \'iridl.ldeo.columbia.edu\'}\n\ndef baixar(registro, fonte, url, perfil, esperado=None):\n    exigir(urlparse(url).scheme == \'https\' and urlparse(url).hostname in HOSTS, "Use the producer\'s authorized HTTPS endpoint.")\n    inicio = agora().isoformat()\n    try:\n        with requests.Session() as s:\n            s.headers[\'User-Agent\'] = \'WorCAP-rainfall-research/1.0 (prospective-data-capture)\'\n            with s.get(url, stream=True, timeout=(20, 90), allow_redirects=False) as r:\n                exigir(r.status_code == 200, f\'HTTP {r.status_code}\')\n                partes, tamanho = ([], 0)\n                for chunk in r.iter_content(128 * 1024):\n                    tamanho += len(chunk)\n                    exigir(tamanho <= 32 * 1024 * 1024, \'Response exceeds the 32 MiB limit.\')\n                    partes.append(chunk)\n                b = b\'\'.join(partes)\n                exigir(bool(b), \'Empty response.\')\n                try:\n                    meta = indices_psl(b) if perfil == \'psl\' else inspecionar_netcdf(b, perfil, esperado)\n                except Exception as erro_validacao:\n                    meta = dict(validado=False, meses=[], erro_inspecao=type(erro_validacao).__name__, pendencia=\'Response preserved in quarantine; do not use for inference.\')\n                transporte = dict(url=url, metodo=\'HTTPS GET\', iniciado_em_utc=inicio, resposta_em_utc=agora().isoformat(), status_http=r.status_code, cabecalhos={k: r.headers[k] for k in [\'Date\', \'Last-Modified\', \'ETag\', \'Content-Type\', \'Content-Length\'] if k in r.headers}, last_modified_nao_e_primeira_publicacao=True)\n        e = registro.recibo(fonte, b, meta, transporte)\n        print(f\'{fonte}: received; validated={meta[\'validado\']}; {len(b):,} bytes; id={e[\'id\'][:12]}\', flush=True)\n        return e\n    except Exception as erro:\n        registro.adicionar(\'coleta_falhou\', dict(fonte=fonte, url=url, iniciado_em_utc=inicio, erro_tipo=type(erro).__name__, objetos=[]))\n        print(f\'{fonte}: acquisition rejected ({type(erro).__name__}). No substitution with data from a different month.\', flush=True)\n        return None\n\ndef executar(pasta, plano_path, alvo, incluir_cfsv2=False):\n    r = Registro(pasta)\n    plano = json.loads(Path(plano_path).read_text(encoding=\'utf-8\'))\n    r.iniciar(plano)\n    r.prontidao(alvo)\n    saidas = [baixar(r, n, u, \'psl\') for n, u in FONTES.items()]\n    t = deslocar(alvo, -2).replace(\'-\', \'\')\n    u = f\'https://www.ncei.noaa.gov/pub/data/cmb/ersst/v5/netcdf/ersst.v5.{t}.nc\'\n    saidas.append(baixar(r, \'ersstv5\', u, \'ersstv5\', deslocar(alvo, -2)))\n    if incluir_cfsv2:\n        origem = deslocar(alvo, -1)\n        ano, m = map(int, origem.split(\'-\'))\n        mon = (\'Jan\', \'Feb\', \'Mar\', \'Apr\', \'May\', \'Jun\', \'Jul\', \'Aug\', \'Sep\', \'Oct\', \'Nov\', \'Dec\')[m - 1]\n        data = f\'(0000%201%20{mon}%20{ano})\'\n        raiz = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\n        u = raiz + f\'S/{data}/{data}/RANGE/L/1.5/VALUES/Y/-61/16/RANGE/X/269/336/RANGE/data.nc\'\n        saidas.append(baixar(r, \'cfsv2\', u, \'cfsv2_bruto\', alvo))\n    r.exportar_resumo(Path(pasta) / \'resumo_registro.json\')\n    estado = r.prontidao(alvo)\n    (Path(pasta) / \'prontidao.json\').write_text(json.dumps(estado, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    print(json.dumps(estado, indent=2, ensure_ascii=False), flush=True)\n    return estado\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--registro\', default=\'outputs/prospective\')\n    p.add_argument(\'--plano\', default=str(Path(__file__).with_name(\'plan.json\')))\n    p.add_argument(\'--alvo\', required=True)\n    p.add_argument(\'--cfsv2\', action=\'store_true\')\n    a = p.parse_args()\n    executar(a.registro, a.plano, a.alvo, a.cfsv2)\n', 'research/prospective/data_contracts.py': '"""Inspect acquired bytes and validate forecast field contracts."""\nfrom pathlib import Path\nimport tempfile\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom registry import exigir, ContratoError, mes\nLAT = np.arange(-60, 15.001, 0.25)\nLON = np.arange(-90, -24.999, 0.25)\n\ndef indices_psl(dados):\n    linhas = dados.decode(\'utf-8-sig\').strip().splitlines()\n    exigir(len(linhas) > 3 and len(linhas[0].split()) == 2, \'Invalid PSL header.\')\n    a, b = [int(x) for x in linhas[0].split()]\n    exigir(1800 <= a <= b <= 2200, \'Invalid declared years.\')\n    exigir(len(linhas) > b - a + 2, \'Incomplete PSL series.\')\n    matriz = []\n    for ano, linha in zip(range(a, b + 1), linhas[1:b - a + 2]):\n        v = linha.split()\n        exigir(len(v) == 13 and int(v[0]) == ano, \'Invalid PSL years or months.\')\n        matriz.extend(((f\'{ano:04d}-{m:02d}\', float(x)) for m, x in enumerate(v[1:], 1)))\n    sentinela = float(linhas[b - a + 2].strip())\n    exigir(sentinela < -90, \'Unexpected missing-value marker.\')\n    pares = [(t, v) for t, v in matriz if v != sentinela]\n    exigir(pares and all((np.isfinite(v) and abs(v) < 20 for _, v in pares)), \'Index outside the expected range.\')\n    return dict(validado=True, formato=\'monthly NOAA PSL\', unidades=\'degC\', meses=[t for t, _ in pares], ultimo_mes=pares[-1][0], ausentes=len(matriz) - len(pares), rodape=\'\\n\'.join(linhas[b - a + 3:]))\n\ndef inspecionar_netcdf(dados, perfil, esperado=None):\n    exigir(dados[:3] == b\'CDF\' or dados[:8] == b\'\\x89HDF\\r\\n\\x1a\\n\', \'Response is not NetCDF.\')\n    with tempfile.TemporaryDirectory() as td:\n        path = Path(td) / \'entrada.nc\'\n        path.write_bytes(dados)\n        with xr.open_dataset(path, decode_times=perfil != \'cfsv2_bruto\') as ds:\n            if perfil == \'ersstv5\':\n                exigir(ds.attrs.get(\'product_version\') == \'Version v5\', \'Different ERSST version.\')\n                exigir(\'sst\' in ds and ds.sst.attrs.get(\'units\') == \'degree_C\', \'Different SST variable or units.\')\n                ts = pd.DatetimeIndex(ds.time.values)\n                meses = ts.strftime(\'%Y-%m\').tolist()\n                exigir(len(meses) == 1 and meses[0] == esperado, \'SST month does not match the request.\')\n                exigir(ds.sizes.get(\'lat\') == 89 and ds.sizes.get(\'lon\') == 180, \'Unexpected ERSST grid.\')\n                exigir(np.array_equal(ds.lat, np.arange(-88, 90, 2)) and np.array_equal(ds.lon, np.arange(0, 360, 2)), \'Unexpected ERSST coordinates.\')\n                vals = ds.sst.values\n                f = vals[np.isfinite(vals)]\n                exigir(f.size > 1000 and (not np.isinf(vals).any()) and (f.min() >= -1.801) and (f.max() <= 45), \'Invalid SST field.\')\n                return dict(validado=True, meses=meses, versao=\'ERSSTv5\', unidades=\'degree_C\', dimensoes=dict(ds.sizes), minimo=float(f.min()), maximo=float(f.max()))\n            exigir(perfil == \'cfsv2_bruto\', \'Unknown inspection profile.\')\n            return dict(validado=False, meses=[], dimensoes=dict(ds.sizes), variaveis=list(ds.data_vars), pendencia=\'Audit real initialization dates, members, lead 1.5, units and grid before inference.\')\n\ndef validar_previsao(path, alvo):\n    with xr.open_dataset(path) as ds:\n        exigir(set(ds.data_vars) == {\'precipitacao\', \'climatologia\'}, \'Both forecast and control climatology are required.\')\n        exigir(set(ds.dims) == {\'time\', \'lat\', \'lon\'} and ds.sizes[\'time\'] == 1, \'Invalid dimensions.\')\n        exigir(np.array_equal(ds.lat, LAT) and np.array_equal(ds.lon, LON), \'Forecast grid differs from the reference.\')\n        t = pd.DatetimeIndex(ds.time.values)\n        exigir(t.equals(pd.DatetimeIndex([mes(alvo).replace(tzinfo=None)])), \'Incorrect forecast target.\')\n        for n in ds.data_vars:\n            a = ds[n]\n            exigir(a.dims == (\'time\', \'lat\', \'lon\') and a.attrs.get(\'units\') == \'mm/day\', \'Invalid output dimension order or units.\')\n            exigir(np.isfinite(a.values).all() and (a.values >= 0).all(), \'Prediction contains missing or negative values.\')\n        return dict(pontos=int(ds.precipitacao.size), minimo=float(ds.precipitacao.min()), maximo=float(ds.precipitacao.max()), unidades=\'mm/day\')\n\ndef metricas(previsao, verdade):\n    """Require exact alignment; do not mask observation, grid or unit errors."""\n    with xr.open_dataset(previsao) as a, xr.open_dataset(verdade) as b:\n        exigir(set(b.data_vars) == {\'tp\'}, \'Observation must contain only tp.\')\n        exigir(b.tp.attrs.get(\'units\') == \'mm/day\' and b.tp.dims == (\'time\', \'lat\', \'lon\'), \'Invalid observation contract.\')\n        exigir(np.isfinite(b.tp.values).all() and (b.tp.values >= 0).all(), \'Incomplete or invalid observation.\')\n        linhas = []\n        for nome in (\'precipitacao\', \'climatologia\'):\n            p, y = xr.align(a[nome], b.tp, join=\'exact\')\n            exigir(p.shape == y.shape, \'Observation does not cover the complete prediction.\')\n            erro = np.asarray(p.values, dtype=np.float64) - np.asarray(y.values, dtype=np.float64)\n            area = np.broadcast_to(np.cos(np.deg2rad(y.lat.values))[None, :, None], erro.shape)\n            for regiao, mask in [(\'dominio\', np.ones(len(y.lat), dtype=bool)), (\'sul_35\', y.lat.values < -35), (\'35_a_15\', (y.lat.values >= -35) & (y.lat.values < -15)), (\'norte_15\', y.lat.values >= -15)]:\n                e, w = (erro[:, mask, :], area[:, mask, :])\n                exigir(e.size > 0, \'Empty region.\')\n                sse, n = (float((e * e).sum()), int(e.size))\n                linhas.append(dict(modelo=nome, regiao=regiao, n=n, sse=sse, soma_erro=float(e.sum()), soma_erro_absoluto=float(np.abs(e).sum()), rmse=float(np.sqrt(sse / n)), mae=float(np.abs(e).mean()), vies=float(e.mean()), sse_area=float((e * e * w).sum()), peso_area=float(w.sum()), rmse_area=float(np.sqrt((e * e * w).sum() / w.sum()))))\n        return linhas\n\ndef conferir_observacao_bruta(bruto, normalizado, alvo):\n    """Validate ERA5 monthly conversion; reject unknown format or expver."""\n    with xr.open_dataset(bruto) as ds, xr.open_dataset(normalizado) as n:\n        exigir(\'tp\' in ds and set(n.data_vars) == {\'tp\'}, \'Missing rainfall variable tp.\')\n        a = ds.tp\n        exigir(a.attrs.get(\'units\') in {\'m\', \'m/day\', \'m day**-1\'}, \'Unexpected raw ERA5 units.\')\n        stream = a.attrs.get(\'GRIB_stream\', ds.attrs.get(\'GRIB_stream\'))\n        exigir(stream == \'moda\', \'Require monthly means of daily means, stream moda; do not infer conversion for a different product.\')\n        if \'expver\' in a.dims:\n            exigir(a.sizes[\'expver\'] == 1 and int(a.expver.values[0]) == 1, \'ERA5T or mixed versions are not accepted.\')\n            a = a.isel(expver=0, drop=True)\n        elif \'expver\' in ds:\n            exigir(np.asarray(ds.expver).size == 1 and int(np.asarray(ds.expver).ravel()[0]) == 1, \'Nonfinal expver.\')\n        else:\n            ver = a.attrs.get(\'GRIB_experimentVersionNumber\', ds.attrs.get(\'GRIB_experimentVersionNumber\'))\n            exigir(str(ver) in {\'1\', \'0001\'}, \'Raw file contains no evidence of final expver.\')\n        a = a.rename({x: y for x, y in [(\'valid_time\', \'time\'), (\'latitude\', \'lat\'), (\'longitude\', \'lon\')] if x in a.dims})\n        exigir(set(a.dims) == {\'time\', \'lat\', \'lon\'} and a.sizes[\'time\'] == 1, \'Unexpected raw target dimensions.\')\n        ts = pd.DatetimeIndex(a.time.values)\n        exigir(ts[0].day == 1 and ts.strftime(\'%Y-%m\').tolist() == [alvo], \'Raw month differs from the target.\')\n        a = a.assign_coords(time=[np.datetime64(alvo + \'-01\')], lon=(a.lon + 180) % 360 - 180).sortby(\'lat\').sortby(\'lon\')\n        a = a.transpose(\'time\', \'lat\', \'lon\')\n        a, b = xr.align(a, n.tp, join=\'exact\')\n        exigir(a.shape == b.shape and np.allclose(a.values * 1000, b.values, rtol=1e-06, atol=1e-06), \'Target conversion does not match raw ERA5 multiplied by 1000.\')\n        return dict(stream=\'moda\', expver=1, conversao=\'m * 1000 -> mm/day\', mes_alvo=alvo)\n', 'research/prospective/evaluate.py': '"""Verify frozen predictions without selecting models."""\nfrom pathlib import Path\nimport json\nimport tempfile\nfrom data_contracts import validar_previsao, metricas, conferir_observacao_bruta\nfrom registry import Registro, agora, canonico, deslocar, exigir, mes\n\ndef avaliar(registro, alvo, observacao, evidencia):\n    r = registro if isinstance(registro, Registro) else Registro(registro)\n    exigir(agora() >= mes(deslocar(alvo, 4)), \'Primary verification requires T+4 and a final product.\')\n    p = r.protocolo()\n    previsoes = [e for e in r.eventos() if e[\'tipo\'] == \'previsao_congelada\' and e[\'dados\'][\'mes_alvo\'] == alvo]\n    exigir(len(previsoes) == 1, \'No previously frozen prediction for this month.\')\n    previsao = previsoes[0]\n    exigir(evidencia.get(\'produto\') == \'ERA5 monthly_averaged_reanalysis\' and evidencia.get(\'expver\') == 1 and (evidencia.get(\'mes_alvo\') == alvo), \'Final-product metadata and the correct month are required.\')\n    exigir(evidencia.get(\'url\', \'\').startswith(\'https://cds.climate.copernicus.eu/\'), \'Unexpected verification source.\')\n    raw = Path(evidencia[\'arquivo_bruto\']).read_bytes()\n    auditoria = conferir_observacao_bruta(evidencia[\'arquivo_bruto\'], observacao, alvo)\n    with r.transacao(), tempfile.TemporaryDirectory() as td:\n        fp = Path(td) / \'previsao.nc\'\n        fp.write_bytes(r.ler_objeto(previsao[\'dados\'][\'previsao\']))\n        validar_previsao(fp, alvo)\n        linhas = metricas(fp, observacao)\n        anteriores = [e for e in r.eventos() if e[\'tipo\'] == \'verificacao\' and e[\'dados\'][\'mes_alvo\'] == alvo]\n        ob = r.guardar(Path(observacao).read_bytes())\n        bruto = r.guardar(raw)\n        ev = {k: v for k, v in evidencia.items() if k != \'arquivo_bruto\'}\n        ev[\'conferencia_bytes\'] = auditoria\n        return r._adicionar(\'verificacao\', dict(mes_alvo=alvo, protocolo=p[\'id\'], previsao=previsao[\'id\'], revisao=len(anteriores), primaria=not anteriores, metricas=linhas, evidencia=ev, observacao=ob, bruto=bruto, objetos=[ob, bruto]))\n\ndef painel(registro):\n    r = registro if isinstance(registro, Registro) else Registro(registro)\n    p = r.protocolo()[\'dados\'][\'plano\']\n    ev = r.eventos(conferir_objetos=True)\n    atual = p[\'primeiro_mes\']\n    rows = []\n    while atual <= p[\'ultimo_mes\']:\n        preds = [e for e in ev if e[\'tipo\'] == \'previsao_congelada\' and e[\'dados\'][\'mes_alvo\'] == atual]\n        obs = [e for e in ev if e[\'tipo\'] == \'verificacao\' and e[\'dados\'][\'mes_alvo\'] == atual and e[\'dados\'][\'primaria\']]\n        estado = \'avaliado\' if obs else \'emitido_aguardando_observacao\' if preds else \'nao_emitido_prazo_encerrado\' if agora() >= mes(atual) else \'aguardando_emissao\'\n        rows.append(dict(mes_alvo=atual, estado=estado, previsao=preds[0][\'id\'] if preds else None, verificacao=obs[0][\'id\'] if obs else None))\n        atual = deslocar(atual, 1)\n    resumo = dict(meses_planejados=len(rows), meses_emitidos=sum((x[\'previsao\'] is not None for x in rows)), meses_avaliados=sum((x[\'verificacao\'] is not None for x in rows)), meses=rows)\n    sums = {}\n    for e in ev:\n        if e[\'tipo\'] != \'verificacao\' or not e[\'dados\'][\'primaria\']:\n            continue\n        for a in e[\'dados\'][\'metricas\']:\n            chave = a[\'modelo\'] + \'/\' + a[\'regiao\']\n            z = sums.setdefault(chave, {k: 0 for k in [\'n\', \'sse\', \'soma_erro\', \'soma_erro_absoluto\', \'sse_area\', \'peso_area\']})\n            for k in z:\n                z[k] += a[k]\n    for z in sums.values():\n        z.update(rmse=(z[\'sse\'] / z[\'n\']) ** 0.5, mae=z[\'soma_erro_absoluto\'] / z[\'n\'], vies=z[\'soma_erro\'] / z[\'n\'], rmse_area=(z[\'sse_area\'] / z[\'peso_area\']) ** 0.5)\n    resumo[\'metricas_primarias\'] = sums\n    return resumo\n', 'research/prospective/fit_baseline.py': '"""Fit the fixed 1993–2022 lagged hybrid and register its complete model package."""\nfrom pathlib import Path\nimport argparse\nimport json\nimport shutil\nimport sys\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path.insert(0, str(ROOT))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom research.common.inputs import Inputs, LIBRARY, require, write_json, sha256\nfrom research.common.baseline import fit\nfrom registry import Registro\n\n\ndef execute(output, registry, official=None, seas5=None, cfsv2=None):\n    r = Registro(registry)\n    plan = json.loads(Path(__file__).with_name(\'plan.json\').read_text())\n    event = r.iniciar(plan)\n    require(not any(e[\'tipo\'] == \'modelo_congelado\' for e in r.eventos()), \'The existing model is immutable; no refit is allowed in this series.\')\n    output = Path(output)\n    require(not output.exists(), \'Preserve existing model files; choose a new package directory.\')\n    output.mkdir(parents=True)\n    data = Inputs(official, seas5, cfsv2, output=output, final=True)\n    state = fit(data, \'2022-12-01\', output)\n    require(state[\'audit\'][\'training_start\'] == \'1993-01-01\' and state[\'audit\'][\'training_months\'] == 360, \'Training differs from the frozen plan.\')\n    shutil.copyfile(LIBRARY, output / \'biblioteca.py\')\n    shutil.copyfile(Path(__file__).with_name(\'infer_baseline.py\'), output / \'inferir.py\')\n    write_json(output / \'ambiente.json\', data.audit[\'versions\'])\n    audit = dict(state[\'audit\'], treino_inicio=\'1993-01\', treino_fim=\'2022-12\', protocolo_id=event[\'id\'],\n        lags=dict(atmosphere=4, indices=3, seasonal=1, rainfall_training=4),\n        inference_sha256=sha256(output / \'inferir.py\'), library_sha256=sha256(output / \'biblioteca.py\'),\n        fitting_code_sha256=sha256(__file__), model_package_is_competition_model=False,\n        historical_vintages_verified=False)\n    write_json(output / \'auditoria_treino.json\', audit)\n    result = r.modelo({name: output / name for name in plan[\'arquivos_modelo\']}, audit)\n    r.exportar_resumo(Path(registry) / \'resumo_registro.json\')\n    print(\'Lagged hybrid package frozen:\', result[\'id\'], flush=True)\n    return result\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--output\', required=True)\n    parser.add_argument(\'--registry\', required=True)\n    for name in [\'official\', \'seas5\', \'cfsv2\']:\n        parser.add_argument(\'--\' + name)\n    execute(**vars(parser.parse_args()))\n', 'research/prospective/infer_baseline.py': '"""Inference from frozen model files and exact selected source snapshots only."""\nfrom pathlib import Path\nimport importlib.metadata\nimport importlib.util\nimport json\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise ValueError(message)\n\n\ndef psl_value(path, month):\n    lines = Path(path).read_text(encoding=\'utf-8\').strip().splitlines()\n    first, last = map(int, lines[0].split())\n    require(first <= month.year <= last, \'Index source does not cover the requested year.\')\n    row = lines[1 + month.year - first].split()\n    require(len(row) == 13 and int(row[0]) == month.year, \'Malformed NOAA row.\')\n    value = float(row[month.month])\n    require(np.isfinite(value) and abs(value) < 20, \'Unavailable or invalid NOAA value.\')\n    return value\n\n\ndef predict(package, inputs, target):\n    package = Path(package)\n    versions = json.loads((package / \'ambiente.json\').read_text())\n    require(all(importlib.metadata.version(n) == v for n, v in versions.items()), \'Inference environment differs from the fitted baseline.\')\n    spec = importlib.util.spec_from_file_location(\'frozen_baseline_library\', package / \'biblioteca.py\')\n    m = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(m)\n    target = pd.Timestamp(target + \'-01\')\n    source_month = target - pd.DateOffset(months=4)\n    with xr.open_dataset(package / \'clima_chuva.nc\') as d:\n        climate = next(iter(d.data_vars.values())).load()\n    fields = {}\n    for name in [\'era5_sl\', \'era5_pl\']:\n        with xr.open_dataset(inputs[name]) as ds:\n            require(ds.attrs.get(\'schema\') == \'worcap_era5_prospective_v1\' and ds.attrs.get(\'expver\') == 1,\n                    \'Require audited final ERA5, never an unaudited or ERA5T input.\')\n            require(pd.DatetimeIndex(ds.time.values).equals(pd.DatetimeIndex([source_month])), \'Wrong atmospheric source month.\')\n            for key in ds.data_vars:\n                require(key not in fields, \'Duplicate atmospheric field.\')\n                m.conferir_campo(ds[key], climate, [source_month])\n                fields[key] = ds[key].load().astype(\'float32\')\n    require(set(fields) == set(m.VARIAVEIS), \'Incomplete atmospheric input.\')\n    index_month = target - pd.DateOffset(months=3)\n    m.INDICES_OC = pd.DataFrame([[psl_value(inputs[n], index_month) for n in m.INDICES_NOMES]], index=[index_month], columns=m.INDICES_NOMES)\n    seasonal = []\n    for name, variable, schema in [(\'seas5\', \'seas5_tp_media\', \'worcap_seas5_prospective_v1\'),\n                                    (\'cfsv2\', \'cfsv2_tp_media\', \'nimbus_cfsv2_prospectivo_v1\')]:\n        with xr.open_dataset(inputs[name]) as ds:\n            require(ds.attrs.get(\'schema\') == schema and ds[variable].attrs.get(\'units\') == \'mm/day\', \'Unaudited seasonal artifact.\')\n            require(pd.DatetimeIndex(ds.time.values).equals(pd.DatetimeIndex([target])), \'Wrong seasonal target.\')\n            require(pd.DatetimeIndex(ds.time_origem.values).equals(pd.DatetimeIndex([target - pd.DateOffset(months=1)])), \'Wrong seasonal initialization.\')\n            a = ds[variable].interp(lat=climate.lat, lon=climate.lon, method=\'linear\').astype(\'float32\').load()\n            for coord in [\'time_origem\', \'inicializacao_mais_recente\']:\n                if coord in ds:\n                    a = a.assign_coords({coord: (\'time\', ds[coord].values)})\n            require(np.isfinite(a.values).all() and (a.values >= 0).all(), \'Invalid seasonal interpolation.\')\n            seasonal.append(a)\n    seas, cfs = seasonal\n    with np.load(package / \'clima_atmosfera_indices.npz\', allow_pickle=False) as z:\n        atmosphere = {n: z[n].copy() for n in z.files}\n    with np.load(package / \'ridge.npz\', allow_pickle=False) as z:\n        ridge = {n: z[n].copy() for n in z.files}\n    ridge[\'corte\'] = str(ridge[\'corte\'].item())\n    climates = []\n    for name in [\'seas5\', \'cfsv2\']:\n        with xr.open_dataset(package / (\'clima_\' + name + \'.nc\')) as ds:\n            climates.append(next(iter(ds.data_vars.values())).load())\n    pieces = []\n    for name, scale, features in zip(m.NOMES_MODELOS, [.9, .875], [m.FEATURES_SEAS5, m.FEATURES_MULTISSISTEMA]):\n        model = m.lgb.Booster(model_file=str(package / (name + \'.txt\')))\n        require(model.feature_name() == features, \'Model feature order differs from the frozen library.\')\n        anom = m.prever_par(model, name, fields, seas, cfs, atmosphere, climate, *climates, [target])\n        pieces.append(m.reconstruir_chuva(climate, anom, scale).values.astype(\'float64\'))\n    prediction = .75 * (.5 * pieces[0] + .5 * pieces[1]) + .25 * m.mos_predict(ridge, seas, cfs, [target])\n    require(np.isfinite(prediction).all() and (prediction >= 0).all(), \'Invalid hybrid prediction.\')\n    return xr.Dataset({\n        \'precipitacao\': ((\'time\', \'lat\', \'lon\'), prediction, dict(units=\'mm/day\')),\n        \'climatologia\': ((\'time\', \'lat\', \'lon\'), climate.values[[target.month - 1]], dict(units=\'mm/day\'))},\n        coords=dict(time=[target], lat=climate.lat, lon=climate.lon))\n', 'research/prospective/issue_forecast.py': '"""Materialize verified registry objects, infer once and freeze before the deadline."""\nfrom pathlib import Path\nimport argparse\nimport tempfile\nfrom registry import Registro, exigir, digest\nfrom infer_baseline import predict\n\n\ndef execute(registry, target, output):\n    r = Registro(registry)\n    status = r.prontidao(target)\n    exigir(status[\'pronto\'], \'Forecast blocked: \' + \'; \'.join(status[\'faltantes\']))\n    model = r.obter(status[\'modelo\'], \'modelo_congelado\')\n    source = Path(__file__).with_name(\'infer_baseline.py\').read_bytes()\n    exigir(model[\'dados\'][\'arquivos\'][\'inferir.py\'][\'sha256\'] == digest(source), \'Frozen inference code differs; use its reviewed version.\')\n    output = Path(output)\n    exigir(not output.exists(), \'Never overwrite an existing forecast file.\')\n    with tempfile.TemporaryDirectory() as temp:\n        temp = Path(temp)\n        package = temp / \'model\'\n        package.mkdir()\n        for name, ref in model[\'dados\'][\'arquivos\'].items():\n            exigir(Path(name).name == name, \'Invalid model filename.\')\n            (package / name).write_bytes(r.ler_objeto(ref))\n        inputs = {}\n        for source_name, ident in status[\'fontes\'].items():\n            event = r.obter(ident, \'fonte_recebida\')\n            path = temp / (source_name + \'.input\')\n            path.write_bytes(r.ler_objeto(event[\'dados\'][\'objeto\']))\n            inputs[source_name] = path\n        result = predict(package, inputs, target)\n        output.parent.mkdir(parents=True, exist_ok=True)\n        result.to_netcdf(temp / \'forecast.nc\', engine=\'h5netcdf\')\n        content = (temp / \'forecast.nc\').read_bytes()\n        with output.open(\'xb\') as f:\n            f.write(content)\n    inference = {k: status[k] for k in [\'modelo\', \'fontes\', \'protocolo\']}\n    inference.update(previsao_sha256=digest(content), inference_code_sha256=digest(source))\n    event = r.congelar(target, output, inference)\n    r.exportar_resumo(Path(registry) / \'resumo_registro.json\')\n    print(\'Forecast frozen:\', event[\'id\'])\n    return event\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--registry\', required=True)\n    p.add_argument(\'--target\', required=True)\n    p.add_argument(\'--output\', required=True)\n    execute(**vars(p.parse_args()))\n', 'research/prospective/plan.json': '{\n  "id": "nimbus_acompanhamento_controle_defasado_v1",\n  "equipe": "Nimbus PA",\n  "primeiro_mes": "2026-10",\n  "ultimo_mes": "2027-09",\n  "prazo": "antes de 00:00 UTC do primeiro dia do mes alvo; sem emissao retroativa",\n  "treino_inicio": "1993-01",\n  "treino_fim": "2022-12",\n  "politica_modelo": "um pacote final do controle com defasagens, reajustado na janela fixa; sem SST/PCA; sem atualizar pesos ou treino durante estes doze meses",\n  "modelo": {\n    "arvores": 300,\n    "folhas": 31,\n    "taxa": 0.05,\n    "semente": 42,\n    "pontos_por_mes": 5000,\n    "ridge_alpha": 0.1,\n    "pesos": [0.375, 0.375, 0.25],\n    "escalas_residuais": [0.9, 0.875],\n    "maximo_meses_treino": 360\n  },\n  "arquivos_modelo": [\n    "arvores_seas5.txt", "arvores_multissistema.txt", "ridge.npz",\n    "clima_chuva.nc", "clima_atmosfera_indices.npz", "clima_seas5.nc",\n    "clima_cfsv2.nc", "biblioteca.py", "inferir.py", "ambiente.json",\n    "auditoria_treino.json"\n  ],\n  "fontes": {\n    "era5_sl": {"lag": 4, "obrigatoria": true, "produto": "ERA5 final mensal; tcc,t2m,sp"},\n    "era5_pl": {"lag": 4, "obrigatoria": true, "produto": "ERA5 final mensal 850 hPa; z,r,q,t,u,v"},\n    "nino34": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL nino34.long.anom.data"},\n    "nino12": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL nino12.long.anom.data"},\n    "tna": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL tna.data"},\n    "tsa": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL tsa.data"},\n    "seas5": {"lag": 1, "obrigatoria": true, "produto": "SEAS5 sistema 51, inicializacao T-1, forecastMonth 2; datas e membros auditados"},\n    "cfsv2": {"lag": 1, "obrigatoria": true, "produto": "CFSv2 IRI PENTAD_SAMPLES_FULL; emissao nominal T-1 e lead 1.5; membros auditados"},\n    "ersstv5": {"lag": 2, "obrigatoria": false, "produto": "ERSSTv5; somente arquivo de chegada, nao entra na referencia"}\n  },\n  "falha_de_fonte": "bloquear emissao; nao preencher, trocar mes, versao ou modelo silenciosamente",\n  "selecao_versao": "ultima versao recebida e validada antes da emissao; registrar IDs exatos",\n  "avaliacao": {\n    "alvo": "ERA5 mensal final total_precipitation em mm/day; nao representa pluviometro independente",\n    "primeira_verificacao": "primeira versao final conferida recebida a partir do primeiro dia de T+4",\n    "metrica_primaria": "RMSE global sqrt(soma SSE / soma n)",\n    "secundarias": ["MAE", "vies", "RMSE por area", "tres faixas de latitude", "meses emitidos/previstos"],\n    "comparacao": "climatologia do mesmo treino congelada junto a cada previsao",\n    "revisoes": "adicionar outra verificacao rotulada; nunca sobrescrever a primeira",\n    "faltantes": "reportar cobertura; nao omitir meses sem emissao dos relatorios"\n  },\n  "agendamento_automatico": false,\n  "carimbo_independente": false,\n  "limitacao": "Registro local com relogio UTC do computador e hashes. Sem assinatura/ancoragem externa; nao prova contra reescrita integral ou exclusao da ponta. Qualidade e calendario de cada adaptador devem ser auditados."\n}\n', 'research/prospective/prepare_cds.py': '"""Validate final ERA5 or complete SEAS5 GRIBs and preserve their receipt lineage."""\nfrom pathlib import Path\nimport argparse\nimport tempfile\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom registry import Registro, exigir, deslocar\nfrom data_contracts import LAT, LON\n\nFIELDS = {\n    \'era5_sl\': {164: (\'cloud_cover\', {\'(0 - 1)\', \'(0-1)\', \'1\'}), 167: (\'t2\', {\'K\'}), 134: (\'surface_pressure\', {\'Pa\'})},\n    \'era5_pl\': {129: (\'geopotential_850\', {\'m**2 s**-2\'}), 157: (\'rel_hum_850\', {\'%\'}),\n                133: (\'shum_850\', {\'kg kg**-1\'}), 130: (\'temperature_850\', {\'K\'}),\n                131: (\'u_850\', {\'m s**-1\'}), 132: (\'v_850\', {\'m s**-1\'})}}\n\n\ndef regular_grid(latitudes, longitudes, values):\n    latitudes = np.asarray(latitudes, dtype=\'float64\')\n    longitudes = (np.asarray(longitudes, dtype=\'float64\') + 180) % 360 - 180\n    values = np.asarray(values, dtype=\'float64\')\n    lat, lon = np.unique(latitudes), np.unique(longitudes)\n    exigir(values.size == len(lat) * len(lon) and np.isfinite(values).all(), \'Incomplete regular grid.\')\n    index = np.searchsorted(lat, latitudes) * len(lon) + np.searchsorted(lon, longitudes)\n    exigir(len(np.unique(index)) == values.size, \'Duplicate geographic points.\')\n    out = np.empty(values.size)\n    out[index] = values\n    return lat, lon, out.reshape(len(lat), len(lon))\n\n\ndef convert(path, source, target):\n    import eccodes as ec\n    exigir(source in {*FIELDS, \'seas5\'}, \'Unsupported CDS source.\')\n    origin = pd.Timestamp(deslocar(target, -1 if source == \'seas5\' else -4) + \'-01\')\n    target_date = pd.Timestamp(target + \'-01\')\n    arrays, audit, headers = {}, [], set()\n    coordinates = None\n    with Path(path).open(\'rb\') as f:\n        while (g := ec.codes_grib_new_from_file(f)) is not None:\n            try:\n                get = lambda key: ec.codes_get(g, key)\n                exigir(int(get(\'numberOfMissing\')) == 0 and int(get(\'dataDate\')) == int(origin.strftime(\'%Y%m%d\')) and int(get(\'dataTime\')) == 0,\n                        \'Missing pixels or wrong GRIB source month/time.\')\n                values = np.asarray(ec.codes_get_values(g), dtype=\'float64\')\n                if source == \'seas5\':\n                    exigir(str(get(\'system\')) == \'51\' and str(get(\'origin\')) == \'ecmf\', \'Wrong SEAS5 system or centre.\')\n                    exigir(get(\'type\') == \'fcmean\' and get(\'stream\') == \'msmm\' and int(get(\'paramId\')) == 172228, \'Wrong seasonal product.\')\n                    exigir(int(get(\'forecastMonth\')) == 2 and int(get(\'verifyingMonth\')) == int(target_date.strftime(\'%Y%m\')), \'Wrong forecast month.\')\n                    exigir(str(get(\'units\')).replace(\' \', \'\') in {\'ms**-1\', \'ms^-1\', \'ms-1\', \'m/s\'}, \'Unknown rainfall rate units.\')\n                    key = int(get(\'number\'))\n                    headers.add(int(get(\'numberOfForecastsInEnsemble\')))\n                    values *= 86400000.\n                    exigir(np.isfinite(values).all() and values.min() >= -.01, \'Invalid precipitation or a negative value beyond the fixed tolerance.\')\n                    negative = np.flatnonzero(values < 0)\n                    audit.append(dict(member=key, corrected=int(len(negative)), minimum_before=float(values.min()),\n                                      indices_grib=negative.tolist(), values_before_mm_day=values[negative].tolist()))\n                    values = np.maximum(values, 0)\n                    units = \'mm/day\'\n                else:\n                    exigir(str(get(\'class\')) == \'ea\' and str(get(\'expver\')) in {\'1\', \'0001\'} and get(\'stream\') == \'moda\', \'Require final ERA5 monthly means: class ea, expver 1, stream moda.\')\n                    param = int(get(\'paramId\'))\n                    exigir(param in FIELDS[source], \'Unexpected atmospheric parameter.\')\n                    key, allowed_units = FIELDS[source][param]\n                    units = get(\'units\')\n                    exigir(units in allowed_units, f\'Unexpected units for {key}: {units}\')\n                    if source == \'era5_pl\':\n                        exigir(get(\'typeOfLevel\') == \'isobaricInhPa\' and int(get(\'level\')) == 850, \'Pressure level must be 850 hPa.\')\n                    audit.append(dict(variable=key, paramId=param, units=units, expver=1))\n                exigir(key not in arrays, \'Duplicate variable or member.\')\n                lat, lon, field = regular_grid(ec.codes_get_array(g, \'latitudes\'), ec.codes_get_array(g, \'longitudes\'), values)\n                if coordinates is not None:\n                    exigir(np.array_equal(lat, coordinates[0]) and np.array_equal(lon, coordinates[1]), \'Inconsistent GRIB grids.\')\n                coordinates = lat, lon\n                arrays[key] = field\n            finally:\n                ec.codes_release(g)\n    exigir(bool(arrays), \'Empty GRIB file.\')\n    if source == \'seas5\':\n        exigir(set(arrays) == set(range(51)) and headers <= {0, 51}, \'The future series requires every SEAS5 member 0–50.\')\n        exigir(np.all(np.diff(lat) == 1) and np.all(np.diff(lon) == 1) and lat.min() <= -60 and lat.max() >= 15 and lon.min() <= -90 and lon.max() >= -25,\n                \'Seasonal grid must cover the target domain at one degree.\')\n        mean = np.stack([arrays[i] for i in range(51)]).mean(0, dtype=\'float64\').astype(\'float32\')\n        ds = xr.Dataset(dict(seas5_tp_media=((\'time\', \'lat\', \'lon\'), mean[None], dict(units=\'mm/day\')),\n            time_origem=(\'time\', [origin]), n_membros=(\'time\', [51])), coords=dict(time=[target_date], lat=lat, lon=lon),\n            attrs=dict(schema=\'worcap_seas5_prospective_v1\', system=\'51\', forecastMonth=2, correction_tolerance_mm_day=.01))\n    else:\n        exigir(set(arrays) == {value[0] for value in FIELDS[source].values()}, \'Incomplete atmospheric variables.\')\n        exigir(np.array_equal(lat, LAT) and np.array_equal(lon, LON), \'ERA5 must match the quarter-degree official grid exactly.\')\n        ds = xr.Dataset({key: ((\'time\', \'lat\', \'lon\'), a[None].astype(\'float32\'), dict(units=next(r[\'units\'] for r in audit if r[\'variable\'] == key))) for key, a in arrays.items()},\n            coords=dict(time=[origin], lat=lat, lon=lon), attrs=dict(schema=\'worcap_era5_prospective_v1\', expver=1, stream=\'moda\'))\n    return ds, dict(validado=True, meses=[origin.strftime(\'%Y-%m\')], meses_alvo=[target], auditoria=audit)\n\n\ndef prepare(registry, source, target, file, transport=None):\n    r = registry if isinstance(registry, Registro) else Registro(registry)\n    r.prontidao(target)\n    raw = Path(file).read_bytes()\n    receipt = r.recibo(source, raw, dict(validado=False, meses=[], formato=\'GRIB awaiting audit\'),\n                       transport or dict(metodo=\'local import\', filename=Path(file).name, previous_download_time_unknown=True))\n    with tempfile.TemporaryDirectory() as temp:\n        raw_file = Path(temp) / \'original.grib\'\n        raw_file.write_bytes(r.ler_objeto(receipt[\'dados\'][\'objeto\']))\n        try:\n            ds, metadata = convert(raw_file, source, target)\n        except Exception as error:\n            r.adicionar(\'auditoria_falhou\', dict(fonte=source, pais=[receipt[\'id\']], erro_tipo=type(error).__name__, objetos=[]))\n            raise\n        result = Path(temp) / \'normalized.nc\'\n        ds.to_netcdf(result, engine=\'h5netcdf\')\n        return r.derivado(source, result.read_bytes(), metadata, [receipt[\'id\']], Path(__file__))\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    for name in [\'registry\', \'source\', \'target\', \'file\']:\n        p.add_argument(\'--\' + name, required=True)\n    print(prepare(**vars(p.parse_args()))[\'id\'])\n', 'research/prospective/prepare_cfsv2.py': '"""Acquire and audit next-month CFSv2 predictions while preserving raw inputs."""\nfrom pathlib import Path\nimport tempfile\nimport pandas as pd\nfrom registry import Registro, deslocar, exigir\nfrom collect_sources import baixar\nimport cfsv2_adapter as a\n\ndef preparar(registro, alvo, atualizar=False):\n    r = registro if isinstance(registro, Registro) else Registro(registro)\n    origem = deslocar(alvo, -1)\n    ano, mm = map(int, origem.split(\'-\'))\n    m = (\'Jan\', \'Feb\', \'Mar\', \'Apr\', \'May\', \'Jun\', \'Jul\', \'Aug\', \'Sep\', \'Oct\', \'Nov\', \'Dec\')[mm - 1]\n    d = f\'(0000%201%20{m}%20{ano})\'\n    intervalo = f\'S/{d}/{d}/RANGE/\'\n    urls = [a.FONTE + intervalo + \'L/1.5/VALUES/Y/-61/16/RANGE/X/269/336/RANGE/data.nc\', \'https://iridl.ldeo.columbia.edu/SOURCES/.NOAA/.NCEP/.EMC/.CFSv2/.REALTIME_ENSEMBLE/.FLXF/sampleS/\' + intervalo + \'data.nc\']\n    recibos = []\n    for u in urls:\n        antigos = [e for e in r.eventos() if e[\'tipo\'] == \'fonte_recebida\' and e[\'dados\'][\'fonte\'] == \'cfsv2\' and (e[\'dados\'][\'transporte\'].get(\'url\') == u)]\n        e = antigos[-1] if antigos and (not atualizar) else baixar(r, \'cfsv2\', u, \'cfsv2_bruto\', alvo)\n        exigir(e is not None, \'CFSv2 was not acquired; keep the forecast blocked.\')\n        recibos.append(e)\n    with tempfile.TemporaryDirectory() as td:\n        paths = []\n        for i, e in enumerate(recibos):\n            p = Path(td) / f\'original_{i}.nc\'\n            p.write_bytes(r.ler_objeto(e[\'dados\'][\'objeto\']))\n            paths.append(p)\n        ts = pd.DatetimeIndex([origem + \'-01\'])\n        tabela = a.tabela_inicializacoes(paths[1], ts)\n        try:\n            ds, auditoria = a.processar_bloco(paths[0], ts, tabela)\n        except ValueError as erro:\n            campos = a.carregar_membros(paths[0], ts)\n            import numpy as np\n            nvalidos = np.isfinite(campos[0]).sum(axis=(-1, -2))\n            info = dict(fonte=\'cfsv2\', mes_origem=origem, mes_alvo=alvo, motivo=str(erro), pais=[e[\'id\'] for e in recibos], pixels_finitos_por_membro={str(i + 1): int(n) for i, n in enumerate(nvalidos)}, membros_esperados=(np.flatnonzero(tabela[ts[0]][\'mascara\']) + 1).tolist(), objetos=[], acao=\'blocked; no exception or partial ensemble mean was applied\')\n            r.adicionar(\'auditoria_falhou\', info)\n            print(\'CFSv2 blocked:\', info[\'motivo\'], flush=True)\n            return None\n        exigir(pd.DatetimeIndex(ds.time.values).strftime(\'%Y-%m\').tolist() == [alvo], \'Incorrect CFSv2 target.\')\n        p = Path(td) / \'cfsv2.nc\'\n        ds.to_netcdf(p)\n        e = r.derivado(\'cfsv2\', p.read_bytes(), dict(validado=True, meses=[origem], meses_alvo=[alvo], unidades=\'mm/day\', lead=1.5, auditoria=auditoria), [e[\'id\'] for e in recibos], Path(a.__file__))\n        print(\'CFSv2 audited:\', origem, \'→\', alvo, \';\', auditoria[0][\'membros\'], \'members; id\', e[\'id\'][:12], flush=True)\n        return e\nif __name__ == \'__main__\':\n    import argparse\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--registro\', default=\'outputs/prospective\')\n    p.add_argument(\'--alvo\', required=True)\n    p.add_argument(\'--atualizar\', action=\'store_true\', help=\'Query the source again and preserve the new input version.\')\n    args = p.parse_args()\n    preparar(args.registro, args.alvo, args.atualizar)\n', 'research/prospective/prepare_cfsv2_recipe.py': '"""Re-evaluate IRI\'s published CFSv2 recipe when its named product is incomplete.\n\nThis keeps the original provider, member selection, regridAverage operation and\nunit conversion. It never uses CCSR values or joins a partial ensemble to another\nproduct. All members are acquired together through the explicit IRI expression.\nEvery available cached member must match exactly before a derived input is valid.\n"""\nfrom pathlib import Path\nimport tempfile\n\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nif __package__:\n    from . import cfsv2_adapter as adapter\nelse:\n    import cfsv2_adapter as adapter\n\n\nCATALOG_URL = (\'https://raw.githubusercontent.com/iridl/dlentries/\'\n               \'7b8bcdd8be1c2beec01e441bcc09e31d5854958a/\'\n               \'entries/Models/NMME/NCEP-CFSv2/FORECAST/PENTAD_SAMPLES/MONTHLY/index.tex\')\n\n\ndef recipe_url(origin):\n    t = pd.Timestamp(origin)\n    adapter.exigir(t == t.to_period(\'M\').start_time and t >= pd.Timestamp(\'2011-04-01\'),\n                   \'Explicit recipe is for operational forecasts from April 2011 onward.\')\n    month = (\'Jan\', \'Feb\', \'Mar\', \'Apr\', \'May\', \'Jun\', \'Jul\', \'Aug\', \'Sep\', \'Oct\', \'Nov\', \'Dec\')[t.month - 1]\n    s = f\'(0000%201%20{month}%20{t.year})\'\n    # Follow the producer\'s expression in order; the first field supplies the\n    # destination X/Y grid. Only S and L are restricted before the same regrid.\n    return (\'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.MONTHLY/.prec/\'\n            \'L/0.5/VALUE/M/1.0/VALUE/S/last/VALUE/\'\n            \'SOURCES/.NOAA/.NCEP/.EMC/.CFSv2/.REALTIME_ENSEMBLE/.FLXF/.surface/.PRATE/\'\n            f\'S/{s}/{s}/RANGE/L/1.5/VALUES/%5BX/Y%5D/regridAverage/nip/\'\n            \'c:/0.001/(m3%20kg-1)/:c/mul/c:/1000/(mm%20m-1)/:c/mul/\'\n            \'c:/86400/(s%20day-1)/:c/mul/Y/-61/16/RANGE/X/269/336/RANGE/data.nc\')\n\n\ndef normalize_recipe(source, destination, origin):\n    """Rename the expression output; values, coordinates and units stay intact."""\n    with xr.open_dataset(source, decode_times=False, engine=\'netcdf4\') as ds:\n        ds = ds.load()\n    adapter.exigir(set(ds.data_vars) == {\'aprod\'}, \'Unexpected IRI expression output.\')\n    adapter.exigir(ds.aprod.attrs.get(\'units\') == \'mm/day\', \'Recipe did not return mm/day.\')\n    adapter.exigir(ds.aprod.attrs.get(\'long_name\') == \'Precipitation Rate\', \'Unexpected source variable.\')\n    ds = ds.rename({\'aprod\': \'prec\'})\n    ds.prec.attrs[\'standard_name\'] = \'lwe_precipitation_rate\'\n    ds.attrs.update(acquisition=\'explicit evaluation of the original IRI catalog expression\',\n                    catalog_definition=CATALOG_URL, acquisition_url=recipe_url(origin))\n    ds.to_netcdf(destination, engine=\'netcdf4\')\n    # Apply the unchanged contract for lead, units, grid, origin and member IDs.\n    return adapter.carregar_membros(destination, pd.DatetimeIndex([origin]))[0]\n\n\ndef verify_overlap(cached, explicit, expected):\n    """Exact equality, without fitted tolerance, rescaling, interpolation or filling."""\n    adapter.exigir(cached.shape == explicit.shape == (28, 78, 68), \'Wrong field shape.\')\n    expected = np.asarray(expected, dtype=bool)\n    adapter.exigir(expected.shape == (28,) and expected.sum() in (24, 28), \'Wrong member mask.\')\n    adapter.exigir(not np.isinf(cached).any(), \'Infinite values in named product.\')\n    adapter.exigir(np.isnan(cached[~expected]).all() and np.isnan(explicit[~expected]).all(),\n                   \'Unexpected inactive members.\')\n    adapter.exigir(np.isfinite(explicit[expected]).all() and (explicit[expected] >= 0).all(),\n                   \'Explicit recipe still has missing, partial, infinite or negative members.\')\n    complete = np.isfinite(cached).all(axis=(1, 2)) & expected\n    absent = np.isnan(cached).all(axis=(1, 2))\n    adapter.exigir((complete | absent).all(), \'Partial named-product member; investigate separately.\')\n    adapter.exigir(complete.sum() >= expected.sum() - 4, \'Insufficient complete overlap with named product.\')\n    adapter.exigir(np.array_equal(cached[complete], explicit[complete]),\n                   \'Explicit recipe differs from named product; source compatibility not established.\')\n    return dict(complete_cached_members=(np.flatnonzero(complete) + 1).tolist(),\n                recovered_members=(np.flatnonzero(expected & absent) + 1).tolist(),\n                values_identical_on_overlap=True, maximum_absolute_difference=0.0,\n                complete_expected_members=int(expected.sum()),\n                ensemble_joined_or_imputed=False, alternate_provider_used=False)\n\n\ndef prepare(registry, target):\n    # Existing modules are also used by the standalone Kaggle bundle.\n    from registry import Registro, deslocar\n    from collect_sources import baixar\n    from cfsv2_compatibility import legacy_url\n\n    registry = Path(registry)\n    adapter.exigir((registry / \'eventos\').is_dir() and any((registry / \'eventos\').glob(\'*.json\')),\n                   \'Restore the existing registry; do not start another history.\')\n    r = Registro(registry)\n    r.eventos(conferir_objetos=True)\n    origin = pd.Timestamp(deslocar(target, -1) + \'-01\')\n    nominal_url = legacy_url(origin)\n    table_url = (\'https://iridl.ldeo.columbia.edu/SOURCES/.NOAA/.NCEP/.EMC/.CFSv2/\'\n                 \'.REALTIME_ENSEMBLE/.FLXF/sampleS/\' + nominal_url.split(\'/.prec/\')[1].split(\'L/1.5\')[0] + \'data.nc\')\n    parents = []\n    for url in [nominal_url, table_url, recipe_url(origin)]:\n        event = baixar(r, \'cfsv2\', url, \'cfsv2_bruto\', target)\n        adapter.exigir(event is not None, \'Acquisition failed; no forecast input created.\')\n        parents.append(event)\n    try:\n        with tempfile.TemporaryDirectory() as td:\n            root = Path(td)\n            paths = []\n            for i, event in enumerate(parents):\n                p = root / f\'parent_{i}.nc\'\n                p.write_bytes(r.ler_objeto(event[\'dados\'][\'objeto\']))\n                paths.append(p)\n            normalized = root / \'explicit_recipe_normalized.nc\'\n            explicit = normalize_recipe(paths[2], normalized, origin)\n            cached = adapter.carregar_membros(paths[0], pd.DatetimeIndex([origin]))[0]\n            table = adapter.tabela_inicializacoes(paths[1], pd.DatetimeIndex([origin]))\n            equivalence = verify_overlap(cached, explicit, table[origin][\'mascara\'])\n            ds, audit = adapter.processar_bloco(normalized, pd.DatetimeIndex([origin]), table)\n            ds.attrs.update(acquisition_url=recipe_url(origin), catalog_definition=CATALOG_URL,\n                            named_product_equivalence=\'exact equality on every complete cached member\',\n                            grid=\'IRI regridAverage on a one-degree grid; not the native Gaussian grid\')\n            out = root / \'cfsv2.nc\'\n            ds.to_netcdf(out, engine=\'netcdf4\')\n            # Keep every transformation source, including the unchanged adapter.\n            code_ref = r.guardar(Path(adapter.__file__).read_bytes())\n            proof = r.adicionar(\'equivalencia_fonte\', dict(\n                fonte=\'cfsv2\', mes_alvo=target, pais=[e[\'id\'] for e in parents],\n                objects_description=\'unchanged member/date/mean adapter\', objetos=[code_ref],\n                catalog_definition=CATALOG_URL, comparison=equivalence,\n                model_changed=False, source_product_changed=False))\n            event = r.derivado(\'cfsv2\', out.read_bytes(), dict(\n                validado=True, meses=[origin.strftime(\'%Y-%m\')], meses_alvo=[target],\n                unidades=\'mm/day\', lead=1.5, auditoria=audit, equivalencia=equivalence,\n                equivalencia_evento=proof[\'id\'], aquisicao=\'original IRI recipe, explicitly evaluated\'),\n                [e[\'id\'] for e in parents], Path(__file__))\n    except ValueError as error:\n        r.adicionar(\'auditoria_falhou\', dict(fonte=\'cfsv2\', mes_alvo=target,\n                    motivo=str(error), pais=[e[\'id\'] for e in parents], objetos=[],\n                    acao=\'explicit recipe rejected; no filling or alternate product used\'))\n        raise\n    finally:\n        r.exportar_resumo(registry / \'resumo_registro.json\')\n    print(f\'CFSv2 validated: {audit[0]["membros"]} members; exact original-product overlap; {event["id"]}\', flush=True)\n    return event\n\n\nif __name__ == \'__main__\':\n    import argparse\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--registry\', required=True)\n    parser.add_argument(\'--target\', required=True)\n    args = parser.parse_args()\n    prepare(args.registry, args.target)\n', 'research/prospective/registry.py': '"""Auditable local registry; not a signature or independent timestamp."""\nfrom __future__ import annotations\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport re\n\nclass ContratoError(ValueError):\n    pass\n\ndef exigir(ok, mensagem):\n    if not ok:\n        raise ContratoError(mensagem)\n\ndef agora():\n    return datetime.now(timezone.utc)\n\ndef instante(valor):\n    t = datetime.fromisoformat(str(valor).replace(\'Z\', \'+00:00\'))\n    exigir(t.tzinfo is not None, \'Timestamp has no timezone.\')\n    return t.astimezone(timezone.utc)\n\ndef mes(valor):\n    exigir(bool(re.fullmatch(\'\\\\d{4}-\\\\d{2}\', str(valor))), \'Month must use YYYY-MM.\')\n    t = datetime.strptime(valor, \'%Y-%m\').replace(tzinfo=timezone.utc)\n    return t\n\ndef deslocar(valor, n):\n    t = mes(valor)\n    k = t.year * 12 + t.month - 1 + n\n    return f\'{k // 12:04d}-{k % 12 + 1:02d}\'\n\ndef canonico(obj):\n    return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(\',\', \':\'), allow_nan=False).encode(\'utf-8\')\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n# Exclusive creation prevents silent replacement of a recorded artifact.\ndef escrever_novo(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'xb\') as f:\n        f.write(data)\n        f.flush()\n        os.fsync(f.fileno())\n\nclass Registro:\n\n    def __init__(self, pasta):\n        self.pasta = Path(pasta).resolve()\n        self.pasta.mkdir(parents=True, exist_ok=True)\n        for nome in (\'objetos\', \'eventos\'):\n            (self.pasta / nome).mkdir(exist_ok=True)\n\n    @contextmanager\n    def transacao(self):\n        lock = self.pasta / \'.escrita.lock\'\n        try:\n            fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)\n        except FileExistsError:\n            raise ContratoError(\'Registry is in use or has a lock from an interrupted run. Do not remove it automatically.\') from None\n        os.close(fd)\n        try:\n            yield\n        finally:\n            lock.unlink()\n\n    def guardar(self, dados):\n        h = digest(dados)\n        p = self.pasta / \'objetos\' / (h + \'.bin\')\n        if p.exists():\n            exigir(p.read_bytes() == dados, \'Registry object changed.\')\n        else:\n            try:\n                escrever_novo(p, dados)\n            except FileExistsError:\n                exigir(p.read_bytes() == dados, \'Object collision.\')\n        return dict(sha256=h, bytes=len(dados))\n\n    def ler_objeto(self, ref):\n        h = ref[\'sha256\']\n        exigir(bool(re.fullmatch(\'[0-9a-f]{64}\', h)), \'Invalid hash.\')\n        p = self.pasta / \'objetos\' / (h + \'.bin\')\n        exigir(p.is_file(), \'Missing object: \' + h)\n        b = p.read_bytes()\n        exigir(len(b) == ref[\'bytes\'] and digest(b) == h, \'Modified object: \' + h)\n        return b\n\n    def eventos(self, conferir_objetos=False):\n        out, anterior, ultimo = ([], None, None)\n        for i, p in enumerate(sorted((self.pasta / \'eventos\').glob(\'*.json\')), 1):\n            e = json.loads(p.read_text(encoding=\'utf-8\'))\n            h = e.pop(\'id\')\n            exigir(digest(canonico(e)) == h, \'Event changed: \' + p.name)\n            exigir(p.name == f\'{i:08d}_{h}.json\', \'Incomplete event sequence.\')\n            exigir(e[\'sequencia\'] == i and e[\'anterior\'] == anterior, \'Invalid event chain.\')\n            t = instante(e[\'registrado_em_utc\'])\n            exigir(ultimo is None or t >= ultimo, \'Clock moved backwards; forecast blocked.\')\n            e[\'id\'] = h\n            if conferir_objetos:\n                for ref in e[\'dados\'].get(\'objetos\', []):\n                    self.ler_objeto(ref)\n            out.append(e)\n            anterior, ultimo = (h, t)\n        return out\n\n    def _adicionar(self, tipo, dados, prazo_exclusivo=None):\n        existentes = self.eventos()\n        t = agora()\n        exigir(not existentes or t >= instante(existentes[-1][\'registrado_em_utc\']), \'Clock moved backwards.\')\n        exigir(prazo_exclusivo is None or t < prazo_exclusivo, \'Issue deadline passed before the event could be registered.\')\n        e = dict(sequencia=len(existentes) + 1, anterior=existentes[-1][\'id\'] if existentes else None, registrado_em_utc=t.isoformat(), tipo=tipo, dados=dados)\n        e[\'id\'] = digest(canonico(e))\n        escrever_novo(self.pasta / \'eventos\' / f\'{e[\'sequencia\']:08d}_{e[\'id\']}.json\', canonico(e))\n        return e\n\n    def adicionar(self, tipo, dados):\n        with self.transacao():\n            return self._adicionar(tipo, dados)\n\n    def obter(self, ident, tipo=None):\n        encontrados = [e for e in self.eventos() if e[\'id\'] == ident]\n        exigir(len(encontrados) == 1, \'Unknown event.\')\n        e = encontrados[0]\n        exigir(tipo is None or e[\'tipo\'] == tipo, \'Incorrect event type.\')\n        return e\n\n    def iniciar(self, plano):\n        with self.transacao():\n            anteriores = [e for e in self.eventos() if e[\'tipo\'] == \'protocolo\']\n            if anteriores:\n                exigir(len(anteriores) == 1 and anteriores[0][\'dados\'][\'plano\'] == plano, \'Plan changed. Start a separate registry and experiment.\')\n                return anteriores[0]\n            exigir(agora() < mes(plano[\'primeiro_mes\']), \'The protocol cannot start with a past or ongoing month.\')\n            return self._adicionar(\'protocolo\', dict(plano=plano, objetos=[]))\n\n    def protocolo(self):\n        e = [e for e in self.eventos() if e[\'tipo\'] == \'protocolo\']\n        exigir(len(e) == 1, \'Initialize the protocol first.\')\n        return e[0]\n\n    def recibo(self, fonte, dados, metadados, transporte):\n        """Use the actual write time, never file mtime or a nominal monthly date."""\n        exigir(fonte in self.protocolo()[\'dados\'][\'plano\'][\'fontes\'], \'Source outside the protocol.\')\n        with self.transacao():\n            ref = self.guardar(dados)\n            return self._adicionar(\'fonte_recebida\', dict(fonte=fonte, objeto=ref, objetos=[ref], inspecao=metadados, transporte=transporte, primeira_publicacao_utc=None, evidencia=\'bytes received during this acquisition; not proof of first historical publication\'))\n\n    def modelo(self, arquivos, auditoria):\n        """Register the final model package; this does not train or certify the algorithm."""\n        p = self.protocolo()\n        plano = p[\'dados\'][\'plano\']\n        exigir(set(arquivos) == set(plano[\'arquivos_modelo\']), \'Incomplete or unexpected model package.\')\n        exigir(auditoria[\'treino_inicio\'] == plano[\'treino_inicio\'] and auditoria[\'treino_fim\'] == plano[\'treino_fim\'], \'Training differs from the plan.\')\n        exigir(auditoria.get(\'protocolo_id\') == p[\'id\'], \'Audit does not match the protocol.\')\n        with self.transacao():\n            exigir(not any((e[\'tipo\'] == \'modelo_congelado\' for e in self.eventos())), \'A model is already frozen for this experiment.\')\n            refs = {nome: self.guardar(Path(path).read_bytes()) for nome, path in arquivos.items()}\n            return self._adicionar(\'modelo_congelado\', dict(protocolo=p[\'id\'], arquivos=refs, objetos=list(refs.values()), auditoria=auditoria))\n\n    def derivado(self, fonte, dados, metadados, pais, codigo):\n        exigir(bool(pais) and len(pais) == len(set(pais)), \'Missing or duplicate parent files.\')\n        exigir(fonte in self.protocolo()[\'dados\'][\'plano\'][\'fontes\'], \'Source outside the protocol.\')\n        with self.transacao():\n            for ident in pais:\n                e = self.obter(ident, \'fonte_recebida\')\n                exigir(e[\'dados\'][\'fonte\'] == fonte, \'Parent belongs to a different source.\')\n                self.ler_objeto(e[\'dados\'][\'objeto\'])\n            ref = self.guardar(dados)\n            cod = self.guardar(Path(codigo).read_bytes())\n            return self._adicionar(\'fonte_recebida\', dict(fonte=fonte, objeto=ref, objetos=[ref, cod], inspecao=metadados, pais=pais, codigo=cod, primeira_publicacao_utc=None, transporte=dict(metodo=\'local transformation of registered parent files\'), evidencia=\'parent acquisition and processing with preserved adapter code\'))\n\n    def prontidao(self, alvo):\n        mes(alvo)\n        protocolo = self.protocolo()\n        p = protocolo[\'dados\'][\'plano\']\n        exigir(p[\'primeiro_mes\'] <= alvo <= p[\'ultimo_mes\'], \'Month outside the monitoring period.\')\n        ev = self.eventos(conferir_objetos=True)\n        fontes, faltantes = ({}, [])\n        for fonte, regra in p[\'fontes\'].items():\n            if not regra[\'obrigatoria\']:\n                continue\n            esperado = deslocar(alvo, -regra[\'lag\'])\n            candidatos = [e for e in ev if e[\'tipo\'] == \'fonte_recebida\' and e[\'dados\'][\'fonte\'] == fonte and (esperado in e[\'dados\'][\'inspecao\'].get(\'meses\', [])) and (e[\'dados\'][\'inspecao\'].get(\'validado\') is True) and (instante(e[\'registrado_em_utc\']) < mes(alvo))]\n            if candidatos:\n                fontes[fonte] = candidatos[-1][\'id\']\n            else:\n                faltantes.append(f\'{fonte}: month {esperado}\')\n        modelos = [e for e in ev if e[\'tipo\'] == \'modelo_congelado\']\n        if not modelos:\n            faltantes.append(\'model: final package for the lagged-source baseline has not been registered\')\n        if agora() >= mes(alvo):\n            faltantes.append(\'deadline: the month has started; backdating is prohibited\')\n        emitida = any(e[\'tipo\'] == \'previsao_congelada\' and e[\'dados\'][\'mes_alvo\'] == alvo for e in ev)\n        return dict(mes_alvo=alvo, pronto=not faltantes, faltantes=faltantes, fontes=fontes, modelo=modelos[0][\'id\'] if modelos else None, protocolo=protocolo[\'id\'], previsao_emitida=emitida)\n\n    def congelar(self, alvo, arquivo, inferencia):\n        """Accept audited inference only; no backdating or overwriting."""\n        from data_contracts import validar_previsao\n        with self.transacao():\n            exigir(not any((e[\'tipo\'] == \'previsao_congelada\' and e[\'dados\'][\'mes_alvo\'] == alvo for e in self.eventos())), \'A prediction already exists; revisions require a separate series.\')\n            status = self.prontidao(alvo)\n            exigir(status[\'pronto\'], \'Forecast blocked: \' + \'; \'.join(status[\'faltantes\']))\n            exigir(inferencia.get(\'modelo\') == status[\'modelo\'] and inferencia.get(\'fontes\') == status[\'fontes\'] and (inferencia.get(\'protocolo\') == status[\'protocolo\']), \'Inference did not use the selected input versions.\')\n            b = Path(arquivo).read_bytes()\n            exigir(inferencia.get(\'previsao_sha256\') == digest(b), \'Inference hash differs from the file.\')\n            resumo = validar_previsao(arquivo, alvo)\n            ref = self.guardar(b)\n            exigir(agora() < mes(alvo), \'Deadline passed during preparation.\')\n            return self._adicionar(\'previsao_congelada\', dict(mes_alvo=alvo, protocolo=status[\'protocolo\'], modelo=status[\'modelo\'], fontes=status[\'fontes\'], inferencia=inferencia, previsao=ref, objetos=[ref], validacao=resumo, prazo_exclusivo_utc=mes(alvo).isoformat(), carimbo_independente=False), prazo_exclusivo=mes(alvo))\n\n    def exportar_resumo(self, destino):\n        ev = self.eventos(conferir_objetos=True)\n        d = dict(eventos=len(ev), ponta=ev[-1][\'id\'] if ev else None, conferido_em_utc=agora().isoformat(), garantia=\'local event chain and hashes; not a signature or tamper-proof storage\', fontes=[dict(id=e[\'id\'], recebido_em_utc=e[\'registrado_em_utc\'], **e[\'dados\']) for e in ev if e[\'tipo\'] == \'fonte_recebida\'], previsoes=[e for e in ev if e[\'tipo\'] == \'previsao_congelada\'])\n        Path(destino).write_text(json.dumps(d, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n        return d\n', 'research/prospective/test_cfsv2_compatibility.py': '"""Synthetic source-contract checks; no network or model training."""\nimport unittest\n\nimport numpy as np\nimport xarray as xr\n\nfrom research.prospective import cfsv2_compatibility as c\n\n\ndef fixture():\n    a = np.ones((1, 28, 1, 78, 68), dtype=\'float64\')\n    a[:, 24:] = np.nan\n    return xr.Dataset(\n        {\'pr\': ((\'S\', \'M\', \'L\', \'Y\', \'X\'), a,\n                dict(units=\'mm/day\', standard_name=\'lwe_precipitation_rate\'))},\n        coords=dict(S=np.array([\'2026-09-01\'], dtype=\'datetime64[ns]\'),\n                    M=np.arange(1, 29), L=[1], Y=c.legacy.LAT, X=c.legacy.LON_360,\n                    target=((\'S\', \'L\'), np.array([[\'2026-10-01\']], dtype=\'datetime64[ns]\')),\n                    target_bnds=((\'S\', \'L\', \'nbound\'), np.array(\n                        [[[\'2026-10-01\', \'2026-11-01\']]], dtype=\'datetime64[ns]\'))))\n\n\nclass CompatibilityTests(unittest.TestCase):\n    def test_matching_overlap_does_not_authorize_missing_legacy_members(self):\n        new = c.successor_fields(fixture(), \'2026-09-01\')\n        old = new.copy()\n        old[20:] = np.nan\n        # Exclude new members from the comparison, even if their rainfall is large.\n        new[20:24] = 100\n        report = c.compare_fields(old, new, \'2026-09-01\')\n        self.assertEqual(report[\'ensemble_mean_difference_rmse\'], 0)\n        self.assertTrue(report[\'numerical_agreement_on_overlap\'])\n        self.assertFalse(report[\'legacy_complete\'])\n        self.assertTrue(report[\'successor_complete\'])\n        self.assertFalse(report[\'operationally_authorized\'])\n        self.assertEqual(report[\'common_complete_members\'], list(range(1, 21)))\n\n    def test_changed_values_are_not_accepted_due_to_matching_metadata(self):\n        old = c.successor_fields(fixture(), \'2026-09-01\')\n        new = old.copy()\n        new[:24, 0, 0] += 10\n        report = c.compare_fields(old, new, \'2026-09-01\')\n        self.assertTrue(report[\'legacy_complete\'] and report[\'successor_complete\'])\n        self.assertFalse(report[\'numerical_agreement_on_overlap\'])\n        self.assertFalse(report[\'operationally_authorized\'])\n        self.assertEqual(report[\'maximum_absolute_member_difference\'], 10)\n\n    def test_target_bounds_and_units_are_verified_not_inferred_from_lead(self):\n        for change in [\'bounds\', \'target\', \'units\', \'origin\', \'member\', \'grid\']:\n            ds = fixture()\n            if change == \'bounds\':\n                ds.target_bnds.values[0, 0, 1] = np.datetime64(\'2026-12-01\')\n            elif change == \'target\':\n                ds.target.values[0, 0] = np.datetime64(\'2026-09-01\')\n            elif change == \'units\':\n                ds.pr.attrs[\'units\'] = \'mm\'\n            elif change == \'origin\':\n                ds = ds.assign_coords(S=np.array([\'2026-08-01\'], dtype=\'datetime64[ns]\'))\n            elif change == \'member\':\n                ds = ds.assign_coords(M=np.arange(28))\n            elif change == \'grid\':\n                ds = ds.isel(Y=slice(None, None, -1))\n            with self.subTest(change=change), self.assertRaises(ValueError):\n                c.successor_fields(ds, \'2026-09-01\')\n\n    def test_partial_member_is_not_treated_as_a_complete_member(self):\n        new = c.successor_fields(fixture(), \'2026-09-01\')\n        old = new.copy()\n        new[10, 0, 0] = np.nan\n        report = c.compare_fields(old, new, \'2026-09-01\')\n        self.assertFalse(report[\'successor_complete\'])\n        self.assertNotIn(11, report[\'common_complete_members\'])\n        self.assertEqual(report[\'successor_finite_pixels\'][10], 5303)\n\n    def test_no_overlap_is_inconclusive_and_november_requires_28(self):\n        new = c.successor_fields(fixture(), \'2026-09-01\')\n        old = np.full_like(new, np.nan)\n        report = c.compare_fields(old, new, \'2026-09-01\')\n        self.assertEqual(report[\'status\'], \'no_complete_overlap\')\n        self.assertIsNone(report[\'numerical_agreement_on_overlap\'])\n        self.assertEqual(int(c.expected_mask(\'2025-11-01\').sum()), 28)\n        self.assertFalse(c.compare_fields(new, new, \'2025-11-01\')[\'successor_complete\'])\n\n    def test_unexpected_members_and_invalid_values_are_rejected(self):\n        for value, index in [(1, 24), (-1, 0), (np.inf, 0)]:\n            ds = fixture()\n            ds.pr.values[0, index, 0, 0, 0] = value\n            with self.subTest(value=value, index=index), self.assertRaises(ValueError):\n                c.successor_fields(ds, \'2026-09-01\')\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'research/prospective/test_cfsv2_recipe.py': '"""Guards against turning an IRI recipe refresh into a source substitution."""\nimport tempfile\nfrom pathlib import Path\nimport unittest\n\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nfrom research.prospective import prepare_cfsv2_recipe as c\n\n\nclass RecipeTests(unittest.TestCase):\n    def arrays(self):\n        explicit = np.full((28, 78, 68), np.nan)\n        explicit[:24] = np.arange(24)[:, None, None] / 2\n        cached = explicit.copy()\n        cached[20:] = np.nan\n        return cached, explicit, np.arange(28) < 24\n\n    def test_complete_original_recipe_can_recover_wholly_missing_members(self):\n        cached, explicit, mask = self.arrays()\n        result = c.verify_overlap(cached, explicit, mask)\n        self.assertEqual(result[\'recovered_members\'], [21, 22, 23, 24])\n        self.assertTrue(result[\'values_identical_on_overlap\'])\n        self.assertFalse(result[\'ensemble_joined_or_imputed\'])\n        self.assertFalse(result[\'alternate_provider_used\'])\n\n    def test_even_tiny_value_change_fails_exact_compatibility(self):\n        cached, explicit, mask = self.arrays()\n        explicit[2, 4, 5] += 1e-8\n        with self.assertRaisesRegex(ValueError, \'differs from named product\'):\n            c.verify_overlap(cached, explicit, mask)\n\n    def test_partial_or_still_missing_or_negative_members_are_rejected(self):\n        for scenario in [\'partial_cached\', \'missing_new\', \'negative_new\', \'inactive_new\', \'too_little_overlap\']:\n            cached, explicit, mask = self.arrays()\n            if scenario == \'partial_cached\': cached[0, 0, 0] = np.nan\n            elif scenario == \'missing_new\': explicit[21, 0, 0] = np.nan\n            elif scenario == \'negative_new\': explicit[21, 0, 0] = -1\n            elif scenario == \'inactive_new\': explicit[24, 0, 0] = 1\n            elif scenario == \'too_little_overlap\': cached[19] = np.nan\n            with self.subTest(scenario=scenario), self.assertRaises(ValueError):\n                c.verify_overlap(cached, explicit, mask)\n\n    def test_wrong_units_or_lead_cannot_be_fixed_by_renaming(self):\n        _, values, _ = self.arrays()\n        ds = xr.Dataset({\'aprod\': ((\'S\',\'L\',\'M\',\'Y\',\'X\'), values[None, None],\n                         dict(units=\'mm/day\', long_name=\'Precipitation Rate\'))},\n                        coords=dict(S=[800.0], L=[1.5], M=np.arange(1,29), Y=np.arange(-61,17), X=np.arange(269,337)))\n        ds.S.attrs = dict(calendar=\'360\', units=\'months since 1960-01-01\')\n        ds.L.attrs = dict(units=\'months\', pointwidth=1.0)\n        ds.Y.attrs[\'units\'] = \'degree_north\'; ds.X.attrs[\'units\'] = \'degree_east\'\n        with tempfile.TemporaryDirectory() as td:\n            p, q = Path(td)/\'raw.nc\', Path(td)/\'norm.nc\'\n            ds.to_netcdf(p)\n            actual = c.normalize_recipe(p, q, pd.Timestamp(\'2026-09-01\'))\n            np.testing.assert_array_equal(actual, values)\n            for field in [\'units\', \'lead\']:\n                changed = ds.copy(deep=True)\n                if field == \'units\': changed.aprod.attrs[\'units\'] = \'mm\'\n                else: changed = changed.assign_coords(L=[0.5]); changed.L.attrs = ds.L.attrs\n                changed.to_netcdf(p)\n                with self.subTest(field=field), self.assertRaises(ValueError):\n                    c.normalize_recipe(p, q, pd.Timestamp(\'2026-09-01\'))\n\n    def test_hindcast_is_not_routed_to_forecast_recipe(self):\n        with self.assertRaises(ValueError):\n            c.recipe_url(\'2010-09-01\')\n        url = c.recipe_url(\'2026-09-01\')\n        self.assertIn(\'.REALTIME_ENSEMBLE/.FLXF/.surface/.PRATE/\', url)\n        self.assertIn(\'regridAverage/nip/\', url)\n        self.assertNotIn(\'ccsr\', url)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'research/prospective/test_inference.py': '"""Synthetic checks for fitted-package replay and receipt adapters."""\nfrom pathlib import Path\nimport importlib.metadata\nimport json\nimport shutil\nimport sys\nimport tempfile\nimport unittest\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path.insert(0, str(ROOT))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom research.common.inputs import LIBRARY, write_json\nfrom research.common.baseline import fit, predict as predict_reference\nfrom research.common.synthetic_fixture import fixture\nfrom infer_baseline import predict, psl_value\nfrom prepare_cds import regular_grid, convert, FIELDS\nfrom collect_cds import request_for\n\n\nclass InferenceTests(unittest.TestCase):\n    def test_coordinate_reordering_and_duplicate_rejection(self):\n        lat, lon, values = regular_grid([1, 1, 0, 0], [270, 271, 270, 271], [3, 4, 1, 2])\n        np.testing.assert_array_equal(lat, [0, 1])\n        np.testing.assert_array_equal(lon, [-90, -89])\n        np.testing.assert_array_equal(values, [[1, 2], [3, 4]])\n        with self.assertRaises(ValueError):\n            regular_grid([0, 0, 1, 1], [270, 270, 271, 271], [1, 2, 3, 4])\n\n    def test_requests_respect_lags_and_final_grib_evidence(self):\n        for source in [\'era5_sl\', \'era5_pl\', \'seas5\']:\n            dataset, request = request_for(source, \'2026-10\')\n            self.assertEqual(request[\'month\'], [\'09\'] if source == \'seas5\' else [\'06\'])\n            self.assertEqual(request[\'data_format\'], \'grib\')\n        self.assertEqual(request_for(\'era5_pl\', \'2026-10\')[1][\'pressure_level\'], [\'850\'])\n\n    def test_psl_rejects_unavailable_month(self):\n        with tempfile.TemporaryDirectory() as temp:\n            p = Path(temp) / \'psl.data\'\n            p.write_text(\'2026 2026\\n2026 1 2 3 4 5 6 7 -99.99 -99.99 -99.99 -99.99 -99.99\\n-99.99\\n\')\n            self.assertEqual(psl_value(p, pd.Timestamp(\'2026-07-01\')), 7)\n            with self.assertRaises(ValueError):\n                psl_value(p, pd.Timestamp(\'2026-08-01\'))\n\n    def test_model_package_replays_direct_baseline(self):\n        data = fixture()\n        data.audit = {\'fixture\': True}\n        dates = pd.DatetimeIndex([\'2007-01-01\'])\n        with tempfile.TemporaryDirectory() as temp:\n            temp = Path(temp)\n            package = temp / \'model\'\n            state = fit(data, pd.Timestamp(\'2006-09-01\'), package)\n            expected = predict_reference(data, state, dates)\n            shutil.copyfile(LIBRARY, package / \'biblioteca.py\')\n            write_json(package / \'ambiente.json\', {n: importlib.metadata.version(n) for n in [\'numpy\',\'pandas\',\'xarray\',\'lightgbm\']})\n            paths = {}\n            for source, fields in FIELDS.items():\n                names = [row[0] for row in fields.values()]\n                ds = xr.Dataset({n: data.atmosphere[n].sel(time=[pd.Timestamp(\'2006-09-01\')]) for n in names},\n                    attrs=dict(schema=\'worcap_era5_prospective_v1\', expver=1))\n                path = temp / (source + \'.nc\')\n                ds.to_netcdf(path)\n                paths[source] = path\n            for name in data.lib.INDICES_NOMES:\n                path = temp / (name + \'.data\')\n                row = data.lib.INDICES_OC.loc[\'2006-01-01\':\'2006-12-01\', name]\n                path.write_text(\'2006 2006\\n2006 \' + \' \'.join(str(float(v)) for v in row) + \'\\n-99.99\\n\')\n                paths[name] = path\n            for name, field, schema in [(\'seas5\', data.seas, \'worcap_seas5_prospective_v1\'),\n                                        (\'cfsv2\', data.cfs, \'nimbus_cfsv2_prospectivo_v1\')]:\n                ds = field.sel(time=dates).rename(name + \'_tp_media\').to_dataset()\n                ds.attrs[\'schema\'] = schema\n                ds[name + \'_tp_media\'].attrs[\'units\'] = \'mm/day\'\n                path = temp / (name + \'.nc\')\n                ds.to_netcdf(path)\n                paths[name] = path\n            actual = predict(package, paths, \'2007-01\')\n            np.testing.assert_allclose(actual.precipitacao, expected, rtol=0, atol=1e-7)\n            np.testing.assert_array_equal(actual.climatologia.values, state[\'climate\'].values[[0]])\n            with self.assertRaises(ValueError):\n                predict(package, paths, \'2007-02\')\n\n    def test_real_grib_decoder_final_era5_and_era5t_rejection(self):\n        import eccodes as ec\n        from data_contracts import LAT, LON\n        def write(path, expver):\n            with Path(path).open(\'wb\') as f:\n                for param in FIELDS[\'era5_sl\']:\n                    g = ec.codes_grib_new_from_samples(\'regular_ll_sfc_grib1\')\n                    try:\n                        for k, v in dict(Ni=len(LON), Nj=len(LAT), latitudeOfFirstGridPointInDegrees=15.,\n                            latitudeOfLastGridPointInDegrees=-60., longitudeOfFirstGridPointInDegrees=270.,\n                            longitudeOfLastGridPointInDegrees=335., iDirectionIncrementInDegrees=.25,\n                            jDirectionIncrementInDegrees=.25, dataDate=20260601, dataTime=0,\n                            **{\'class\':\'ea\', \'stream\':\'moda\', \'expver\':expver, \'paramId\':param}).items():\n                            ec.codes_set(g, k, v)\n                        ec.codes_set_values(g, np.ones(len(LAT) * len(LON)))\n                        ec.codes_write(g, f)\n                    finally:\n                        ec.codes_release(g)\n        with tempfile.TemporaryDirectory() as temp:\n            path = Path(temp) / \'era5.grib\'\n            write(path, \'0001\')\n            ds, meta = convert(path, \'era5_sl\', \'2026-10\')\n            self.assertEqual(set(ds.data_vars), {\'cloud_cover\',\'t2\',\'surface_pressure\'})\n            self.assertEqual(meta[\'meses\'], [\'2026-06\'])\n            np.testing.assert_array_equal(ds.lat, LAT)\n            write(path, \'0005\')\n            with self.assertRaises(ValueError):\n                convert(path, \'era5_sl\', \'2026-10\')\n\n    def test_seas5_requires_all_members_and_correct_target(self):\n        import eccodes as ec\n        def write(path, members):\n            with path.open(\'wb\') as f:\n                for member in range(members):\n                    g = ec.codes_grib_new_from_samples(\'regular_ll_sfc_grib1\')\n                    try:\n                        settings = dict(localDefinitionNumber=16, stream=\'msmm\', type=\'fcmean\', system=51,\n                            origin=\'ecmf\', dataDate=20260901, dataTime=0, forecastMonth=2, verifyingMonth=202610,\n                            number=member, numberOfForecastsInEnsemble=51, paramId=172228,\n                            Ni=68, Nj=78, latitudeOfFirstGridPointInDegrees=16., latitudeOfLastGridPointInDegrees=-61.,\n                            longitudeOfFirstGridPointInDegrees=269., longitudeOfLastGridPointInDegrees=336.,\n                            iDirectionIncrementInDegrees=1., jDirectionIncrementInDegrees=1.)\n                        for key, value in settings.items():\n                            ec.codes_set(g, key, value)\n                        ec.codes_set_values(g, np.full(68 * 78, (member + 1) / 86400000.))\n                        ec.codes_write(g, f)\n                    finally:\n                        ec.codes_release(g)\n        with tempfile.TemporaryDirectory() as temp:\n            path = Path(temp) / \'seas5.grib\'\n            write(path, 51)\n            ds, meta = convert(path, \'seas5\', \'2026-10\')\n            np.testing.assert_allclose(ds.seas5_tp_media.values, 26., rtol=1e-6)\n            self.assertEqual(len(meta[\'auditoria\']), 51)\n            self.assertEqual(meta[\'meses\'], [\'2026-09\'])\n            with self.assertRaises(ValueError):\n                convert(path, \'seas5\', \'2026-11\')\n            write(path, 50)\n            with self.assertRaises(ValueError):\n                convert(path, \'seas5\', \'2026-10\')\n\n    def test_pressure_level_contract(self):\n        import eccodes as ec\n        from data_contracts import LAT, LON\n        with tempfile.TemporaryDirectory() as temp:\n            path = Path(temp) / \'pressure.grib\'\n            with path.open(\'wb\') as f:\n                for param in FIELDS[\'era5_pl\']:\n                    g = ec.codes_grib_new_from_samples(\'regular_ll_pl_grib1\')\n                    try:\n                        for key, value in dict(Ni=len(LON), Nj=len(LAT), latitudeOfFirstGridPointInDegrees=15.,\n                            latitudeOfLastGridPointInDegrees=-60., longitudeOfFirstGridPointInDegrees=270.,\n                            longitudeOfLastGridPointInDegrees=335., iDirectionIncrementInDegrees=.25,\n                            jDirectionIncrementInDegrees=.25, dataDate=20260601, dataTime=0,\n                            level=850, **{\'class\':\'ea\',\'stream\':\'moda\',\'expver\':\'0001\',\'paramId\':param}).items():\n                            ec.codes_set(g, key, value)\n                        ec.codes_set_values(g, np.ones(len(LAT) * len(LON)))\n                        ec.codes_write(g, f)\n                    finally:\n                        ec.codes_release(g)\n            ds, meta = convert(path, \'era5_pl\', \'2026-10\')\n            self.assertEqual(len(ds.data_vars), 6)\n            self.assertEqual(len(meta[\'auditoria\']), 6)\n\n\nif __name__ == \'__main__\':\n    unittest.main(argv=[__file__])\n', 'research/prospective/test_registry.py': '"""Isolated tests: synthetic inputs never enter the real registry."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom unittest.mock import patch\nimport copy\nimport json\nimport tempfile\nimport unittest\nimport numpy as np\nimport xarray as xr\nfrom data_contracts import LAT, LON, indices_psl\nfrom registry import Registro, ContratoError, digest, deslocar\nfrom evaluate import painel, avaliar\nBASE = Path(__file__).resolve().parent\nNOW = datetime(2026, 9, 28, 20, 0, tzinfo=timezone.utc)\n\nclass TestRegistro(unittest.TestCase):\n\n    def setUp(self):\n        self.temp = tempfile.TemporaryDirectory()\n        self.addCleanup(self.temp.cleanup)\n        self.raiz = Path(self.temp.name)\n        self.clock = patch(\'registry.agora\', return_value=NOW)\n        self.clock.start()\n        self.addCleanup(self.clock.stop)\n        self.clock2 = patch(\'evaluate.agora\', return_value=NOW)\n        self.clock2.start()\n        self.addCleanup(self.clock2.stop)\n        self.r = Registro(self.raiz / \'registro\')\n        self.plano = json.loads((BASE / \'plan.json\').read_text())\n        self.protocolo = self.r.iniciar(self.plano)\n\n    def pronto(self):\n        arquivos = {}\n        for nome in self.plano[\'arquivos_modelo\']:\n            p = self.raiz / nome\n            p.write_bytes(b\'fixture apenas sintetica\')\n            arquivos[nome] = p\n        self.r.modelo(arquivos, dict(treino_inicio=\'1993-01\', treino_fim=\'2022-12\', protocolo_id=self.protocolo[\'id\']))\n        for n, regra in self.plano[\'fontes\'].items():\n            if regra[\'obrigatoria\']:\n                self.r.recibo(n, b\'fixture sintetica \' + n.encode(), dict(validado=True, meses=[deslocar(\'2026-10\', -regra[\'lag\'])]), {\'url\': \'fixture\'})\n        self.fp = self.raiz / \'previsao.nc\'\n        a = np.ones((1, len(LAT), len(LON)), dtype=np.float32)\n        ds = xr.Dataset({n: ((\'time\', \'lat\', \'lon\'), a, {\'units\': \'mm/day\'}) for n in [\'precipitacao\', \'climatologia\']}, coords={\'time\': [np.datetime64(\'2026-10-01\')], \'lat\': LAT, \'lon\': LON})\n        ds.to_netcdf(self.fp)\n        s = self.r.prontidao(\'2026-10\')\n        self.inf = {k: s[k] for k in [\'modelo\', \'fontes\', \'protocolo\']}\n        self.inf[\'previsao_sha256\'] = digest(self.fp.read_bytes())\n\n    def test_coleta_e_integridade(self):\n        e = self.r.recibo(\'nino34\', b\'abc\', dict(validado=False, meses=[]), {\'url\': \'fixture\'})\n        self.assertEqual(self.r.ler_objeto(e[\'dados\'][\'objeto\']), b\'abc\')\n        p = self.r.pasta / \'objetos\' / (e[\'dados\'][\'objeto\'][\'sha256\'] + \'.bin\')\n        p.write_bytes(b\'abd\')\n        with self.assertRaises(ContratoError):\n            self.r.eventos(conferir_objetos=True)\n\n    def test_cadeia_detecta_evento_editado(self):\n        self.r.adicionar(\'coleta_falhou\', {\'objetos\': []})\n        p = sorted((self.r.pasta / \'eventos\').glob(\'*.json\'))[-1]\n        e = json.loads(p.read_text())\n        e[\'tipo\'] = \'fonte_recebida\'\n        p.write_text(json.dumps(e))\n        with self.assertRaises(ContratoError):\n            self.r.eventos()\n\n    def test_protocolo_imutavel(self):\n        self.r.iniciar(self.plano)\n        outro = copy.deepcopy(self.plano)\n        outro[\'modelo\'][\'arvores\'] = 900\n        with self.assertRaises(ContratoError):\n            self.r.iniciar(outro)\n\n    def test_faltantes_bloqueiam(self):\n        s = self.r.prontidao(\'2026-10\')\n        self.assertFalse(s[\'pronto\'])\n        self.assertEqual(len(s[\'faltantes\']), 9)\n        with self.assertRaises(ContratoError):\n            self.r.congelar(\'2026-10\', \'inexistente\', {})\n\n    def test_nao_aceita_mes_errado_ou_quarentena(self):\n        self.r.recibo(\'nino34\', b\'abc\', dict(validado=True, meses=[\'2026-08\']), {\'url\': \'fixture\'})\n        self.r.recibo(\'nino12\', b\'abc\', dict(validado=False, meses=[\'2026-07\']), {\'url\': \'fixture\'})\n        self.assertNotIn(\'nino34\', self.r.prontidao(\'2026-10\')[\'fontes\'])\n        self.assertNotIn(\'nino12\', self.r.prontidao(\'2026-10\')[\'fontes\'])\n\n    def test_congelamento_e_reemissao_bloqueada(self):\n        self.pronto()\n        self.assertFalse(self.r.prontidao(\'2026-10\')[\'previsao_emitida\'])\n        e = self.r.congelar(\'2026-10\', self.fp, self.inf)\n        self.assertEqual(e[\'registrado_em_utc\'], NOW.isoformat())\n        self.assertEqual(e[\'dados\'][\'validacao\'][\'pontos\'], 78561)\n        self.assertTrue(self.r.prontidao(\'2026-10\')[\'previsao_emitida\'])\n        self.assertFalse(self.r.prontidao(\'2026-11\')[\'previsao_emitida\'])\n        with self.assertRaises(ContratoError):\n            self.r.congelar(\'2026-10\', self.fp, self.inf)\n\n    def test_prazo_nao_pode_ser_retroagido(self):\n        self.pronto()\n        with patch(\'registry.agora\', return_value=datetime(2026, 10, 1, tzinfo=timezone.utc)):\n            with self.assertRaises(ContratoError):\n                self.r.congelar(\'2026-10\', self.fp, self.inf)\n        self.assertFalse(any((e[\'tipo\'] == \'previsao_congelada\' for e in self.r.eventos())))\n\n    def test_troca_fonte_apos_inferencia_bloqueia(self):\n        self.pronto()\n        self.r.recibo(\'tna\', b\'outra versao\', dict(validado=True, meses=[\'2026-07\']), {\'url\': \'fixture\'})\n        with self.assertRaises(ContratoError):\n            self.r.congelar(\'2026-10\', self.fp, self.inf)\n\n    def test_grid_e_valores_errados_bloqueiam(self):\n        self.pronto()\n        with xr.open_dataset(self.fp) as raw:\n            ds = raw.load()\n        ds[\'precipitacao\'].values[0, 0, 0] = np.nan\n        ds.to_netcdf(self.fp, mode=\'w\')\n        self.inf[\'previsao_sha256\'] = digest(self.fp.read_bytes())\n        with self.assertRaises(ContratoError):\n            self.r.congelar(\'2026-10\', self.fp, self.inf)\n        ds[\'precipitacao\'].values[0, 0, 0] = 1\n        ds = ds.assign_coords(lon=LON + 0.01)\n        ds.to_netcdf(self.fp, mode=\'w\')\n        self.inf[\'previsao_sha256\'] = digest(self.fp.read_bytes())\n        with self.assertRaises(ContratoError):\n            self.r.congelar(\'2026-10\', self.fp, self.inf)\n\n    def test_painel_inclui_meses_sem_emissao(self):\n        d = painel(self.r)\n        self.assertEqual(d[\'meses_planejados\'], 12)\n        self.assertEqual(d[\'meses_emitidos\'], 0)\n        self.assertEqual(d[\'metricas_primarias\'], {})\n        with patch(\'evaluate.agora\', return_value=datetime(2026, 11, 2, tzinfo=timezone.utc)):\n            self.assertEqual(painel(self.r)[\'meses\'][0][\'estado\'], \'nao_emitido_prazo_encerrado\')\n\n    def test_avaliacao_nao_libera_alvo_antes_da_hora(self):\n        self.pronto()\n        self.r.congelar(\'2026-10\', self.fp, self.inf)\n        with self.assertRaises(ContratoError):\n            avaliar(self.r, \'2026-10\', \'inexistente\', {})\n\n    def test_avaliacao_primaria_nao_sobrescrita_por_revisao(self):\n        self.pronto()\n        self.r.congelar(\'2026-10\', self.fp, self.inf)\n        with xr.open_dataset(self.fp) as pred:\n            truth = pred[[\'precipitacao\']].rename({\'precipitacao\': \'tp\'}).load()\n        obs = self.raiz / \'obs.nc\'\n        truth.to_netcdf(obs)\n        raw = truth.copy(deep=True)\n        raw.tp.values[:] = 0.001\n        raw.tp.attrs = dict(units=\'m\', GRIB_stream=\'moda\', GRIB_experimentVersionNumber=\'0001\')\n        bruto = self.raiz / \'bruto.nc\'\n        raw.to_netcdf(bruto)\n        info = dict(produto=\'ERA5 monthly_averaged_reanalysis\', expver=1, mes_alvo=\'2026-10\', url=\'https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means\', arquivo_bruto=str(bruto))\n        mature = datetime(2027, 2, 2, tzinfo=timezone.utc)\n        with patch(\'registry.agora\', return_value=mature), patch(\'evaluate.agora\', return_value=mature):\n            prim = avaliar(self.r, \'2026-10\', obs, info)\n            self.assertTrue(prim[\'dados\'][\'primaria\'])\n            truth.tp.values[:] = 2\n            truth.to_netcdf(obs, mode=\'w\')\n            with self.assertRaises(ContratoError):\n                avaliar(self.r, \'2026-10\', obs, info)\n            raw.tp.values[:] = 0.002\n            raw.to_netcdf(bruto, mode=\'w\')\n            rev = avaliar(self.r, \'2026-10\', obs, info)\n            self.assertFalse(rev[\'dados\'][\'primaria\'])\n            self.assertEqual(painel(self.r)[\'metricas_primarias\'][\'precipitacao/dominio\'][\'rmse\'], 0)\n\n    def test_derivado_exige_pais_da_mesma_fonte(self):\n        e = self.r.recibo(\'nino34\', b\'abc\', dict(validado=False, meses=[]), {\'url\': \'fixture\'})\n        codigo = self.raiz / \'adapter.py\'\n        codigo.write_text(\'fixture\')\n        with self.assertRaises(ContratoError):\n            self.r.derivado(\'cfsv2\', b\'def\', dict(validado=True, meses=[\'2026-09\']), [e[\'id\']], codigo)\n\n    def test_parser_psl_nao_preenche_ausentes(self):\n        b = (\'2026 2026\\n2026 \' + \' \'.join([\'1\'] * 7 + [\'-99.99\'] * 5) + \'\\n-99.99\\nHadISST\\n\').encode()\n        d = indices_psl(b)\n        self.assertEqual(d[\'ultimo_mes\'], \'2026-07\')\n        self.assertEqual(d[\'ausentes\'], 5)\n        with self.assertRaises(ContratoError):\n            indices_psl(b\'<html>login</html>\')\nif __name__ == \'__main__\':\n    suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestRegistro)\n    resultado = unittest.TextTestRunner(verbosity=2).run(suite)\n    if not resultado.wasSuccessful():\n        raise SystemExit(1)\n'}
BUNDLE_SHA256 = {'competition/metadata/NOAA/indices_noaa.csv': 'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4', 'research/common/__init__.py': '4f6bfaae6ba45669c7d41e1b087df0582970932f06cdea87a4ff3b352f4135f0', 'research/common/baseline.py': '653594bf49fa338304129d7d22e7dc31a4e32647b26ee87af7bea4bad94a4dfa', 'research/common/inputs.py': 'd22140d4ad3db4ac3fecdca26c7ebdd1d780486b51622a1f333fce2a637e6851', 'research/common/synthetic_fixture.py': 'da4de7a662c2c8c25f324c5b543ab04914d490ffde5e316c39187eb8f1640bff', 'research/lagged_sources/evidence/metricas_blocos.csv': 'da2c2af92556c0cb64606e4a8dfd6d5d00af215ecf408cbe326d776df60cb457', 'research/lagged_sources/library.py': '4b92dd77d868d7806fd0ae37d2837abfcde36e173df75b1bd6d88760593ee9c3', 'research/lagged_sources/ocean_indices.csv': '69226b2e65f783b4d4971a254d1e325d0908447c76ec1df3bf532ecef1f0cf5b', 'research/lagged_sources/official_hashes.json': 'bd5648197c4cf25f85d89a32c701b422f1e0b788421766cc7b8aca99ea42223d', 'research/prospective/cfsv2_adapter.py': '79c698b6e0cd1cb16aa21f0b919bc521e4836cdb0252a605f3a7268eb24e136b', 'research/prospective/cfsv2_compatibility.py': 'a4885542851816831882ce03d3188ade1f9a935ef4536bf0ef2573de65b54961', 'research/prospective/collect_cds.py': 'f01c729f733d7c92233ac68e72d1111fd1c96e60158df57de50ab000ce174778', 'research/prospective/collect_sources.py': '1b86c81e84070b26b0c322bef3f727e2774717914616cd092cfecfd88810c270', 'research/prospective/data_contracts.py': '8986792dd070fb1a040c969f47498015e06bae9f91ca6eeaa3018d4ca23ded9f', 'research/prospective/evaluate.py': '0e1297cb9acfb22005fb22e6ea2054e484899933a15a3d349a43dd2d978a20a8', 'research/prospective/fit_baseline.py': 'd6c8bd10a98174eef9f7efc24a465e1a92fa58ded3c6c918bcf97c56cabac38b', 'research/prospective/infer_baseline.py': '6440524a5fa4ee7bd2e7966593fddac887ae351f706d33a9d75ec6e1bf100e6c', 'research/prospective/issue_forecast.py': 'ab570a5fd9cbb528354503985c5c1e748700c5652e4b1c354fc7fc1e44fbe7aa', 'research/prospective/plan.json': '7e1ab0eb8aa5fefaddfbf5a3a04a907a1c77bb7cfee6e4a47a639e6adfb04359', 'research/prospective/prepare_cds.py': '91ca8a69316a19f1b465413790d36318aad8b1f21c9f7d9270e135203fc79cfb', 'research/prospective/prepare_cfsv2.py': '962ca1b8661f7492084aba8896577cfe43d3375bbd43e3469367404f4ae2da10', 'research/prospective/prepare_cfsv2_recipe.py': '9b1a17fd2f5f363896993a26f68e7a9e448f27ede8af20c09db465da4333770c', 'research/prospective/registry.py': '71a2d34a21c5f1005aa424f3465e23f963ec92f06491cd184c605c7d9f036690', 'research/prospective/test_cfsv2_compatibility.py': 'cc52b35d0159f4e0e6467db1ed8ea81e2de8f336728cc0fe90f2b1b9a9832b62', 'research/prospective/test_cfsv2_recipe.py': '04bf2ea21b588972be8ff2529e353546286039d13d267090d50f8cda557047f0', 'research/prospective/test_inference.py': '0e32ce166478f8305f815b81e41d82ac8fcf46f9aaadcbd6d0a2d57598a8f4e3', 'research/prospective/test_registry.py': 'bbac9ddaca1c377f14a157432a07f3d66a101680d7c335dc903ab8897183bc0d'}
for name, content in BUNDLE.items():
    assert hashlib.sha256(content.encode('utf-8')).hexdigest() == BUNDLE_SHA256[name]
    path = CODE_ROOT/name
    if path.exists() and path.read_text(encoding='utf-8') != content:
        raise RuntimeError('Different source code already exists. Preserve this run and choose a new RUN_ROOT.')
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8', newline='\n')
sys.path.insert(0, str(CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT/'research/prospective'))
print('Reviewed source bundle:', CODE_ROOT)


## 1. Restore the actual event chain
Attach the previous registry output with its events and objects. An absent or ambiguous chain stops this continuation. Review the target month; a passed deadline cannot be backdated.

In [ ]:
from registry import Registro
REGISTRY = RUN_ROOT/'registry'
previous = [p.parent for p in Path('/kaggle/input').rglob('resumo_registro.json')
            if (p.parent/'eventos').is_dir() and (p.parent/'objetos').is_dir()]
if not (REGISTRY/'eventos').exists():
    if len(previous) != 1:
        raise RuntimeError('Attach exactly one complete existing registry. This continuation will not invent a new chain.')
    Registro(previous[0]).eventos(conferir_objetos=True)
    REGISTRY.mkdir(parents=True, exist_ok=True)
    for folder in ['eventos','objetos']:
        shutil.copytree(previous[0]/folder,REGISTRY/folder)
r = Registro(REGISTRY)
r.eventos(conferir_objetos=True)
TARGET = '2026-11'  # October is already frozen in the latest registry.
print(json.dumps(r.prontidao(TARGET),indent=2))


## 2. Verify inputs and fit the fixed final baseline
The fit uses 1993–2022 with research lags. An already registered model is reused. No competition model is substituted.

In [ ]:
OFFICIAL = None
SEAS5 = None
CFSV2 = None
from fit_baseline import execute as fit_final
if not any(e['tipo']=='modelo_congelado' for e in r.eventos()):
    MODEL = fit_final(RUN_ROOT/'model', REGISTRY, OFFICIAL, SEAS5, CFSV2)
else:
    print('The registry already contains its immutable model package.')


## 3. Collect and normalize sources
Public inputs need Internet. CFSv2 uses the original IRI catalog recipe, requiring all expected members and exact overlap with the named product. It does not substitute CCSR. To request ERA5/SEAS5, configure your existing CDS credential privately and set `COLLECT_CDS=True`. Future sources may not be available yet; missing data continue to block issuance.

In [ ]:
from collect_sources import executar
from prepare_cfsv2_recipe import prepare
COLLECT_CDS = False
assert not any(e['tipo']=='previsao_congelada' and e['dados']['mes_alvo']==TARGET for e in r.eventos()), 'This month already has a frozen forecast.'
executar(REGISTRY, CODE_ROOT/'research/prospective/plan.json', TARGET, incluir_cfsv2=False)
prepare(REGISTRY, TARGET)
if COLLECT_CDS:
    from collect_cds import execute as acquire_cds
    acquire_cds(REGISTRY, TARGET)
else:
    print('CDS acquisition disabled. Audited ERA5/SEAS5 receipts are still required.')


## 4. Freeze only a complete, timely forecast
The file is generated from exactly the model/source bytes selected by the registry. Incomplete inputs or an expired deadline block emission, and an existing issued forecast is not replaced.

In [ ]:
from issue_forecast import execute as issue
state = r.prontidao(TARGET)
print(json.dumps(state,indent=2))
already_issued = any(e['tipo']=='previsao_congelada' and e['dados']['mes_alvo']==TARGET for e in r.eventos())
if state['pronto'] and not already_issued:
    FORECAST = issue(REGISTRY,TARGET,RUN_ROOT/('forecast_'+TARGET+'.nc'))
else:
    print('No new forecast issued:', 'already frozen' if already_issued else state['faltantes'])
r.exportar_resumo(REGISTRY/'resumo_registro.json')
print('Save all registry events and objects with the notebook output.')
